# Arabic Text-to-SQL Prompt-Aligned Training + Inference

| Setting | Value |
|---------|-------|
| Base model | `Qwen/Qwen2.5-Coder-7B-Instruct` |
| Adapter | QLoRA (rank=64, alpha=128, 7 target modules) |
| Embedding | `BAAI/bge-m3` (~2.3 GB VRAM) |
| Dataset | Ar-Spider (Arabic Text-to-SQL benchmark) |

## Install Dependencies

In [ ]:
#@title Install Dependencies { display-mode: "form" }
!pip install -q transformers accelerate peft bitsandbytes trl datasets
!pip install -q sentencepiece sqlparse nltk sentence-transformers
!pip install -q gdown


# Download and unzip
!gdown --id 1ShZPbM2FvKQy5bh-IpLrtbFNe8LpX-ig
!unzip -qo t2s_datasets.zip

# Rename files to standard names
!mv /content/arspider/Ar_dev_spider.json    /content/arspider/dev.json   2>/dev/null || true
!mv /content/arspider/Ar_train_spider.json  /content/arspider/train.json 2>/dev/null || true

import os,  nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
print("All dependencies installed")

## Download Ar-Spider Dataset

In [ ]:
# Download and unzip
!gdown --id 1ShZPbM2FvKQy5bh-IpLrtbFNe8LpX-ig
!unzip -qo t2s_datasets.zip

# Rename files to standard names
!mv /content/arspider/Ar_dev_spider.json    /content/arspider/dev.json   2>/dev/null || true
!mv /content/arspider/Ar_train_spider.json  /content/arspider/train.json 2>/dev/null || true

import os,  nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
print("All dependencies installed")

AR_SPIDER_DIR = "/content/arspider"

for f in ["train.json", "dev.json", "tables.json"]:
    path = os.path.join(AR_SPIDER_DIR, f)
    assert os.path.exists(path), f"Missing: {path}"
    print(f"   {f}")

print(f"Ar-Spider dataset ready at {AR_SPIDER_DIR}")

## Core Imports & Utilities

In [ ]:
import json, re, os, math, gc, sqlite3, time
from pathlib import Path
from typing import List, Dict, Tuple, Optional, Any
from dataclasses import dataclass, field
from collections import Counter, defaultdict
import sqlparse
from tqdm.auto import tqdm

DATA_DIR = Path("./data"); DATA_DIR.mkdir(exist_ok=True)

## Schema Loading & SQL Execution

In [ ]:
import glob

def spider_tables_json_to_ddl(tables_json_data: List[Dict]) -> Dict[str, str]:
    """Convert Spider-format tables.json into {db_id: CREATE TABLE DDL}."""
    db_schemas = {}
    for db in tables_json_data:
        db_id = db["db_id"]
        table_names = db["table_names_original"]
        col_names = db["column_names_original"]
        col_types = db.get("column_types", [])
        pks = set(db.get("primary_keys", []))
        fks = db.get("foreign_keys", [])

        table_cols = defaultdict(list)
        for col_idx, (tbl_idx, col_name) in enumerate(col_names):
            if tbl_idx == -1:
                continue
            ctype = col_types[col_idx] if col_idx < len(col_types) else "TEXT"
            sql_type = {
                "text": "TEXT", "number": "REAL", "time": "TEXT",
                "boolean": "INTEGER", "others": "TEXT",
            }.get(ctype.lower(), "TEXT")
            is_pk = col_idx in pks
            table_cols[tbl_idx].append((col_name, sql_type, is_pk))

        fk_map = {}
        for fk_col, ref_col in fks:
            if ref_col < len(col_names):
                ref_tbl_idx, ref_col_name = col_names[ref_col]
                if ref_tbl_idx >= 0:
                    fk_map[fk_col] = (table_names[ref_tbl_idx], ref_col_name)

        ddl_parts = []
        for tbl_idx, tbl_name in enumerate(table_names):
            cols_sql = []
            for col_name, sql_type, is_pk in table_cols.get(tbl_idx, []):
                line = f"    {col_name} {sql_type}"
                if is_pk:
                    line += " PRIMARY KEY"
                cols_sql.append(line)
            fk_lines = []
            for col_idx, (tbl_i, cn) in enumerate(col_names):
                if tbl_i == tbl_idx and col_idx in fk_map:
                    ref_tbl, ref_col = fk_map[col_idx]
                    fk_lines.append(
                        f"    FOREIGN KEY ({cn}) REFERENCES {ref_tbl}({ref_col})"
                    )
            all_lines = cols_sql + fk_lines
            if not all_lines:
                all_lines = ["    id INTEGER PRIMARY KEY"]
            ddl = f"CREATE TABLE {tbl_name} (\n" + ",\n".join(all_lines) + "\n);"
            ddl_parts.append(ddl)

        db_schemas[db_id] = "\n\n".join(ddl_parts)
    return db_schemas


def execute_sql_on_db(db_path: str, sql: str, timeout=30) -> Tuple[bool, Any]:
    """Execute against a real SQLite database file (read-only)."""
    try:
        conn = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True, timeout=timeout)
        conn.execute("PRAGMA busy_timeout = 5000")
        cur = conn.cursor()
        cur.execute(sql)
        rows = cur.fetchall()
        cols = [d[0] for d in cur.description] if cur.description else []
        conn.close()
        return True, {"columns": cols, "rows": rows}
    except Exception as e:
        return False, str(e)


def compare_results(r1, r2) -> bool:
    """Compare execution results as unordered sets."""
    if r1 is None or r2 is None:
        return False
    try:
        rows1 = r1["rows"] if isinstance(r1, dict) else r1
        rows2 = r2["rows"] if isinstance(r2, dict) else r2
        set1 = set(tuple(sorted(str(v) for v in r)) for r in rows1)
        set2 = set(tuple(sorted(str(v) for v in r)) for r in rows2)
        if set1 == set2:
            return True
        if len(rows1) == len(rows2):
            norm1 = sorted(tuple(sorted(str(v) for v in r)) for r in rows1)
            norm2 = sorted(tuple(sorted(str(v) for v in r)) for r in rows2)
            if norm1 == norm2:
                return True
            def _nv(v):
                s = str(v).strip()
                try:
                    f = float(s)
                    return str(int(f)) if f == int(f) else f"{f:.6f}"
                except (ValueError, OverflowError):
                    return s.lower()
            n1 = sorted(tuple(sorted(_nv(v) for v in r)) for r in rows1)
            n2 = sorted(tuple(sorted(_nv(v) for v in r)) for r in rows2)
            return n1 == n2
        return False
    except Exception:
        return str(r1) == str(r2)


# Load schemas
with open(os.path.join(AR_SPIDER_DIR, "tables.json"), "r", encoding="utf-8") as f:
    _tables_json = json.load(f)
ar_spider_schemas = spider_tables_json_to_ddl(_tables_json)
print(f"Loaded schemas: {len(ar_spider_schemas)} databases")

# Find database files
db_paths = {}
db_base = ""
for candidate in ["databases", "database"]:
    p = os.path.join(AR_SPIDER_DIR, candidate)
    if os.path.isdir(p):
        db_base = p
        break
if db_base:
    for db_name in os.listdir(db_base):
        db_dir = os.path.join(db_base, db_name)
        if os.path.isdir(db_dir):
            for ext in ["*.sqlite", "*.db", "*.sqlite3"]:
                found = glob.glob(os.path.join(db_dir, ext))
                if found:
                    db_paths[db_name] = found[0]
                    break
print(f"Found {len(db_paths)} SQLite database files")

## Evaluation Metrics (BLEU, SQAM, TSED, EM, EX)

In [ ]:
def tokenize_sql(sql: str) -> List[str]:
    s = sql.strip().rstrip(";").lower()
    s = re.sub(r'([(),=<>!+\-*/])', r' \1 ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s.split()

def compute_bleu(pred_sql: str, gold_sql: str, max_n: int = 4) -> float:
    pred_tokens = tokenize_sql(pred_sql)
    gold_tokens = tokenize_sql(gold_sql)
    if not pred_tokens or not gold_tokens:
        return 0.0
    bp = min(1.0, math.exp(1 - len(gold_tokens) / max(len(pred_tokens), 1)))
    log_avg = 0.0
    for n in range(1, max_n + 1):
        pred_ng = Counter(tuple(pred_tokens[i:i+n]) for i in range(len(pred_tokens) - n + 1))
        gold_ng = Counter(tuple(gold_tokens[i:i+n]) for i in range(len(gold_tokens) - n + 1))
        clipped = sum(min(pred_ng[ng], gold_ng[ng]) for ng in pred_ng)
        total = max(sum(pred_ng.values()), 1)
        if clipped == 0 and n > 1:
            clipped = 1
        precision = clipped / total
        log_avg += (1.0 / max_n) * math.log(max(precision, 1e-10))
    return bp * math.exp(log_avg)

SQAM_WEIGHTS = {
    "select": 0.30, "from": 0.20, "where": 0.25,
    "group_by": 0.10, "having": 0.05, "order_by": 0.05, "limit": 0.05,
}

def _extract_clause(sql, clause, next_clauses):
    pattern = rf'\b{clause}\b\s+(.*?)(?={"|".join(rf"(?:\b{nc}\b)" for nc in next_clauses)}|;|$)'
    m = re.search(pattern, sql, re.I | re.DOTALL)
    return m.group(1).strip() if m else ""

def _parse_sql_clauses(sql):
    s = re.sub(r'\s+', ' ', sql.strip().rstrip(";").strip())
    clauses = {}
    clause_order = ["select", "from", "where", "group by", "having", "order by", "limit"]
    next_map = {
        "select": ["from","where","group by","having","order by","limit"],
        "from": ["where","group by","having","order by","limit"],
        "where": ["group by","having","order by","limit"],
        "group by": ["having","order by","limit"],
        "having": ["order by","limit"],
        "order by": ["limit"],
        "limit": [],
    }
    for cl in clause_order:
        nxt = next_map.get(cl, [])
        content = _extract_clause(s, cl.replace(" ",r"\s+"), [nc.replace(" ",r"\s+") for nc in nxt])
        clauses[cl.replace(" ","_")] = content
    return clauses

def _split_items(clause_content):
    if not clause_content: return set()
    items, depth, cur = [], 0, ""
    for ch in clause_content:
        if ch == '(': depth += 1
        elif ch == ')': depth -= 1
        if ch == ',' and depth == 0:
            items.append(cur.strip().lower()); cur = ""
        else: cur += ch
    if cur.strip(): items.append(cur.strip().lower())
    expanded = []
    for item in items:
        parts = re.split(r'\b(?:and|or)\b', item, flags=re.I)
        expanded.extend(p.strip() for p in parts if p.strip())
    return set(expanded)

def compute_sqam(pred_sql, gold_sql):
    pred_c = _parse_sql_clauses(pred_sql)
    gold_c = _parse_sql_clauses(gold_sql)
    tw, ws = 0.0, 0.0
    for cn, w in SQAM_WEIGHTS.items():
        pi = _split_items(pred_c.get(cn,""))
        gi = _split_items(gold_c.get(cn,""))
        if not gi and not pi: continue
        tw += w
        if not gi or not pi: continue
        inter = pi & gi
        p = len(inter)/len(pi) if pi else 0
        r = len(inter)/len(gi) if gi else 0
        f1 = (2*p*r)/(p+r) if (p+r) > 0 else 0
        ws += w * f1
    return ws / tw if tw > 0 else 0.0

class SimpleNode:
    def __init__(self, label, children=None):
        self.label = label; self.children = children or []

def _sqlparse_to_node(token):
    if hasattr(token, 'tokens'):
        label = str(token.ttype or type(token).__name__).split('.')[-1]
        return SimpleNode(label, [_sqlparse_to_node(t) for t in token.tokens if str(t).strip()])
    return SimpleNode(str(token).strip().lower() or "EMPTY")

def _count_nodes(n): return 1 + sum(_count_nodes(c) for c in n.children)

def _tree_sim(t1, t2):
    score = 1.0 if t1.label == t2.label else 0.0
    if not t1.children and not t2.children: return score
    matched, used = 0, set()
    for c1 in t1.children:
        best_s, best_j = 0, -1
        for j, c2 in enumerate(t2.children):
            if j in used: continue
            s = _tree_sim(c1, c2)
            if s > best_s: best_s, best_j = s, j
        if best_j >= 0: matched += best_s; used.add(best_j)
    total = max(len(t1.children), len(t2.children))
    return 0.4 * score + 0.6 * (matched / total if total else 1.0)

def compute_tsed(pred_sql, gold_sql):
    try:
        pp = sqlparse.parse(pred_sql.strip())
        gp = sqlparse.parse(gold_sql.strip())
        if not pp or not gp: return 0.0
        return _tree_sim(_sqlparse_to_node(pp[0]), _sqlparse_to_node(gp[0]))
    except:
        pt = set(tokenize_sql(pred_sql)); gt = set(tokenize_sql(gold_sql))
        return len(pt & gt) / max(len(pt | gt), 1) if gt else 0.0

def _normalize_sql_for_em(sql):
    s = sql.strip().rstrip(";").strip().lower()
    s = re.sub(r'\s+', ' ', s)
    s = re.sub(r'[`"\[\]]', '', s)
    s = re.sub(r'\(\s+', '(', s)
    s = re.sub(r'\s+\)', ')', s)
    s = re.sub(r'\bas\s+(?!integer|real|text|numeric|blob|int|varchar|char|float|double|boolean|date|time|datetime)\w+', '', s)
    return s.strip()

def compute_exact_match(pred_sql, gold_sql):
    return _normalize_sql_for_em(pred_sql) == _normalize_sql_for_em(gold_sql)

print("Evaluation metrics loaded (EX, EM, BLEU, SQAM, TSED)")

## Configuration

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

#@title Configuration { display-mode: "form" }

FINETUNE_7B_BASE  = "Qwen/Qwen2.5-Coder-7B-Instruct"
LORA_RANK_7B      = 64        #@param {type:"integer"}
LORA_ALPHA_7B     = 128        #@param {type:"integer"}
NUM_EPOCHS_7B     = 3         #@param {type:"slider", min:1, max:10, step:1}
LEARNING_RATE_7B  = 5e-5      #@param {type:"number"}
PER_DEVICE_BATCH_7B = 6      #@param {type:"integer"}
GRAD_ACCUM_7B     = 4        #@param {type:"integer"}  # raised 4→16 to compensate for smaller batch
MAX_SEQ_LENGTH_7B = 4096      #@param {type:"integer"}
USE_ENHANCED_PROMPT = True     #@param {type:"boolean"}

ADAPTER_7B_OUTPUT = Path("./data/arabic_7b_adapter")
ADAPTER_7B_OUTPUT.mkdir(parents=True, exist_ok=True)

# Prompt template (SQL-only, no CoT)
SQL_ONLY_PROMPT = '''Database Engine: SQLite

Database Schema:
{db_details}

Question:
{question}

Generate the SQL query that answers this question. Output ONLY the SQL inside a code block:
```sql
-- Your SQL query
```'''

SQL_ONLY_SYSTEM = (
    "You are an expert Arabic-to-SQL translator. Given a database schema "
    "and a question (in Arabic or English), carefully map Arabic terms to "
    "the correct English column and table names in the schema, then output "
    "ONLY the correct SQL query inside a code block. No explanations."
)

# Enhanced prompt template (with table summary + JOIN guidance)
SQL_ENHANCED_PROMPT = '''Database Engine: SQLite

Database Schema:
{db_details}

Table → Column Reference (use ONLY columns from the table that has them):
{table_summary}

IMPORTANT: Use the MINIMUM number of tables needed.
If ALL required columns exist in ONE table, do NOT use JOIN.

Question:
{question}

Generate the SQL query that answers this question. Output ONLY the SQL inside a code block:
```sql
-- Your SQL query
```'''


def build_table_column_summary(schema_ddl: str) -> str:
    """Generate a compact table→columns mapping for the enhanced prompt."""
    tables = {}
    cur_table = None
    for line in schema_ddl.split("\n"):
        line = line.strip()
        m = re.match(
            r'CREATE\s+TABLE\s+(?:IF\s+NOT\s+EXISTS\s+)?[`"\[]?([\w]+)[`"\]]?',
            line, re.I,
        )
        if m:
            cur_table = m.group(1)
            tables[cur_table] = []
        elif cur_table and line and not line.startswith(')'):
            cm = re.match(r'[`"\[]?([\w]+)[`"\]]?\s+\w+', line)
            if cm and not any(
                line.upper().startswith(kw) for kw in
                ('PRIMARY', 'FOREIGN', 'UNIQUE', 'CHECK', 'CONSTRAINT', '--')
            ):
                tables[cur_table].append(cm.group(1))
    if not tables:
        return ""
    lines = []
    for table, cols in tables.items():
        lines.append(f"  • {table}: {', '.join(cols)}")
    return "\n".join(lines)


def format_prompt(schema: str, question: str) -> str:
    """Format prompt using the configured template (base or enhanced)."""
    if USE_ENHANCED_PROMPT:
        table_summary = build_table_column_summary(schema)
        return SQL_ENHANCED_PROMPT.format(
            db_details=schema[:4500],
            table_summary=table_summary,
            question=question,
        )
    else:
        return SQL_ONLY_PROMPT.format(
            db_details=schema[:5000],
            question=question,
        )

print(f"Config ready")
print(f"   Base: {FINETUNE_7B_BASE}")
print(f"   LoRA: rank={LORA_RANK_7B}, alpha={LORA_ALPHA_7B}")
print(f"   LR: {LEARNING_RATE_7B}, Epochs: {NUM_EPOCHS_7B}")
print(f"   Effective batch: {PER_DEVICE_BATCH_7B * GRAD_ACCUM_7B}")
print(f"   Prompt: {'Enhanced (table summary + JOIN guidance)' if USE_ENHANCED_PROMPT else 'Base (schema only)'}")

## Section A: Tier 1 Feature Configuration (Shared: Train + Inference)

In [ ]:
#@title Tier 1 Feature Configuration (Shared: Train + Inference) { display-mode: "form" }

# Prompt Alignment
TRAIN_USE_ALIGNED_PROMPT  = True    #@param {type:"boolean"}
FEATURE_DROPOUT_RATE      = 0.20   #@param {type:"number"}

# Technique Toggles (used by BOTH training prompt and inference)
ENABLE_MSCHEMA            = True   #@param {type:"boolean"}
ENABLE_VALUE_HINTS        = True   #@param {type:"boolean"}
VALUE_HINT_MAX_PER_COL    = 4      #@param {type:"integer"}

ENABLE_COLUMN_LINKING     = True   #@param {type:"boolean"}
COL_LINK_TOP_K            = 6      #@param {type:"integer"}

ENABLE_AR_COL_DESC        = True   #@param {type:"boolean"}
AR_COL_DESC_PATH          = "/content/drive/MyDrive/ar_column_descriptions.json"  #@param {type:"string"}

ENABLE_ENGLISH_HINT       = True   #@param {type:"boolean"}

ENABLE_BILINGUAL_SCHEMA   = True   #@param {type:"boolean"}
AR_TABLE_GLOSS_PATH       = "/content/drive/MyDrive/ar_table_glosses.json"  #@param {type:"string"}

ENABLE_VALUE_INJECTION    = True   #@param {type:"boolean"}
VALUE_INJECTION_TOP_K     = 5      #@param {type:"integer"}
VALUE_INJECTION_THRESHOLD = 0.40   #@param {type:"number"}

ENABLE_BILINGUAL_EMBEDDINGS = True   #@param {type:"boolean"}

ENABLE_SAMPLE_ROWS        = True   #@param {type:"boolean"}
SAMPLE_ROWS_LIMIT         = 3      #@param {type:"integer"}
SAMPLE_ROWS_MAX_TABLES    = 3      #@param {type:"integer"}

# Inference-only settings (not used during training)
ENABLE_SELF_CONSISTENCY   = True   #@param {type:"boolean"}
SC_NUM_CANDIDATES         = 8      #@param {type:"integer"}
SC_TEMPERATURE            = 0.7    #@param {type:"number"}
SC_TOP_P                  = 0.95   #@param {type:"number"}
ENABLE_MULTI_TEMP         = True   #@param {type:"boolean"}
SC_TEMP_TIERS = [
    (0.3, 2),  # 2 conservative candidates
    (0.7, 3),  # 3 medium candidates
    (1.1, 2),  # 2 creative candidates
]


SC_DETERMINISTIC_SEEDS    = True   #@param {type:"boolean"}
SC_BASE_SEED              = 1234   #@param {type:"integer"}
LOG_CANDIDATES            = True   #@param {type:"boolean"}

ENABLE_SELF_CORRECTION    = True   #@param {type:"boolean"}
SC_MAX_RETRIES            = 2      #@param {type:"integer"}
ENABLE_ID_FIXING          = True   #@param {type:"boolean"}
ENABLE_FILTER_REMOVAL     = True   #@param {type:"boolean"}
ENABLE_SQL_VALUE_GROUNDING = True   #@param {type:"boolean"}
SQL_VALUE_SIM_THRESHOLD    = 0.70   #@param {type:"number"}
ENABLE_LOW_CONF_FALLBACK  = True   #@param {type:"boolean"}
LOW_CONF_VOTE_THRESHOLD   = 3      #@param {type:"integer"}
LOW_CONF_EXTRA_CANDIDATES = 8      #@param {type:"integer"}
SC_BATCH_SIZE             = 1024   #@param {type:"integer"}
RESUME_FROM               = 0      #@param {type:"integer"}
REGENERATE_DESCRIPTIONS   = False  #@param {type:"boolean"}
REGENERATE_TABLE_GLOSSES  = False  #@param {type:"boolean"}

print("Tier 1 Feature Configuration:")
print(f"   Prompt alignment for training : {'ON' if TRAIN_USE_ALIGNED_PROMPT else 'OFF'}")
print(f"   Feature dropout rate          : {FEATURE_DROPOUT_RATE}")
print(f"   M-Schema                      : {'ON' if ENABLE_MSCHEMA else 'OFF'}")
print(f"   Value Hints                   : {'ON' if ENABLE_VALUE_HINTS else 'OFF'}")
print(f"   Column Linking (embedding)    : {'ON' if ENABLE_COLUMN_LINKING else 'OFF'}")
print(f"   Arabic Column Descriptions    : {'ON' if ENABLE_AR_COL_DESC else 'OFF'}")
print(f"   English Translation Hint      : {'ON' if ENABLE_ENGLISH_HINT else 'OFF'}")
print(f"   Bilingual Schema (AR gloss)   : {'ON' if ENABLE_BILINGUAL_SCHEMA else 'OFF'}")
print(f"   Value Injection (emb match)   : {'ON' if ENABLE_VALUE_INJECTION else 'OFF'}")
print(f"   Bilingual Embeddings          : {'ON' if ENABLE_BILINGUAL_EMBEDDINGS else 'OFF'}")
print(f"   Sample Rows in Prompt         : {'ON' if ENABLE_SAMPLE_ROWS else 'OFF'}")

In [ ]:
#@title Documented Full Pipeline Sanity Check { display-mode: "form" }
# Asserts ALL components are ON (this is the headline configuration).
_off = [f for f in ["ENABLE_SELF_CORRECTION","ENABLE_ID_FIXING","ENABLE_FILTER_REMOVAL",
                    "ENABLE_SQL_VALUE_GROUNDING","ENABLE_LOW_CONF_FALLBACK","ENABLE_MULTI_TEMP"]
        if not globals().get(f, False)]
if _off:
    raise RuntimeError("DOCUMENTED-FULL MODE VIOLATION — these must be True: " + ", ".join(_off))
print("Documented full pipeline verified — all components ACTIVE:")
print("   5 prompt components + multi-temp voting + repair stage + low-conf fallback")
print(f"   Deterministic seeds: {'ON (base='+str(SC_BASE_SEED)+')' if SC_DETERMINISTIC_SEEDS else 'OFF'}")
print(f"   Per-candidate logging: {'ON' if LOG_CANDIDATES else 'OFF'}")

## Section B: Shared Function Definitions (Train + Inference)

In [ ]:
#@title Shared: Schema Parsing, M-Schema, Lookups { display-mode: "form" }


import numpy as np
from collections import defaultdict
from dataclasses import dataclass, field

# EvalSample dataclass (reused for both train and eval)
@dataclass
class EvalSample:
    id: str
    db_id: str
    arabic_question: str
    gold_sql: str
    schema_ddl: str
    db_path: str = ""
    english_question: str = ""


# Schema parsing

def extract_tables_columns(schema_sql: str) -> Dict[str, List[str]]:
    """Parse CREATE TABLE DDL into {table_name: [column_names]}."""
    tables = {}
    cur_table = None
    for line in schema_sql.split("\n"):
        line = line.strip()
        m = re.match(
            r'CREATE\s+TABLE\s+(?:IF\s+NOT\s+EXISTS\s+)?[`"\[]?([\w]+)[`"\]]?',
            line, re.I,
        )
        if m:
            cur_table = m.group(1)
            tables[cur_table] = []
        elif cur_table and line and not line.startswith(')'):
            cm = re.match(r'[`"\[]?([\w]+)[`"\]]?\s+\w+', line)
            if cm and not any(
                line.upper().startswith(kw) for kw in
                ('PRIMARY', 'FOREIGN', 'UNIQUE', 'CHECK', 'CONSTRAINT', '--')
            ):
                tables[cur_table].append(cm.group(1))
    return tables


def build_table_column_summary(schema_ddl: str) -> str:
    """Compact table→columns mapping for prompt."""
    tables = extract_tables_columns(schema_ddl)
    if not tables:
        return ""
    lines = []
    for table, cols in tables.items():
        lines.append(f"  • {table}: {', '.join(cols)}")
    return "\n".join(lines)


# Index tables.json by db_id
_tables_json_by_id = {}
try:
    for _entry in _tables_json:
        _tables_json_by_id[_entry["db_id"]] = _entry
    print(f"Indexed {len(_tables_json_by_id)} database schemas for M-Schema")
except NameError:
    print("_tables_json not found — will fall back to DDL-based M-Schema")


# Arabic description + gloss lookups (populated in Section D)
_ar_col_descriptions = {}  # {db_id: {"table.column": "وصف عربي", ...}}
_ar_table_glosses    = {}  # {db_id: {"table_name": "وصف عربي", ...}}


def get_column_description(db_id: str, table: str, column: str) -> str:
    """Look up Arabic description for a column. Returns '' if not found."""
    return _ar_col_descriptions.get(db_id, {}).get(f"{table}.{column}", "")


def get_table_gloss(db_id: str, table_name: str) -> str:
    """Look up Arabic gloss for a table. Returns '' if not found."""
    return _ar_table_glosses.get(db_id, {}).get(table_name, "")


# M-Schema builder

def build_mschema(tables_json_entry: dict, db_id: str = "") -> str:
    """Build M-Schema from a single tables.json database entry.
    Includes Arabic table glosses and column descriptions inline."""
    table_names = tables_json_entry["table_names_original"]
    col_names   = tables_json_entry["column_names_original"]
    col_types   = tables_json_entry.get("column_types", [])
    pks         = set(tables_json_entry.get("primary_keys", []))
    fks         = tables_json_entry.get("foreign_keys", [])

    table_cols = defaultdict(list)
    for col_idx, (tbl_idx, col_name) in enumerate(col_names):
        if tbl_idx == -1:
            continue
        ctype = col_types[col_idx] if col_idx < len(col_types) else "text"
        sql_type = {"text": "TEXT", "number": "REAL", "time": "TEXT",
                    "boolean": "INT", "others": "TEXT"}.get(ctype.lower(), "TEXT")
        is_pk = col_idx in pks
        table_cols[tbl_idx].append((col_name, sql_type, is_pk, col_idx))

    fk_map = {}
    for fk_col, ref_col in fks:
        if ref_col < len(col_names):
            ref_tbl_idx, ref_col_name = col_names[ref_col]
            if ref_tbl_idx >= 0:
                fk_map[fk_col] = (table_names[ref_tbl_idx], ref_col_name)

    lines = []
    for tbl_idx, tbl_name in enumerate(table_names):
        cols = table_cols.get(tbl_idx, [])
        if not cols:
            continue
        col_parts = []
        for col_name, sql_type, is_pk, col_idx in cols:
            pk_mark = "*" if is_pk else ""
            ar_desc = ""
            if ENABLE_BILINGUAL_SCHEMA and db_id:
                ar_desc = get_column_description(db_id, tbl_name, col_name)
            if ar_desc:
                col_parts.append(f"({col_name}{pk_mark}, {sql_type}, {ar_desc})")
            else:
                col_parts.append(f"({col_name}{pk_mark}, {sql_type})")
        tbl_gloss = ""
        if ENABLE_BILINGUAL_SCHEMA and db_id:
            tbl_gloss = get_table_gloss(db_id, tbl_name)
        if tbl_gloss:
            lines.append(f"【{tbl_name}】 ({tbl_gloss})")
        else:
            lines.append(f"【{tbl_name}】")
        lines.append("  " + "  ".join(col_parts))
        for col_name, sql_type, is_pk, col_idx in cols:
            if col_idx in fk_map:
                ref_tbl, ref_col = fk_map[col_idx]
                lines.append(f"  -> FK: {col_name} -> {ref_tbl}.{ref_col}")
    return "\n".join(lines)


def build_mschema_from_ddl(schema_ddl: str) -> str:
    """Fallback: build M-Schema from DDL string (no bilingual annotations)."""
    tables = {}
    cur_table = None
    fks_list = []
    for line in schema_ddl.split("\n"):
        line = line.strip()
        m = re.match(
            r'CREATE\s+TABLE\s+(?:IF\s+NOT\s+EXISTS\s+)?[`"\[]?([\w]+)[`"\]]?',
            line, re.I)
        if m:
            cur_table = m.group(1)
            tables[cur_table] = []
        elif cur_table and line and not line.startswith(')'):
            fk_m = re.match(
                r'FOREIGN\s+KEY\s*\((\w+)\)\s*REFERENCES\s+(\w+)\s*\((\w+)\)',
                line, re.I)
            if fk_m:
                fks_list.append((cur_table, fk_m.group(1), fk_m.group(2), fk_m.group(3)))
                continue
            if any(line.upper().startswith(kw) for kw in
                   ('PRIMARY','FOREIGN','UNIQUE','CHECK','CONSTRAINT','--')):
                continue
            cm = re.match(r'[`"\[]?([\w]+)[`"\]]?\s+(\w+)', line)
            if cm:
                col_name = cm.group(1)
                col_type = cm.group(2).upper()
                is_pk = 'PRIMARY KEY' in line.upper()
                tables[cur_table].append((col_name, col_type, is_pk))

    lines = []
    for tbl_name, cols in tables.items():
        col_parts = []
        for col_name, col_type, is_pk in cols:
            pk_mark = "*" if is_pk else ""
            col_parts.append(f"({col_name}{pk_mark}, {col_type})")
        lines.append(f"【{tbl_name}】")
        lines.append("  " + "  ".join(col_parts))
        for fk_tbl, fk_col, ref_tbl, ref_col in fks_list:
            if fk_tbl == tbl_name:
                lines.append(f"  -> FK: {fk_col} -> {ref_tbl}.{ref_col}")
    return "\n".join(lines)


def get_mschema(db_id: str, schema_ddl: str) -> str:
    """Get M-Schema for a database (from tables.json if available, else DDL)."""
    if db_id in _tables_json_by_id:
        return build_mschema(_tables_json_by_id[db_id], db_id=db_id)
    return build_mschema_from_ddl(schema_ddl)

In [ ]:
#@title Shared: Value Hints, Column Linking, Value Injection, Sample Rows { display-mode: "form" }

# Database value hints (question-aware)

def sample_db_values_question_aware(db_path: str, schema_ddl: str,
                                     question: str,
                                     max_per_col: int = 4) -> str:
    """Question-aware value sampling from real DB. Returns compact hint string."""
    if not db_path or not os.path.exists(db_path):
        return ""
    schema_tables = extract_tables_columns(schema_ddl)
    if not schema_tables:
        return ""

    all_hints = []
    try:
        conn = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True, timeout=10)
        conn.text_factory = str
        cursor = conn.cursor()
        for tbl_name, columns in schema_tables.items():
            for col_name in columns:
                try:
                    cursor.execute(f'PRAGMA table_info("{tbl_name}")')
                    col_info = {row[1].lower(): row[2] for row in cursor.fetchall()}
                    col_type = col_info.get(col_name.lower(), "TEXT").upper()
                    is_text = any(t in col_type for t in
                                  ["TEXT", "VARCHAR", "CHAR", "CLOB"])
                    if not is_text:
                        continue
                    cursor.execute(
                        f'SELECT DISTINCT "{col_name}" FROM "{tbl_name}" '
                        f'WHERE "{col_name}" IS NOT NULL AND TRIM("{col_name}") != "" '
                        f'LIMIT {max_per_col * 2}')
                    values = [str(row[0]) for row in cursor.fetchall()
                              if row[0] is not None and str(row[0]).strip()]
                    if not values:
                        continue
                    col_lower = col_name.lower()
                    relevance = 0
                    high_value = ['name', 'type', 'status', 'country', 'city',
                                  'state', 'region', 'continent', 'language',
                                  'nationality', 'category', 'genre', 'major',
                                  'department', 'sex', 'gender', 'color',
                                  'brand', 'title', 'position', 'head']
                    if any(hv in col_lower for hv in high_value):
                        relevance += 2
                    for v in values:
                        if v.lower() in question.lower():
                            relevance += 5
                    display = [v[:35] for v in values[:max_per_col]]
                    all_hints.append((relevance, f"    {tbl_name}.{col_name}: {display}"))
                except Exception:
                    continue
        conn.close()
    except Exception:
        return ""

    if not all_hints:
        return ""
    all_hints.sort(key=lambda x: -x[0])
    selected = [h[1] for h in all_hints[:min(len(all_hints), 15)]]
    return "Sample column values (use these EXACT values in your SQL):\n" + "\n".join(selected)


# Column linking (embedding similarity)

_emb_model    = None  # set in Section D
_col_emb_cache = {}  # {db_id: {"candidates": [...], "embeddings": np.ndarray}}


def _build_column_description(table_name: str, col_name: str,
                               db_id: str = "") -> str:
    """Build natural-language description for embedding.
    Includes Arabic description for bilingual matching."""
    readable = re.sub(r'([a-z])([A-Z])', r'\1 \2', col_name)
    readable = readable.replace('_', ' ').lower()
    tbl_readable = re.sub(r'([a-z])([A-Z])', r'\1 \2', table_name)
    tbl_readable = tbl_readable.replace('_', ' ').lower()
    base = f"{readable} in {tbl_readable}"
    if ENABLE_BILINGUAL_EMBEDDINGS and db_id:
        ar_desc = get_column_description(db_id, table_name, col_name)
        if ar_desc:
            return f"{base} — {ar_desc}"
    return base


def _get_column_embeddings(db_id: str, schema_ddl: str) -> dict:
    """Get or compute cached column embeddings for a database."""
    if db_id in _col_emb_cache:
        return _col_emb_cache[db_id]

    schema_tables = extract_tables_columns(schema_ddl)
    if not schema_tables or _emb_model is None:
        empty = {"candidates": [], "embeddings": np.array([])}
        _col_emb_cache[db_id] = empty
        return empty

    candidates = []
    texts = []
    for tbl, cols in schema_tables.items():
        for col in cols:
            desc = _build_column_description(tbl, col, db_id=db_id)
            candidates.append({
                "ref": f"{tbl}.{col}", "text": desc,
                "tbl": tbl, "col": col,
            })
            texts.append(desc)

    if texts:
        embeddings = _emb_model.encode(
            texts, normalize_embeddings=True,
            show_progress_bar=False, batch_size=64,
        )
    else:
        embeddings = np.array([])

    result = {"candidates": candidates, "embeddings": embeddings}
    _col_emb_cache[db_id] = result
    return result


def link_columns_to_question(db_id: str, schema_ddl: str, question: str,
                              db_path: str = "", top_k: int = 6) -> str:
    """Score every column against the Arabic question using embedding similarity.
    Returns a prompt hint with the top-K most relevant columns."""
    if _emb_model is None:
        return ""

    col_data = _get_column_embeddings(db_id, schema_ddl)
    candidates = col_data["candidates"]
    col_embeddings = col_data["embeddings"]

    if not candidates or len(col_embeddings) == 0:
        return ""

    q_embedding = _emb_model.encode(
        [question], normalize_embeddings=True,
        show_progress_bar=False,
    )[0]
    final_scores = col_embeddings @ q_embedding

    ranked_indices = np.argsort(-final_scores)[:top_k]
    lines = []
    for idx in ranked_indices:
        if final_scores[idx] < 0.50:  # raised to 0.50 — only strong matches survive
            continue
        c = candidates[idx]
        ar_desc = ""
        if ENABLE_AR_COL_DESC:
            ar_desc = get_column_description(db_id, c["tbl"], c["col"])
        if ar_desc:
            lines.append(f"  ★ {c['ref']} — {ar_desc}")
        else:
            lines.append(f"  ★ {c['ref']}")

    if not lines:
        return ""
    return "Likely relevant columns (prefer these):\n" + "\n".join(lines)


# Question-matched value injection (embedding-based)

_db_value_emb_cache = {}  # {db_id: {"values": [...], "embeddings": np.ndarray, "col_refs": [...]}}


def _build_value_embeddings(db_id: str, db_path: str, schema_ddl: str) -> dict:
    """Pre-compute embeddings for all text values in a database."""
    if db_id in _db_value_emb_cache:
        return _db_value_emb_cache[db_id]

    empty = {"values": [], "embeddings": np.array([]), "col_refs": []}
    if not db_path or not os.path.exists(db_path) or _emb_model is None:
        _db_value_emb_cache[db_id] = empty
        return empty

    schema_tables = extract_tables_columns(schema_ddl)
    if not schema_tables:
        _db_value_emb_cache[db_id] = empty
        return empty

    all_values = []
    col_refs = []
    try:
        conn = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True, timeout=10)
        conn.text_factory = str
        cursor = conn.cursor()
        for tbl, cols in schema_tables.items():
            for col in cols:
                try:
                    cursor.execute(f'PRAGMA table_info("{tbl}")')
                    col_info = {row[1].lower(): row[2] for row in cursor.fetchall()}
                    col_type = col_info.get(col.lower(), "TEXT").upper()
                    if not any(t in col_type for t in ["TEXT", "VARCHAR", "CHAR", "CLOB"]):
                        continue
                    cursor.execute(
                        f'SELECT DISTINCT "{col}" FROM "{tbl}" '
                        f'WHERE "{col}" IS NOT NULL AND TRIM("{col}") != "" '
                        f'LIMIT 150')
                    for row in cursor.fetchall():
                        val = str(row[0]).strip()
                        if len(val) >= 2:
                            all_values.append(val)
                            col_refs.append(f"{tbl}.{col}")
                except Exception:
                    continue
        conn.close()
    except Exception:
        _db_value_emb_cache[db_id] = empty
        return empty

    if not all_values:
        _db_value_emb_cache[db_id] = empty
        return empty

    # Deduplicate
    seen = {}
    unique_values = []
    unique_refs = []
    for val, ref in zip(all_values, col_refs):
        key = val.lower()
        if key not in seen:
            seen[key] = len(unique_values)
            unique_values.append(val)
            unique_refs.append(ref)

    embeddings = _emb_model.encode(
        unique_values, normalize_embeddings=True,
        show_progress_bar=False, batch_size=128,
    )

    result = {"values": unique_values, "embeddings": embeddings, "col_refs": unique_refs}
    _db_value_emb_cache[db_id] = result
    return result


def find_question_matched_values(db_id: str, db_path: str, schema_ddl: str,
                                  arabic_question: str, english_question: str = "",
                                  top_k: int = 5, threshold: float = 0.40) -> str:
    """Find DB values semantically closest to the question.
    Uses BOTH Arabic and English queries for dual-language matching."""
    if _emb_model is None:
        return ""

    val_data = _build_value_embeddings(db_id, db_path, schema_ddl)
    values = val_data["values"]
    val_embeddings = val_data["embeddings"]
    col_refs = val_data["col_refs"]

    if not values or len(val_embeddings) == 0:
        return ""

    ar_emb = _emb_model.encode(
        [arabic_question], normalize_embeddings=True,
        show_progress_bar=False,
    )[0]
    sims = val_embeddings @ ar_emb

    if english_question:
        en_emb = _emb_model.encode(
            [english_question], normalize_embeddings=True,
            show_progress_bar=False,
        )[0]
        en_sims = val_embeddings @ en_emb
        sims = np.maximum(sims, en_sims)

    top_indices = np.argsort(-sims)[:top_k]
    hints = []
    seen_vals = set()
    for idx in top_indices:
        if sims[idx] < threshold:
            break
        val = values[idx]
        if val.lower() in seen_vals:
            continue
        seen_vals.add(val.lower())
        col_ref = col_refs[idx]
        hints.append(f"  → {col_ref} = '{val}'")

    if not hints:
        return ""
    return "Detected values in question (use EXACT spelling in WHERE):\n" + "\n".join(hints)


# Sample rows in prompt

def get_sample_rows(db_path: str, schema_ddl: str, column_hints: str = "",
                     limit: int = 3, max_tables: int = 3) -> str:
    """Get sample rows from the most relevant tables."""
    if not db_path or not os.path.exists(db_path):
        return ""
    schema_tables = extract_tables_columns(schema_ddl)
    if not schema_tables:
        return ""

    priority_tables = []
    if column_hints:
        for tbl in schema_tables:
            if tbl in column_hints:
                priority_tables.append(tbl)
    for tbl in schema_tables:
        if tbl not in priority_tables:
            priority_tables.append(tbl)
    selected_tables = priority_tables[:max_tables]

    lines = []
    try:
        conn = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True, timeout=10)
        conn.text_factory = str
        cursor = conn.cursor()
        for tbl in selected_tables:
            cols = schema_tables.get(tbl, [])
            if not cols:
                continue
            try:
                display_cols = cols[:6]
                col_str = ', '.join(f'"{c}"' for c in display_cols)
                cursor.execute(f'SELECT {col_str} FROM "{tbl}" LIMIT {limit}')
                rows = cursor.fetchall()
                if not rows:
                    continue
                header = " | ".join(c[:15] for c in display_cols)
                lines.append(f"  {tbl}: {header}")
                for row in rows:
                    row_str = " | ".join(
                        str(v)[:15] if v is not None else "NULL"
                        for v in row
                    )
                    lines.append(f"    {row_str}")
            except Exception:
                continue
        conn.close()
    except Exception:
        return ""

    if not lines:
        return ""
    return "Sample data (actual rows from database):\n" + "\n".join(lines)

In [ ]:
#@title Shared: Prompt Templates & Builders { display-mode: "form" }

# Prompt templates (shared between training and inference)

TIER1_SYSTEM = SQL_ONLY_SYSTEM  # reuse the proven system prompt

TIER1_PROMPT = '''Database Engine: SQLite

Database Schema:
{schema}

{value_hints}
{matched_values}
{column_hints}
{sample_rows}
Table -> Column Reference (use ONLY columns from the table that has them):
{table_summary}

IMPORTANT: Use the MINIMUM number of tables needed.
If ALL required columns exist in ONE table, do NOT use JOIN.
Only JOIN tables when the question requires data from multiple tables.
Use the exact column values shown above when filtering.

Question:
{question}
{english_hint}
Generate the SQL query. Output ONLY SQL inside a code block:
```sql
```'''

TIER1_RETRY_PROMPT = '''Database Engine: SQLite

Database Schema:
{schema}

{value_hints}
Your previous SQL had an error:
  SQL: {failed_sql}
  Error: {error_msg}

Question:
{question}

Fix the SQL. Output ONLY the corrected SQL inside a code block:
```sql
```'''


def _question_has_text_entities(question: str, english_question: str = "") -> bool:
    """Check if the question likely needs text value matching.
    Returns False for pure numeric/count/aggregation queries where
    value injection would just add noise.

    Heuristic: True if the question contains proper nouns, quoted strings,
    or named entities that could match DB text values."""
    combined = question + " " + english_question

    # Check for quoted strings in the question
    if re.search(r'["\'].+?["\'"]', combined):
        return True

    sql_words = {'select','from','where','join','and','or','not','in','the',
                 'what','which','how','many','show','find','list','give','all',
                 'is','are','was','were','has','have','had','does','did',
                 'who','when','with','for','that','this','than','then',
                 'more','less','each','every','any','some','no','by',
                 'on','at','to','of','a','an','do','be','it','its',
                 'count','average','maximum','minimum','total','number',
                 'most','least','highest','lowest','largest','smallest',
                 'between','above','below','greater','smaller','equal',
                 'name','names','date','dates','id','type','types',
                 'started','ended','released','called','located',
                 'higher','lower','different','same','both','either'}
    latin_words = re.findall(r'\b([A-Z][a-z]{2,})\b', combined)
    proper_nouns = [w for w in latin_words if w.lower() not in sql_words]
    if len(proper_nouns) >= 1:
        return True

    arabic_sql_vocab = {
        'ما', 'هو', 'هي', 'كم', 'أي', 'من', 'في', 'على', 'إلى', 'عن',
        'مع', 'بين', 'أو', 'و', 'لا', 'كل', 'جميع', 'عدد', 'اسم',
        'أسماء', 'أظهر', 'اعرض', 'جد', 'أعط', 'أكبر', 'أصغر', 'أعلى',
        'أدنى', 'أقل', 'أكثر', 'متوسط', 'مجموع', 'الذي', 'التي',
        'الذين', 'اللذين', 'كانت', 'كان', 'يكون', 'تكون', 'لديه',
        'لديها', 'يوجد', 'توجد', 'عدد', 'رقم', 'تاريخ', 'وقت',
    }

    has_numbers = bool(re.search(r'\b\d+\b', combined))
    has_comparison = bool(re.search(
        r'(أعلى|أكبر|أقل|أدنى|أكثر|greater|less|more|than|higher|lower|above|below|between|>|<|>=|<=)',
        combined, re.I))

    # Pure numeric comparison with no proper nouns → skip
    if has_numbers and has_comparison and not proper_nouns:
        return False

    # Very short questions that are just counts → skip
    count_patterns = ['كم عدد', 'how many', 'count', 'عدد']
    q_lower = combined.lower()
    if any(p in q_lower for p in count_patterns):
        # Count query — only inject if there's a named entity
        if not proper_nouns and not re.search(r'["\'"].+?["\'"]', combined):
            return False

    # Default: inject (safer to include than exclude)
    return True

def format_tier1_prompt(sample, use_ddl_override: bool = False) -> str:
    """Build enhanced prompt with all features (no dropout). Used at inference time."""
    if ENABLE_MSCHEMA and not use_ddl_override:
        schema_str = get_mschema(sample.db_id, sample.schema_ddl)
    else:
        schema_str = sample.schema_ddl[:4500]

    table_summary = build_table_column_summary(sample.schema_ddl)

    value_hints = ""
    if ENABLE_VALUE_HINTS and sample.db_path:
        value_hints = sample_db_values_question_aware(
            sample.db_path, sample.schema_ddl,
            sample.arabic_question, max_per_col=VALUE_HINT_MAX_PER_COL)

    column_hints = ""
    if ENABLE_COLUMN_LINKING:
        column_hints = link_columns_to_question(
            sample.db_id, sample.schema_ddl,
            sample.arabic_question,
            db_path=sample.db_path,
            top_k=COL_LINK_TOP_K)

    matched_values = ""
    if ENABLE_VALUE_INJECTION and sample.db_path:
        if _question_has_text_entities(sample.arabic_question, sample.english_question):
            matched_values = find_question_matched_values(
                sample.db_id, sample.db_path, sample.schema_ddl,
                sample.arabic_question, sample.english_question,
                top_k=VALUE_INJECTION_TOP_K, threshold=VALUE_INJECTION_THRESHOLD)

    sample_rows = ""
    if ENABLE_SAMPLE_ROWS and sample.db_path:
        sample_rows = get_sample_rows(
            sample.db_path, sample.schema_ddl, column_hints,
            limit=SAMPLE_ROWS_LIMIT, max_tables=SAMPLE_ROWS_MAX_TABLES)

    english_hint = ""
    if sample.english_question:
        english_hint = f"(English: {sample.english_question})\n"

    return TIER1_PROMPT.format(
        schema=schema_str, value_hints=value_hints,
        matched_values=matched_values,
        column_hints=column_hints,
        sample_rows=sample_rows,
        table_summary=table_summary, question=sample.arabic_question,
        english_hint=english_hint)


# Training prompt builder (with feature dropout)

import random

def format_training_prompt(sample, dropout_rate: float = 0.20) -> str:
    """
    Build training prompt ALIGNED with inference format, but with
    random feature dropout for robustness.

    Each feature is independently dropped with probability = dropout_rate.
    This teaches the model to USE features when present but not DEPEND
    on any single one — graceful degradation if a hint is noisy or absent.
    """
    # Schema: M-Schema or fallback to DDL
    if ENABLE_MSCHEMA and random.random() >= dropout_rate:
        schema_str = get_mschema(sample.db_id, sample.schema_ddl)
    else:
        schema_str = sample.schema_ddl[:4500]

    table_summary = build_table_column_summary(sample.schema_ddl)

    # Value hints
    value_hints = ""
    if (ENABLE_VALUE_HINTS and sample.db_path
            and random.random() >= dropout_rate):
        value_hints = sample_db_values_question_aware(
            sample.db_path, sample.schema_ddl,
            sample.arabic_question, max_per_col=VALUE_HINT_MAX_PER_COL)

    # Column linking (requires embedding model)
    column_hints = ""
    if (ENABLE_COLUMN_LINKING and _emb_model is not None
            and random.random() >= 0.40):
        column_hints = link_columns_to_question(
            sample.db_id, sample.schema_ddl,
            sample.arabic_question,
            db_path=sample.db_path,
            top_k=COL_LINK_TOP_K)

    # Question-matched value injection (requires embedding model)
    matched_values = ""
    if (ENABLE_VALUE_INJECTION and sample.db_path
            and _emb_model is not None
            and random.random() >= dropout_rate
            and _question_has_text_entities(sample.arabic_question, sample.english_question)):
        matched_values = find_question_matched_values(
            sample.db_id, sample.db_path, sample.schema_ddl,
            sample.arabic_question, sample.english_question,
            top_k=VALUE_INJECTION_TOP_K, threshold=VALUE_INJECTION_THRESHOLD)

    # Sample rows
    sample_rows = ""
    if (ENABLE_SAMPLE_ROWS and sample.db_path
            and random.random() >= dropout_rate):
        sample_rows = get_sample_rows(
            sample.db_path, sample.schema_ddl, column_hints,
            limit=SAMPLE_ROWS_LIMIT, max_tables=SAMPLE_ROWS_MAX_TABLES)

    english_hint = ""
    if sample.english_question:
        english_hint = f"(English: {sample.english_question})\n"

    return TIER1_PROMPT.format(
        schema=schema_str, value_hints=value_hints,
        matched_values=matched_values,
        column_hints=column_hints,
        sample_rows=sample_rows,
        table_summary=table_summary, question=sample.arabic_question,
        english_hint=english_hint)


def prewarm_column_cache():
    """Pre-compute column embeddings for all databases."""
    if _emb_model is None:
        print("Embedding model not loaded — skipping column cache")
        return
    count = 0
    for db_id, schema_ddl in ar_spider_schemas.items():
        _get_column_embeddings(db_id, schema_ddl)
        count += 1
    total_cols = sum(len(v['candidates']) for v in _col_emb_cache.values())
    print(f"Column embeddings cached for {count} databases ({total_cols} columns)")


def prewarm_value_cache():
    """Pre-compute value embeddings for all databases with SQLite files."""
    if _emb_model is None:
        print("Embedding model not loaded — skipping value cache")
        return
    count = 0
    total_vals = 0
    for db_id, schema_ddl in ar_spider_schemas.items():
        db_path = db_paths.get(db_id, "")
        if db_path:
            data = _build_value_embeddings(db_id, db_path, schema_ddl)
            total_vals += len(data["values"])
            count += 1
    print(f"Value embeddings cached for {count} databases ({total_vals} unique values)")


print("Shared functions defined (M-Schema, column linking, value hints, "
      "sample rows, prompt builders)")

## Section C: Pre-Training Resource Loading

In [ ]:
#@title Pre-Training Resource Loading { display-mode: "form" }

from google.colab import drive
drive.mount('/content/drive')

# 1. Load Arabic column descriptions from cache

if ENABLE_AR_COL_DESC and os.path.exists(AR_COL_DESC_PATH):
    try:
        with open(AR_COL_DESC_PATH, "r", encoding="utf-8") as f:
            _ar_col_descriptions = json.load(f)
        total_descs = sum(len(v) for v in _ar_col_descriptions.values())
        print(f"Loaded {total_descs} Arabic column descriptions from cache")
    except Exception as e:
        print(f"Failed to load column descriptions: {e}")
        print(f"   Column linking will work without Arabic descriptions.")
elif ENABLE_AR_COL_DESC:
    print(f"Arabic column descriptions not found at: {AR_COL_DESC_PATH}")
    print(f"   They will be generated later using the fine-tuned model.")
    print(f"   For now, training prompts will use English-only column descriptions.")

# 2. Load Arabic table glosses from cache

if ENABLE_BILINGUAL_SCHEMA and os.path.exists(AR_TABLE_GLOSS_PATH):
    try:
        with open(AR_TABLE_GLOSS_PATH, "r", encoding="utf-8") as f:
            _ar_table_glosses = json.load(f)
        total_glosses = sum(len(v) for v in _ar_table_glosses.values())
        print(f"Loaded {total_glosses} Arabic table glosses from cache")
    except Exception as e:
        print(f"Failed to load table glosses: {e}")
elif ENABLE_BILINGUAL_SCHEMA:
    print(f"Table glosses not found at: {AR_TABLE_GLOSS_PATH}")
    print(f"   M-Schema will use English-only table names for training.")

# 3. Load embedding model

TRAIN_LOAD_EMBEDDINGS = True  #@param {type:"boolean"}

if TRAIN_LOAD_EMBEDDINGS and (ENABLE_COLUMN_LINKING or ENABLE_VALUE_INJECTION):
    from sentence_transformers import SentenceTransformer
    print("Loading BGE-M3 embedding model for training prompt construction...")
    _emb_model = SentenceTransformer("BAAI/bge-m3", device="cuda")
    print(f"BGE-M3 loaded ({_emb_model.get_sentence_embedding_dimension()}d, ~2.3GB VRAM)")

    # Pre-warm caches (one-time cost, ~2-5 min)
    print("Pre-warming embedding caches for all databases...")
    prewarm_column_cache()
    prewarm_value_cache()
else:
    print(" Embedding model not loaded for training.")
    print("   Column linking and value injection will be SKIPPED in training prompts.")
    print("   Other features (M-Schema, value hints, English hint, sample rows) still active.")

print()
print("Pre-training resources ready. Training prompts will use:")
print(f"   M-Schema with bilingual annotations : {'OK' if ENABLE_MSCHEMA and _ar_table_glosses else 'English-only' if ENABLE_MSCHEMA else 'FAIL'}")
print(f"   Column linking (embedding)          : {'OK' if _emb_model is not None and ENABLE_COLUMN_LINKING else 'FAIL'}")
print(f"   Arabic column descriptions          : {'OK' if _ar_col_descriptions else 'FAIL'}")
print(f"   Value hints from DB                 : {'OK' if ENABLE_VALUE_HINTS else 'FAIL'}")
print(f"   Question-matched value injection    : {'OK' if _emb_model is not None and ENABLE_VALUE_INJECTION else 'FAIL'}")
print(f"   Sample rows                         : {'OK' if ENABLE_SAMPLE_ROWS else 'FAIL'}")
print(f"   English translation hint            : {'OK' if ENABLE_ENGLISH_HINT else 'FAIL'}")
print(f"   Feature dropout rate                : {FEATURE_DROPOUT_RATE}")

## Translate Training Questions to English

## Section D: Load train.json → Training Data (Prompt-Aligned)

In [ ]:
#@title Translate Training Questions (Google Translate + Cache) { display-mode: "form" }

TRAIN_ENGLISH_CACHE = "/content/drive/MyDrive/ar_spider_train_english.json"  #@param {type:"string"}
REGENERATE_TRAIN_TRANSLATIONS = False  #@param {type:"boolean"}

def _is_arabic(text: str) -> bool:
    """Check if text contains Arabic characters."""
    if not text:
        return False
    arabic_chars = sum(1 for c in text if '\u0600' <= c <= '\u06FF' or '\u0750' <= c <= '\u077F' or '\uFB50' <= c <= '\uFDFF' or '\uFE70' <= c <= '\uFEFF')
    return arabic_chars > len(text) * 0.2

_translator = None

def _load_translator():
    global _translator
    if _translator is not None:
        return
    try:
        from deep_translator import GoogleTranslator
    except ImportError:
        import subprocess
        subprocess.run(["pip", "install", "-q", "deep-translator"], check=True)
        from deep_translator import GoogleTranslator
    _translator = GoogleTranslator(source='ar', target='en')
    print("Google Translate loaded")


def _translate_arabic_to_english(arabic_text: str) -> str:
    """Translate Arabic to English using Google Translate."""
    try:
        result = _translator.translate(arabic_text)
        return result.strip() if result and len(result.strip()) > 3 else ""
    except Exception:
        import time
        time.sleep(0.5)
        try:
            result = _translator.translate(arabic_text)
            return result.strip() if result and len(result.strip()) > 3 else ""
        except Exception:
            return ""

print("Translation functions defined")
print("   Translations will be applied after training data is loaded (Section D)")

In [ ]:
def _is_arabic(text: str) -> bool:
    """Check if text contains Arabic characters."""
    if not text:
        return False
    arabic_chars = sum(1 for c in text if '\u0600' <= c <= '\u06FF' or '\u0750' <= c <= '\u077F' or '\uFB50' <= c <= '\uFDFF' or '\uFE70' <= c <= '\uFEFF')
    return arabic_chars > len(text) * 0.2


#@title Load train.json → Training Data (Prompt-Aligned) { display-mode: "form" }

print("Loading Ar-Spider train.json...")
with open(os.path.join(AR_SPIDER_DIR, "train.json"), "r", encoding="utf-8") as f:
    train_raw = json.load(f)
print(f"   Raw samples: {len(train_raw)}")

# Build training samples as EvalSample objects
print(f"\n Building training data...")
if TRAIN_USE_ALIGNED_PROMPT:
    print(f"   PROMPT ALIGNMENT ON — using Tier 1 format with feature dropout ({FEATURE_DROPOUT_RATE})")
else:
    print(f"    Prompt alignment OFF — using base format (DDL + table summary)")

training_data = []
train_samples = []  # EvalSample objects for preview
skipped = {"no_schema": 0, "no_sql": 0}

random.seed(42)  # reproducible dropout for this construction pass

for idx, r in enumerate(train_raw):
    db_id = r["db_id"]
    schema = ar_spider_schemas.get(db_id, "")
    arabic_question = r.get("Arabic", r.get("question", ""))
    _raw_q = r.get("question", "")
    english_question = "" if _is_arabic(_raw_q) else _raw_q
    sql = r.get("query", "").strip()

    if not schema:
        skipped["no_schema"] += 1
        continue
    if not sql or not re.search(r'\bSELECT\b', sql, re.I):
        skipped["no_sql"] += 1
        continue

    # Build EvalSample for this training example
    sample = EvalSample(
        id=f"train_{idx:04d}",
        db_id=db_id,
        arabic_question=arabic_question,
        gold_sql=sql,
        schema_ddl=schema,
        db_path=db_paths.get(db_id, ""),
        english_question=english_question,
    )
    train_samples.append(sample)

    # Build prompt — aligned with inference or base format
    if TRAIN_USE_ALIGNED_PROMPT:
        prompt = format_training_prompt(sample, dropout_rate=FEATURE_DROPOUT_RATE)
    else:
        prompt = format_prompt(schema, arabic_question)

    training_data.append({
        "input": prompt,
        "output": f"```sql\n{sql}\n```",
        "sql": sql,
        "question": arabic_question,
        "db_id": db_id,
    })

    # Progress
    if (idx + 1) % 1000 == 0:
        print(f"   [{idx + 1}/{len(train_raw)}] processed...")

print(f"\n Training set: {len(training_data)} examples")
print(f"   Skipped: {skipped['no_schema']} no schema, {skipped['no_sql']} no SQL")
print(f"   Unique databases: {len(set(d['db_id'] for d in training_data))}")

# Show samples
for i, s in enumerate(training_data[:3]):
    print(f"\n   [{i+1}] Q: {s['question'][:70]}")
    print(f"       SQL: {s['sql'][:70]}")

# Token length statistics
try:
    sample_lengths = []
    for d in training_data[:200]:
        # Rough estimate: 1 token ≈ 2.5 chars for mixed Arabic/English
        est_tokens = len(d["input"]) / 2.5 + len(d["output"]) / 2.5
        sample_lengths.append(est_tokens)
    avg_len = sum(sample_lengths) / len(sample_lengths)
    max_len = max(sample_lengths)
    over_limit = sum(1 for l in sample_lengths if l > MAX_SEQ_LENGTH_7B)
    print(f"\n   Estimated token lengths (first 200):")
    print(f"      Avg: {avg_len:.0f}, Max: {max_len:.0f}")
    print(f"      Over MAX_SEQ_LENGTH ({MAX_SEQ_LENGTH_7B}): {over_limit}/{len(sample_lengths)}")
    if over_limit > 0:
        print(f"       {over_limit} examples may be truncated. Consider MAX_SEQ_LENGTH_7B = 4096.")
except Exception:
    pass

In [ ]:
#@title Apply English Translations to Training Data { display-mode: "form" }

_train_english_cache = {}

if ENABLE_ENGLISH_HINT:
    # Try cache first
    cache_hit = False
    if os.path.exists(TRAIN_ENGLISH_CACHE) and not REGENERATE_TRAIN_TRANSLATIONS:
        try:
            with open(TRAIN_ENGLISH_CACHE, "r", encoding="utf-8") as f:
                _train_english_cache = json.load(f)
            print(f"Loaded {len(_train_english_cache)} cached training translations")
            cache_hit = True
        except Exception as e:
            print(f"Cache load failed: {e}")

    # Apply cached translations
    applied = 0
    missing_ids = []
    for s in train_samples:
        if s.id in _train_english_cache:
            eng = _train_english_cache[s.id]
            if eng and not _is_arabic(eng):
                s.english_question = eng
                applied += 1
            else:
                s.english_question = ""
                missing_ids.append(s)
        else:
            missing_ids.append(s)

    if cache_hit:
        print(f"   Applied {applied}/{len(train_samples)} from cache")

    # Translate missing samples
    if missing_ids:
        _load_translator()
        print(f"Translating {len(missing_ids)} missing training questions...")
        for i, s in enumerate(missing_ids):
            eng = _translate_arabic_to_english(s.arabic_question)
            if eng and not _is_arabic(eng):
                s.english_question = eng
                _train_english_cache[s.id] = eng
                applied += 1
            else:
                s.english_question = ""
                _train_english_cache[s.id] = ""
            if (i + 1) % 500 == 0:
                print(f"   [{i+1}/{len(missing_ids)}]")

        # Save updated cache
        os.makedirs(os.path.dirname(TRAIN_ENGLISH_CACHE), exist_ok=True)
        with open(TRAIN_ENGLISH_CACHE, "w", encoding="utf-8") as f:
            json.dump(_train_english_cache, f, ensure_ascii=False, indent=2)
        print(f"   Cache saved: {TRAIN_ENGLISH_CACHE}")

    print(f"\n English translations applied: {applied}/{len(train_samples)}")

    # Show examples
    shown = 0
    for s in train_samples[:50]:
        if s.english_question and shown < 5:
            print(f"   {s.id}: AR: {s.arabic_question[:55]}")
            print(f"         EN: {s.english_question[:55]}")
            shown += 1
else:
    # Clear any Arabic text that leaked into english_question
    for s in train_samples:
        if _is_arabic(s.english_question):
            s.english_question = ""
    print(" English hints disabled")

print(f"\n Rebuilding training prompts with correct English translations...")
random.seed(42)
rebuilt = 0
for i, (td, s) in enumerate(zip(training_data, train_samples)):
    if TRAIN_USE_ALIGNED_PROMPT:
        td["input"] = format_training_prompt(s, dropout_rate=FEATURE_DROPOUT_RATE)
    rebuilt += 1
print(f"Rebuilt {rebuilt} training prompts")

# Verify no Arabic in English hints
arabic_in_english = 0
for td in training_data[:200]:
    m = re.search(r'\(English: (.+?)\)', td["input"])
    if m and _is_arabic(m.group(1)):
        arabic_in_english += 1
if arabic_in_english > 0:
    print(f"WARNING: {arabic_in_english}/200 prompts still have Arabic in English hint!")
else:
    print(f"Verified: 0/200 prompts have Arabic in English hint")

## Section E: Preview — What the Training Model Sees

In [ ]:
#@title Preview: What the Training Model Sees { display-mode: "form" }

TRAIN_PREVIEW_IDS = [0, 50, 150]  #@param {type:"raw"}

print("=" * 80)
print("  TRAINING PROMPT PREVIEW — What the model learns from")
print("=" * 80)

for sample_idx in TRAIN_PREVIEW_IDS:
    if sample_idx >= len(train_samples):
        continue
    sample = train_samples[sample_idx]

    # Build a FRESH prompt (with dropout) to show what training sees
    random.seed(sample_idx)  # deterministic for preview
    if TRAIN_USE_ALIGNED_PROMPT:
        prompt = format_training_prompt(sample, dropout_rate=FEATURE_DROPOUT_RATE)
    else:
        prompt = format_prompt(sample.schema_ddl, sample.arabic_question)

    # Estimate token count
    est_tokens = int(len(prompt) / 2.5)

    print(f"\n{'━' * 80}")
    print(f"  Sample: {sample.id} | {sample.db_id} | ~{est_tokens} est. tokens")
    print(f"  Question (AR): {sample.arabic_question[:100]}")
    print(f"  Question (EN): {sample.english_question[:100]}")
    print(f"  Gold SQL:      {sample.gold_sql[:100]}")
    print(f"{'━' * 80}")

    for line in prompt.split("\n"):
        if line.strip().startswith("★"):
            print(f"  \033[92m{line}\033[0m")  # green: column hints
        elif line.strip().startswith("→"):
            print(f"  \033[95m{line}\033[0m")  # magenta: value injection
        elif line.strip().startswith("Likely relevant"):
            print(f"  \033[92m{line}\033[0m")
        elif line.strip().startswith("Detected values"):
            print(f"  \033[95m{line}\033[0m")
        elif line.strip().startswith("Sample column values"):
            print(f"  \033[93m{line}\033[0m")  # yellow: value hints
        elif line.strip().startswith("Sample data"):
            print(f"  \033[93m{line}\033[0m")
        elif line.strip().startswith("Question:"):
            print(f"  \033[96m{line}\033[0m")  # cyan: question
        elif "(English:" in line:
            print(f"  \033[94m{line}\033[0m")  # blue: English hint
        elif line.strip().startswith("【"):
            print(f"  \033[91m{line}\033[0m")  # red: M-Schema tables
        else:
            print(f"  {line}")

    # Show the target (SQL output)
    print(f"\n  \033[1m  TARGET OUTPUT:\033[0m")
    print(f"  ```sql")
    print(f"  {sample.gold_sql}")
    print(f"  ```")
    print()

In [ ]:
#@title Training Prompt Feature Statistics { display-mode: "form" }

# Feature presence statistics across training data
print("\n" + "=" * 80)
print("  TRAINING PROMPT FEATURE STATISTICS (sampled from first 200 examples)")
print("=" * 80)

feature_counts = {
    "M-Schema (【)": 0,
    "Arabic table gloss": 0,
    "Arabic column desc": 0,
    "Value hints": 0,
    "Column linking (★)": 0,
    "Value injection (→)": 0,
    "Sample rows": 0,
    "English hint": 0,
}

n_check = min(200, len(training_data))
for i in range(n_check):
    p = training_data[i]["input"]
    if "【" in p:                           feature_counts["M-Schema (【)"] += 1
    if "(" in p and any(  # check for Arabic gloss after 【table】
        re.search(r'【\w+】\s*\(', p)
        for _ in [1]):                      feature_counts["Arabic table gloss"] += 1
    if "—" in p and "★" in p:              feature_counts["Arabic column desc"] += 1
    if "Sample column values" in p:         feature_counts["Value hints"] += 1
    if "★" in p:                           feature_counts["Column linking (★)"] += 1
    if "→" in p and "Detected values" in p: feature_counts["Value injection (→)"] += 1
    if "Sample data" in p:                  feature_counts["Sample rows"] += 1
    if "(English:" in p:                    feature_counts["English hint"] += 1

print(f"  Checked {n_check} training examples:")
for feat, count in feature_counts.items():
    pct = 100 * count / n_check
    expected = 100 * (1 - FEATURE_DROPOUT_RATE)
    bar = "█" * int(pct / 2) + "░" * (50 - int(pct / 2))
    print(f"    {feat:<25s}: {count:>4d}/{n_check} ({pct:>5.1f}%) {bar}  [expected ~{expected:.0f}%]")

print(f"\n  Each feature should appear in ~{100*(1-FEATURE_DROPOUT_RATE):.0f}% of examples "
      f"(dropout={FEATURE_DROPOUT_RATE})")
print(f"  The model learns to USE features when present but not DEPEND on any single one.")

In [ ]:
#@title Prompt Comparison: Old vs New { display-mode: "form" }

# Side-by-side comparison: old vs new prompt
print("\n" + "=" * 80)
print("  PROMPT COMPARISON: Old (base) vs New (aligned)")
print("=" * 80)

if train_samples:
    compare_sample = train_samples[0]

    old_prompt = format_prompt(compare_sample.schema_ddl, compare_sample.arabic_question)
    random.seed(999)
    new_prompt = format_training_prompt(compare_sample, dropout_rate=0.0)  # no dropout for comparison

    old_chars = len(old_prompt)
    new_chars = len(new_prompt)

    print(f"\n  Sample: {compare_sample.id} | {compare_sample.db_id}")
    print(f"  Question: {compare_sample.arabic_question[:80]}")
    print(f"\n  Old prompt: {old_chars:,} chars (~{old_chars//3} tokens)")
    print(f"  New prompt: {new_chars:,} chars (~{new_chars//3} tokens)")
    print(f"  Expansion:  {new_chars/max(old_chars,1):.1f}x")
    print(f"\n  ── Old prompt features ──")
    print(f"    DDL schema     : {'OK' if 'CREATE TABLE' in old_prompt else 'FAIL'}")
    print(f"    M-Schema       : {'OK' if '【' in old_prompt else 'FAIL'}")
    print(f"    Table summary  : {'OK' if '•' in old_prompt else 'FAIL'}")
    print(f"    Value hints    : {'OK' if 'Sample column' in old_prompt else 'FAIL'}")
    print(f"    Column linking : {'OK' if '★' in old_prompt else 'FAIL'}")
    print(f"    Value injection: {'OK' if 'Detected values' in old_prompt else 'FAIL'}")
    print(f"    Sample rows    : {'OK' if 'Sample data' in old_prompt else 'FAIL'}")
    print(f"    English hint   : {'OK' if '(English:' in old_prompt else 'FAIL'}")

    print(f"\n  ── New prompt features ──")
    print(f"    DDL schema     : {'OK' if 'CREATE TABLE' in new_prompt else 'FAIL'}")
    print(f"    M-Schema       : {'OK' if '【' in new_prompt else 'FAIL'}")
    print(f"    Table summary  : {'OK' if '•' in new_prompt else 'FAIL'}")
    print(f"    Value hints    : {'OK' if 'Sample column' in new_prompt else 'FAIL'}")
    print(f"    Column linking : {'OK' if '★' in new_prompt else 'FAIL'}")
    print(f"    Value injection: {'OK' if 'Detected values' in new_prompt else 'FAIL'}")
    print(f"    Sample rows    : {'OK' if 'Sample data' in new_prompt else 'FAIL'}")
    print(f"    English hint   : {'OK' if '(English:' in new_prompt else 'FAIL'}")

print("\n Training preview complete — prompts are aligned with inference format.")

## Load dev.json → Evaluation Data

In [ ]:
#@title Load dev.json → Evaluation Data { display-mode: "form" }

print("Loading Ar-Spider dev.json...")
with open(os.path.join(AR_SPIDER_DIR, "dev.json"), "r", encoding="utf-8") as f:
    dev_raw = json.load(f)
print(f"   Raw samples: {len(dev_raw)}")

@dataclass
class EvalSample:
    id: str
    db_id: str
    arabic_question: str
    gold_sql: str
    schema_ddl: str
    db_path: str = ""
    english_question: str = ""

eval_samples = []
for idx, r in enumerate(dev_raw):
    db_id = r["db_id"]
    schema = ar_spider_schemas.get(db_id, "")
    if not schema:
        continue
    eval_samples.append(EvalSample(
        id=f"ar_{idx:04d}",
        db_id=db_id,
        arabic_question=r.get("Arabic", r.get("question", "")),
        gold_sql=r.get("query", ""),
        schema_ddl=schema,
        db_path=db_paths.get(db_id, ""),
        english_question=r.get("question", ""),
    ))

print(f"Evaluation set: {len(eval_samples)} samples (full dev)")
print(f"   Unique databases: {len(set(s.db_id for s in eval_samples))}")
eng_count = sum(1 for s in eval_samples if s.english_question)
print(f"   English questions: {eng_count}/{len(eval_samples)} ({'OK' if eng_count == len(eval_samples) else 'WARN'})")
if eval_samples:
    s = eval_samples[0]
    print(f"   Sample: AR: {s.arabic_question[:60]}")
    print(f"           EN: {s.english_question[:60]}")
print(f"\n   ZERO overlap with training data (train.json ≠ dev.json)")

## Load Model + Apply QLoRA

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

print(f"Loading {FINETUNE_7B_BASE}...")

try:
    tokenizer_7b = AutoTokenizer.from_pretrained(FINETUNE_7B_BASE, trust_remote_code=True)
except Exception:
    tokenizer_7b = AutoTokenizer.from_pretrained(FINETUNE_7B_BASE, trust_remote_code=True, use_fast=False)

if tokenizer_7b.pad_token is None:
    eot_id = tokenizer_7b.convert_tokens_to_ids("<|endoftext|>")
    if eot_id != tokenizer_7b.unk_token_id:
        tokenizer_7b.pad_token = "<|endoftext|>"
    else:
        tokenizer_7b.pad_token = tokenizer_7b.eos_token
tokenizer_7b.padding_side = "left"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)
model_7b = AutoModelForCausalLM.from_pretrained(
    FINETUNE_7B_BASE, quantization_config=bnb_config,
    device_map="auto", trust_remote_code=True,
)
model_7b = prepare_model_for_kbit_training(model_7b, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=LORA_RANK_7B, lora_alpha=LORA_ALPHA_7B,
    lora_dropout=0.05, bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)
model_7b = get_peft_model(model_7b, lora_config)

trainable = sum(p.numel() for p in model_7b.parameters() if p.requires_grad)
total = sum(p.numel() for p in model_7b.parameters())
print(f"Model loaded with QLoRA")
print(f"   Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

## Section F: Prepare Training Dataset (Prompt-Aligned)

In [ ]:
#@title Prepare Training Dataset (Prompt-Aligned) { display-mode: "form" }

from datasets import Dataset

print(f"Formatting {len(training_data)} training examples...")

formatted_7b = []
skipped_long = 0

for ex in training_data:
    messages = [
        {"role": "system", "content": TIER1_SYSTEM},  # ← aligned system prompt
        {"role": "user", "content": ex["input"]},
        {"role": "assistant", "content": ex["output"]},
    ]
    try:
        token_ids = tokenizer_7b.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=False,
        )
        if len(token_ids) <= MAX_SEQ_LENGTH_7B:
            text = tokenizer_7b.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=False,
            )
            formatted_7b.append({"text": text})
        else:
            skipped_long += 1
    except Exception:
        text = (
            f"<|im_start|>system\n{TIER1_SYSTEM}<|im_end|>\n"
            f"<|im_start|>user\n{ex['input']}<|im_end|>\n"
            f"<|im_start|>assistant\n{ex['output']}<|im_end|>"
        )
        if len(text) / 2.5 <= MAX_SEQ_LENGTH_7B:
            formatted_7b.append({"text": text})
        else:
            skipped_long += 1

print(f"Ready: {len(formatted_7b)} examples (skipped {skipped_long} too long)")

if skipped_long > len(training_data) * 0.05:
    print(f"    {skipped_long} examples ({100*skipped_long/len(training_data):.1f}%) "
          f"exceeded MAX_SEQ_LENGTH={MAX_SEQ_LENGTH_7B}")
    print(f"   The aligned prompts are longer. Consider increasing MAX_SEQ_LENGTH_7B.")
    print(f"      Current: {MAX_SEQ_LENGTH_7B} → Suggested: 4096 (if not already)")

full_dataset = Dataset.from_list(formatted_7b)

# Hold out 5% for overfitting detection
split = full_dataset.train_test_split(test_size=0.05, seed=42)
train_ds = split["train"]
val_ds = split["test"]
print(f"   Train: {len(train_ds)} | Val: {len(val_ds)} (5% held out for loss monitoring)")

# Token length distribution of actual formatted examples
print(f"\n   Actual token lengths (chat-formatted):")
sample_lens = []
for ex in formatted_7b[:300]:
    try:
        tids = tokenizer_7b.encode(ex["text"])
        sample_lens.append(len(tids))
    except:
        pass
if sample_lens:
    print(f"      Min: {min(sample_lens)}, Max: {max(sample_lens)}, "
          f"Avg: {sum(sample_lens)//len(sample_lens)}")
    print(f"      Median: {sorted(sample_lens)[len(sample_lens)//2]}")
    p95 = sorted(sample_lens)[int(len(sample_lens)*0.95)]
    print(f"      P95: {p95}  {'OK' if p95 <= MAX_SEQ_LENGTH_7B else '> MAX_SEQ_LENGTH'}")

## Train (QLoRA Fine-Tuning)

In [ ]:
from trl import SFTTrainer, SFTConfig
'''
training_args = SFTConfig(
    output_dir=str(ADAPTER_7B_OUTPUT),
    num_train_epochs=NUM_EPOCHS_7B,
    per_device_train_batch_size=PER_DEVICE_BATCH_7B,
    gradient_accumulation_steps=GRAD_ACCUM_7B,
    learning_rate=LEARNING_RATE_7B,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    weight_decay=0.01,
    bf16=True, fp16=False,
    max_grad_norm=1.0,
    logging_steps=25,
    save_strategy="epoch",
    save_total_limit=2,
    max_length=MAX_SEQ_LENGTH_7B,
    dataset_text_field="text",
    report_to="none",
    seed=42,
    eval_strategy="epoch",
    per_device_eval_batch_size=PER_DEVICE_BATCH_7B,
)

trainer = SFTTrainer(
    model=model_7b, processing_class=tokenizer_7b,
    train_dataset=train_ds, eval_dataset=val_ds,
    args=training_args,
)

print(f"Starting QLoRA fine-tuning on train.json...")
print(f"   Base: {FINETUNE_7B_BASE}")
print(f"   Prompt: {'Enhanced' if USE_ENHANCED_PROMPT else 'Base'}")
print(f"   Data: {len(train_ds)} train + {len(val_ds)} val")
print(f"   Epochs: {NUM_EPOCHS_7B}")
print(f"   Effective batch: {PER_DEVICE_BATCH_7B * GRAD_ACCUM_7B}")
print(f"   LoRA: rank={LORA_RANK_7B}, alpha={LORA_ALPHA_7B}")
print(f"   LR: {LEARNING_RATE_7B}")
print(f"   Max seq length: {MAX_SEQ_LENGTH_7B}")
print()

trainer.train()

# Save
model_7b.save_pretrained(str(ADAPTER_7B_OUTPUT))
tokenizer_7b.save_pretrained(str(ADAPTER_7B_OUTPUT))
print(f"\n Adapter saved → {ADAPTER_7B_OUTPUT}")

# Log metrics
if trainer.state.log_history:
    tl = [h["loss"] for h in trainer.state.log_history if "loss" in h]
    el = [h["eval_loss"] for h in trainer.state.log_history if "eval_loss" in h]
    if tl: print(f"   Initial loss: {tl[0]:.4f} → Final: {tl[-1]:.4f}")
    if el:
        print(f"   Val losses: {' → '.join(f'{l:.4f}' for l in el)}")
        if len(el) > 1 and el[-1] > el[-2]:
            print(f"    Val loss increased — possible overfitting")
'''

## Save Adapter & Tokenizer to Google Drive

In [ ]:
#@title Save Adapter & Tokenizer to Google Drive { display-mode: "form" }
'''
import shutil

DRIVE_SAVE_DIR = "/content/drive/MyDrive/arabic_text2sql_7b_adapter_v5_aligned"  #@param {type:"string"}
ADAPTER_PATH = str(ADAPTER_7B_OUTPUT)

# Validate adapter exists locally
assert os.path.exists(os.path.join(ADAPTER_PATH, "adapter_config.json")), \
    f"Adapter not found at {ADAPTER_PATH}"

os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

# Copy all adapter files
print(f"Source: {ADAPTER_PATH}")
print(f"Destination: {DRIVE_SAVE_DIR}\n")

count = 0
total_size = 0
for f in os.listdir(ADAPTER_PATH):
    src = os.path.join(ADAPTER_PATH, f)
    if os.path.isfile(src):
        dst = os.path.join(DRIVE_SAVE_DIR, f)
        shutil.copy2(src, dst)
        size_mb = os.path.getsize(src) / 1024 / 1024
        total_size += size_mb
        print(f"   {f} ({size_mb:.1f} MB)")
        count += 1

# Verify key files made it
for required in ["adapter_config.json", "adapter_model.safetensors"]:
    assert os.path.exists(os.path.join(DRIVE_SAVE_DIR, required)), \
        f"Missing {required} in destination"

print(f"\n Saved {count} files ({total_size:.1f} MB total) to Google Drive")
print(f"   Path: {DRIVE_SAVE_DIR}")
'''

## Free Training Memory

In [ ]:
#@title Free Training Memory { display-mode: "form" }

try:
  del trainer
except:
  print('Cannot delete trainer')

try:
  del model_7b
except:
  print('Cannot delete model_7b')

try:
  del train_ds
except:
  print('Cannot delete train_ds')

try:
  del val_ds
except:
  print('Cannot delete val_ds')

try:
  del full_dataset
except:
  print('Cannot delete full_dataset')


gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"   GPU after cleanup: {torch.cuda.memory_allocated()/1024**3:.1f}GB")
print("Training memory freed — ready for evaluation")

try:
  del base_model
except:
  print('Cannot delete base_model')
try:
  del infer_model
except:
  print('Cannot delete infer_model')
try:
  del infer_tok
except:
  print('Cannot delete base_model')

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"   GPU after cleanup: {torch.cuda.memory_allocated()/1024**3:.1f}GB")
print("Training memory freed — ready for evaluation")

## Load Adapter from Google Drive (Inference)

In [ ]:
#@title Load Adapter from Google Drive { display-mode: "form" }

from google.colab import drive
drive.mount('/content/drive')

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import torch

DRIVE_SAVE_DIR = "/content/drive/MyDrive/arabic_text2sql_7b_adapter_v5_aligned"  #@param {type:"string"}
BASE_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"

# Load base model in 4-bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
)

# Load adapter from Drive
infer_model = PeftModel.from_pretrained(base_model, DRIVE_SAVE_DIR)
infer_model.eval()

# Load tokenizer
infer_tok = AutoTokenizer.from_pretrained(DRIVE_SAVE_DIR)
if infer_tok.pad_token is None:
    infer_tok.pad_token = infer_tok.eos_token

print(f"Model loaded: {BASE_MODEL} + adapter from Google Drive")
print(f"   Adapter: {DRIVE_SAVE_DIR}")

## Load Embedding Model for Column Linking (Inference)

In [ ]:
#@title Load Embedding Model for Column Linking { display-mode: "form" }

from sentence_transformers import SentenceTransformer
import numpy as np

print("Loading BGE-M3 embedding model for column linking...")
_emb_model = SentenceTransformer(
    "BAAI/bge-m3",
    device="cuda",
)
print(f"BGE-M3 loaded ({_emb_model.get_sentence_embedding_dimension()}d, ~2.3GB VRAM)")

# Pre-warm caches for inference
print("Pre-warming embedding caches...")
_col_emb_cache.clear()
_db_value_emb_cache.clear()
prewarm_column_cache()
prewarm_value_cache()
print("Embedding caches ready for inference")

## Load Arabic Descriptions + Table Glosses (Inference)

In [ ]:
#@title Load Arabic Descriptions + Table Glosses { display-mode: "form" }

# Load Arabic column descriptions from cache
if ENABLE_AR_COL_DESC and os.path.exists(AR_COL_DESC_PATH):
    with open(AR_COL_DESC_PATH, "r", encoding="utf-8") as f:
        _ar_col_descriptions = json.load(f)
    total_descs = sum(len(v) for v in _ar_col_descriptions.values())
    print(f"Loaded {total_descs} Arabic column descriptions from cache")
else:
    print(f"Arabic column descriptions not found \u2014 will generate using model")

# Load Arabic table glosses from cache
if ENABLE_BILINGUAL_SCHEMA and os.path.exists(AR_TABLE_GLOSS_PATH):
    with open(AR_TABLE_GLOSS_PATH, "r", encoding="utf-8") as f:
        _ar_table_glosses = json.load(f)
    total_glosses = sum(len(v) for v in _ar_table_glosses.values())
    print(f"Loaded {total_glosses} Arabic table glosses from cache")
else:
    print(f"Table glosses not found \u2014 M-Schema will use English-only names")

# Rebuild column embeddings with bilingual text
if ENABLE_BILINGUAL_EMBEDDINGS and _ar_col_descriptions and ENABLE_COLUMN_LINKING:
    print("Rebuilding column embeddings with bilingual (EN+AR) descriptions...")
    _col_emb_cache.clear()
    prewarm_column_cache()
    print("Column embeddings now include Arabic descriptions")

## Translate Dev Questions to English (Google Translate)

In [ ]:
ENGLISH_TRANS_CACHE = "/content/drive/MyDrive/ar_spider_english_grounded.json"  #@param {type:"string"}
REGENERATE_TRANSLATIONS = False  #@param {type:"boolean"}
GROUNDING_SIM_THRESHOLD = 0.65  #@param {type:"number"}

from difflib import SequenceMatcher as _SM

# Load Google Translate

_translator = None

def _load_translator():
    global _translator
    if _translator is not None:
        return
    try:
        from deep_translator import GoogleTranslator
    except ImportError:
        import subprocess
        subprocess.run(["pip", "install", "-q", "deep-translator"], check=True)
        from deep_translator import GoogleTranslator
    _translator = GoogleTranslator(source='ar', target='en')
    print("Google Translate loaded (no VRAM cost)")


# Phase 1: Google Translate

def _translate_arabic_to_english(arabic_text: str) -> str:
    """Translate Arabic → English using Google Translate."""
    try:
        result = _translator.translate(arabic_text)
        return result.strip() if result and len(result.strip()) > 3 else ""
    except Exception as e:
        # Rate limit or network error — retry once after short delay
        import time
        time.sleep(0.5)
        try:
            result = _translator.translate(arabic_text)
            return result.strip() if result and len(result.strip()) > 3 else ""
        except Exception:
            return ""


# Phase 2: DB Value Grounding

def _load_db_text_values(db_path: str, schema_ddl: str) -> list:
    """Load all distinct TEXT values from a database."""
    if not db_path or not os.path.exists(db_path):
        return []
    schema_tables = extract_tables_columns(schema_ddl)
    if not schema_tables:
        return []
    all_values = set()
    try:
        conn = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True, timeout=10)
        conn.text_factory = str
        cursor = conn.cursor()
        for tbl, cols in schema_tables.items():
            for col in cols:
                try:
                    cursor.execute(f'PRAGMA table_info("{tbl}")')
                    col_info = {row[1].lower(): row[2] for row in cursor.fetchall()}
                    col_type = col_info.get(col.lower(), "TEXT").upper()
                    if not any(t in col_type for t in ["TEXT", "VARCHAR", "CHAR", "CLOB"]):
                        continue
                    cursor.execute(
                        f'SELECT DISTINCT "{col}" FROM "{tbl}" '
                        f'WHERE "{col}" IS NOT NULL AND TRIM("{col}") != "" '
                        f'LIMIT 100')
                    for row in cursor.fetchall():
                        val = str(row[0]).strip()
                        if len(val) >= 2:
                            all_values.add(val)
                except Exception:
                    continue
        conn.close()
    except Exception:
        pass
    return list(all_values)

_db_value_cache = {}

def _get_db_values(db_id: str, db_path: str, schema_ddl: str) -> list:
    """Get cached DB values for a database."""
    if db_id not in _db_value_cache:
        _db_value_cache[db_id] = _load_db_text_values(db_path, schema_ddl)
    return _db_value_cache[db_id]


def ground_translation(translation: str, db_values: list,
                        threshold: float = 0.65) -> tuple:
    """
    Ground a translated English question against actual DB values.
    Returns: (grounded_translation, list_of_replacements)
    """
    if not translation or not db_values:
        return translation, []

    replacements = []
    MIN_VALUE_LEN = 4
    safe_values = [v for v in db_values if len(v) >= MIN_VALUE_LEN]
    db_vals_lower = {v.lower(): v for v in safe_values}
    grounded = translation

    # Step 1: Exact case-insensitive match (word boundaries)
    for db_val in sorted(safe_values, key=len, reverse=True):
        try:
            pattern = re.compile(r'\b' + re.escape(db_val) + r'\b', re.IGNORECASE)
            match = pattern.search(grounded)
            if match and match.group(0) != db_val:
                grounded = pattern.sub(db_val, grounded)
                replacements.append(f"case: '{match.group(0)}' → '{db_val}'")
        except re.error:
            continue

    # Step 2: Extract candidate phrases for fuzzy matching
    candidates = []
    for m in re.finditer(r'"([^"]+)"', grounded):
        candidates.append((m.start(), m.end(), m.group(1), True))
    for m in re.finditer(r"'([^']+)'", grounded):
        candidates.append((m.start(), m.end(), m.group(1), True))
    for m in re.finditer(
        r'(?:[A-Z][a-z]+(?:\s+(?:of|the|and|in|for|to|de|la|le|el|al))?'
        r'(?:\s+[A-Z][a-z]+)*(?:\s+\([^)]+\))?)', grounded):
        phrase = m.group(0).strip()
        if len(phrase) >= MIN_VALUE_LEN:
            already_covered = any(
                s <= m.start() and m.end() <= e
                for s, e, _, q in candidates if q)
            if not already_covered:
                candidates.append((m.start(), m.end(), phrase, False))

    # Step 3: Fuzzy match candidates against DB values
    candidates.sort(key=lambda x: -x[0])
    skip_words = {
        'the', 'what', 'which', 'where', 'when', 'who', 'how',
        'find', 'show', 'list', 'give', 'get', 'display',
        'count', 'number', 'average', 'maximum', 'minimum',
        'total', 'all', 'each', 'every', 'many', 'most',
        'name', 'type', 'have', 'has', 'had', 'does', 'did',
        'with', 'from', 'that', 'this', 'than', 'then',
        'more', 'less', 'much', 'some', 'any', 'not',
        'and', 'but', 'for', 'are', 'were', 'been', 'being',
    }
    for start, end, phrase, is_quoted in candidates:
        phrase_lower = phrase.lower()
        if phrase_lower in skip_words:
            continue
        if len(phrase) < MIN_VALUE_LEN:
            continue
        if phrase in safe_values or phrase_lower in db_vals_lower:
            continue
        best_score = 0
        best_match = None
        for db_val in safe_values:
            len_ratio = len(phrase) / max(len(db_val), 1)
            if len_ratio < 0.4 or len_ratio > 2.5:
                continue
            score = _SM(None, phrase_lower, db_val.lower()).ratio()
            if score > best_score:
                best_score = score
                best_match = db_val
        if best_match and best_score >= threshold:
            before_char = grounded[start - 1] if start > 0 else ' '
            after_char = grounded[end] if end < len(grounded) else ' '
            if before_char.isalnum() or after_char.isalnum():
                continue
            grounded = grounded[:start] + best_match + grounded[end:]
            replacements.append(f"fuzzy({best_score:.2f}): '{phrase}' → '{best_match}'")

    return grounded, replacements


# Combined Pipeline: Translate + Ground + Cache

def generate_grounded_translations():
    """Generate English translations with DB value grounding for all eval samples."""
    cache_exists = os.path.exists(ENGLISH_TRANS_CACHE)

    # Try cache
    if cache_exists and not REGENERATE_TRANSLATIONS:
        try:
            with open(ENGLISH_TRANS_CACHE, "r", encoding="utf-8") as f:
                cached = json.load(f)
            print(f"Found translation cache: {len(cached)} entries")
            matched = 0
            missing = []
            for s in eval_samples:
                entry = cached.get(s.id, {})
                if isinstance(entry, dict):
                    eng = entry.get("raw", "")
                elif isinstance(entry, str):
                    eng = entry
                else:
                    eng = ""
                if eng:
                    s.english_question = eng
                    matched += 1
                else:
                    missing.append(s)
            print(f"   Matched: {matched}/{len(eval_samples)}")

            if missing:
                print(f"   {len(missing)} missing — translating + grounding...")
                _load_translator()
                for i, s in enumerate(missing):
                    raw = _translate_arabic_to_english(s.arabic_question)
                    if raw:
                        db_vals = _get_db_values(s.db_id, s.db_path, s.schema_ddl)
                        grounded, reps = ground_translation(raw, db_vals, GROUNDING_SIM_THRESHOLD)
                        s.english_question = raw  # use raw Google Translate (grounding saved for analysis only)
                        cached[s.id] = {"raw": raw, "grounded": grounded, "replacements": reps}
                        matched += 1
                    if (i + 1) % 50 == 0:
                        print(f"      [{i+1}/{len(missing)}]")
                with open(ENGLISH_TRANS_CACHE, "w", encoding="utf-8") as f:
                    json.dump(cached, f, ensure_ascii=False, indent=2)
                print(f"   Updated cache: {len(cached)} total")
            return
        except (json.JSONDecodeError, Exception) as e:
            print(f"Cache failed: {e}")

    # Full generation
    _load_translator()
    if cache_exists and REGENERATE_TRANSLATIONS:
        print(f"REGENERATE_TRANSLATIONS=True — regenerating all")
    else:
        print(f"No translation cache found")
    print(f"Translating + grounding {len(eval_samples)} samples...")
    print(f"   Translation: Google Translate (deep-translator)")
    print(f"   Grounding threshold: {GROUNDING_SIM_THRESHOLD}")
    print()

    cached = {}
    total_replacements = 0
    grounded_count = 0

    for i, s in enumerate(eval_samples):
        raw = _translate_arabic_to_english(s.arabic_question)
        if raw:
            db_vals = _get_db_values(s.db_id, s.db_path, s.schema_ddl)
            grounded, reps = ground_translation(raw, db_vals, GROUNDING_SIM_THRESHOLD)
            s.english_question = raw  # use raw Google Translate (grounding saved for analysis only)
            cached[s.id] = {"raw": raw, "grounded": grounded, "replacements": reps}
            if reps:
                grounded_count += 1
                total_replacements += len(reps)
        if (i + 1) % 100 == 0 or i == len(eval_samples) - 1:
            print(f"   [{i+1}/{len(eval_samples)}] translated: {len(cached)}, "
                  f"grounded: {grounded_count} ({total_replacements} replacements)")

    os.makedirs(os.path.dirname(ENGLISH_TRANS_CACHE), exist_ok=True)
    with open(ENGLISH_TRANS_CACHE, "w", encoding="utf-8") as f:
        json.dump(cached, f, ensure_ascii=False, indent=2)

    populated = sum(1 for s in eval_samples if s.english_question)
    print(f"\n Done: {len(cached)} translations → {ENGLISH_TRANS_CACHE}")
    print(f"   Grounded: {grounded_count} samples ({total_replacements} value replacements)")
    print(f"   Populated: {populated}/{len(eval_samples)} eval samples")

    # Show examples
    print(f"\n Grounding examples:")
    shown = 0
    for sid, entry in cached.items():
        if isinstance(entry, dict) and entry.get("replacements") and shown < 8:
            print(f"\n   {sid}:")
            print(f"     Raw:      {entry['raw'][:80]}")
            print(f"     Grounded: {entry['grounded'][:80]}")
            for rep in entry["replacements"]:
                print(f"     Fix: {rep}")
            shown += 1
    if shown < 3:
        print(f"\n Translation examples:")
        plain_shown = 0
        for s in eval_samples[:20]:
            if s.english_question and plain_shown < 5:
                print(f"   {s.id}: AR: {s.arabic_question[:60]}")
                print(f"         EN: {s.english_question[:60]}")
                plain_shown += 1


if ENABLE_ENGLISH_HINT:
    generate_grounded_translations()
else:
    print(" English hints disabled (ENABLE_ENGLISH_HINT = False)")

## Inference Post-Processing Functions

In [ ]:
# DEPENDENCY FUNCTIONS

from difflib import SequenceMatcher


def extract_sql_from_response(response: str) -> str:
    """Extract clean SQL from model response."""
    m = re.search(r'```(?:sql)?\s*\n?(.*?)```', response, re.DOTALL | re.I)
    if m:
        sql = m.group(1).strip()
        lines = [l for l in sql.split("\n") if not l.strip().startswith("--")]
        return "\n".join(lines).strip().rstrip(";")
    m = re.search(r'((?:SELECT|WITH)\s+.+?\bFROM\b.+?)(?:;|\n\n|$)', response, re.I | re.DOTALL)
    if m:
        return m.group(1).strip().rstrip(";")
    return response.strip().rstrip(";")


def extract_tables_columns(schema_sql: str) -> Dict[str, List[str]]:
    """Parse CREATE TABLE DDL into {table_name: [column_names]}."""
    tables = {}
    cur_table = None
    for line in schema_sql.split("\n"):
        line = line.strip()
        m = re.match(
            r'CREATE\s+TABLE\s+(?:IF\s+NOT\s+EXISTS\s+)?[`"\[]?([\w]+)[`"\]]?',
            line, re.I,
        )
        if m:
            cur_table = m.group(1)
            tables[cur_table] = []
        elif cur_table and line and not line.startswith(')'):
            cm = re.match(r'[`"\[]?([\w]+)[`"\]]?\s+\w+', line)
            if cm and not any(
                line.upper().startswith(kw) for kw in
                ('PRIMARY', 'FOREIGN', 'UNIQUE', 'CHECK', 'CONSTRAINT', '--')
            ):
                tables[cur_table].append(cm.group(1))
    return tables


def normalize_for_match(name: str) -> str:
    return name.lower().replace("_", "")


def find_best_match(identifier: str, candidates: List[str], threshold: float = 0.7) -> Optional[str]:
    id_norm = normalize_for_match(identifier)
    for c in candidates:
        if c.lower() == identifier.lower():
            return c
    for c in candidates:
        if normalize_for_match(c) == id_norm:
            return c
    best_score, best_match = 0, None
    for c in candidates:
        score = SequenceMatcher(None, id_norm, normalize_for_match(c)).ratio()
        if score > best_score:
            best_score, best_match = score, c
    return best_match if best_score >= threshold else None


SQL_KEYWORDS = {
    'select','from','where','join','inner','left','right','outer','cross',
    'natural','using','on','and','or','not','in','exists','between','like',
    'glob','is','null','as','escape',
    'order','by','group','having','limit','offset','union','intersect',
    'except','all','distinct','case','when','then','else','end','cast',
    'asc','desc','values','insert','update','delete','set','into',
    'top','iif','no',
    'count','sum','avg','min','max','abs','length','substr','substring',
    'replace','trim','ltrim','rtrim','round','upper','lower','instr',
    'coalesce','ifnull','nullif','typeof','total','group_concat',
    'create','table','primary','key','foreign','references','unique',
    'check','constraint','default','autoincrement','index','collate',
    'integer','text','real','boolean','date','time','varchar',
    'char','float','double','blob','numeric','int',
    'true','false',
    'with','recursive','over','partition','row','rows','fetch','next',
    'first','last','only','nulls','current','preceding','following',
    'unbounded','range','groups','window','filter',
    'rank','dense_rank','row_number','ntile','lag','lead',
    'first_value','last_value','nth_value',
}


def fix_sql_identifiers(pred_sql: str, schema_ddl: str) -> Tuple[str, List[str]]:
    """Fix column/table names in predicted SQL by matching against schema."""
    schema_tables = extract_tables_columns(schema_ddl)
    if not schema_tables:
        return pred_sql, []

    all_table_names = list(schema_tables.keys())
    all_column_names = []
    for tbl, cols in schema_tables.items():
        all_column_names.extend(cols)

    seen_lower = set()
    unique_columns = []
    for c in all_column_names:
        if c.lower() not in seen_lower:
            seen_lower.add(c.lower())
            unique_columns.append(c)

    fixes = []
    fixed = pred_sql

    # Pass 1: Fix table names after FROM/JOIN
    table_pat = re.compile(r'(?:FROM|JOIN)\s+([A-Za-z_]\w*)', re.I)
    for m in table_pat.finditer(fixed):
        used = m.group(1)
        if used.lower() in SQL_KEYWORDS:
            continue
        match = find_best_match(used, all_table_names, threshold=0.6)
        if match and match != used:
            fixed = re.sub(r'\b' + re.escape(used) + r'\b', match, fixed)
            fixes.append(f"table: {used} → {match}")

    # Pass 2: Fix qualified columns (T1.col, table.col)
    qual_pat = re.compile(r'(\b[A-Za-z_]\w*\b)\.(\b[A-Za-z_]\w*\b)')
    for m in qual_pat.finditer(fixed):
        col_ref = m.group(2)
        if col_ref.lower() in SQL_KEYWORDS:
            continue
        match = find_best_match(col_ref, unique_columns, threshold=0.65)
        if match and match != col_ref:
            old = f"{m.group(1)}.{col_ref}"
            new = f"{m.group(1)}.{match}"
            fixed = fixed.replace(old, new)
            fixes.append(f"col: {old} → {new}")

    # Pass 3: Fix unqualified columns via variant generation
    for col in unique_columns:
        variants = set()
        snake = re.sub(r'(?<=[a-z])([A-Z])', r'_\1', col).lower()
        if snake != col.lower():
            variants.add(snake)
        parts = col.split('_')
        if len(parts) > 1:
            camel = parts[0].lower() + ''.join(p.capitalize() for p in parts[1:])
            variants.add(camel)
            variants.add(camel.lower())
            pascal = ''.join(p.capitalize() for p in parts)
            variants.add(pascal)
        variants.add(col.lower())
        variants.add(col.upper())
        variants.discard(col)

        for variant in variants:
            if variant and variant != col:
                pat = r'\b' + re.escape(variant) + r'\b'
                if re.search(pat, fixed):
                    fixed = re.sub(pat, col, fixed)
                    fixes.append(f"variant: {variant} → {col}")

    return fixed, fixes


# Arabic ↔ English value grounding map
VALUE_GROUNDING_MAP = {
    "dog":      ["كلب", "كلاب", "الكلب", "الكلاب", "كلبا"],
    "cat":      ["قط", "قطة", "قطط", "القط", "القطة", "القطط", "هر", "هرة"],
    "bird":     ["طائر", "طيور", "الطائر", "الطيور", "عصفور"],
    "rabbit":   ["أرنب", "أرانب", "الأرنب"],
    "elephant": ["فيل", "أفيال", "الفيل"],
    "parrot":   ["ببغاء", "الببغاء"],
    "f":        ["طالبات", "الطالبات", "أنثى", "إناث", "نساء", "امرأة", "بنات", "البنات", "طالبة"],
    "m":        ["طلاب", "الطلاب", "ذكر", "ذكور", "رجال", "رجل", "أولاد"],
    "female":   ["طالبات", "الطالبات", "أنثى", "إناث", "نساء", "امرأة"],
    "male":     ["طلاب", "ذكر", "ذكور", "رجال"],
    "france":   ["فرنسا", "الفرنسية", "فرنسي"],
    "usa":      ["الولايات المتحدة", "أمريكا", "أمريكي", "الأمريكية"],
    "japan":    ["اليابان", "ياباني", "اليابانية"],
    "germany":  ["ألمانيا", "ألماني", "الألمانية"],
    "europe":   ["أوروبا", "أوروبي", "الأوروبية"],
    "asia":     ["آسيا", "آسيوي", "الآسيوية"],
    "africa":   ["أفريقيا", "أفريقي", "الأفريقية"],
    "rock":     ["روك", "الروك"],
    "pop":      ["بوب", "البوب"],
    "jazz":     ["جاز", "الجاز"],
    "classical":["كلاسيكي", "الكلاسيكية"],
    "yes":      ["نعم"],
    "no":       ["لا"],
    "true":     ["صحيح", "نعم"],
    "false":    ["خطأ", "لا"],
}


def is_value_grounded(value: str, question: str) -> bool:
    """Check if a string literal value from SQL is grounded in the Arabic question."""
    val_lower = value.strip().lower()
    question_lower = question.lower()
    if val_lower in question_lower:
        return True
    if val_lower in VALUE_GROUNDING_MAP:
        arabic_words = VALUE_GROUNDING_MAP[val_lower]
        for aw in arabic_words:
            if aw in question:
                return True
        return False
    try:
        float(val_lower)
        return True
    except ValueError:
        pass
    if len(value.split()) > 1:
        return True
    if val_lower not in VALUE_GROUNDING_MAP:
        return True
    return False


def remove_hallucinated_filters(sql: str, question: str) -> Tuple[str, List[str]]:
    """Remove WHERE conditions with string literals not grounded in the Arabic question."""
    removals = []
    where_match = re.search(
        r'\bWHERE\b\s+(.*?)(?:\bGROUP\b|\bORDER\b|\bLIMIT\b|\bHAVING\b|\bUNION\b|\bINTERSECT\b|\bEXCEPT\b|$)',
        sql, re.I | re.DOTALL
    )
    if not where_match:
        return sql, []

    where_clause = where_match.group(1).strip()
    where_start = where_match.start(1)
    where_end = where_match.end(1)

    conditions = []
    depth = 0
    current = ""
    tokens = re.split(r'(\bAND\b)', where_clause, flags=re.I)
    i = 0
    while i < len(tokens):
        token = tokens[i]
        depth += token.count('(') - token.count(')')
        if token.strip().upper() == 'AND' and depth == 0:
            if current.strip():
                conditions.append(current.strip())
            current = ""
        else:
            current += token
        i += 1
    if current.strip():
        conditions.append(current.strip())

    keep_conditions = []
    for cond in conditions:
        literals = re.findall(r"""["'](.*?)["']""", cond)
        if not literals:
            keep_conditions.append(cond)
            continue
        all_grounded = True
        for lit in literals:
            if not is_value_grounded(lit, question):
                all_grounded = False
                break
        if all_grounded:
            keep_conditions.append(cond)
        else:
            removals.append(f"removed: {cond.strip()[:80]}")

    final_conditions = []
    for cond in keep_conditions:
        sub_literals = re.findall(r"""["'](.*?)["']""", cond)
        sub_hallucinated = False
        for lit in sub_literals:
            if not is_value_grounded(lit, question):
                sub_hallucinated = True
                break
        if sub_hallucinated:
            removals.append(f"removed (subquery): {cond.strip()[:80]}")
        else:
            final_conditions.append(cond)

    if not removals:
        return sql, []

    if final_conditions:
        new_where = " AND ".join(final_conditions)
        new_sql = sql[:where_start] + new_where + sql[where_end:]
    else:
        where_full = re.search(
            r'\bWHERE\b\s+.*?(?=\bGROUP\b|\bORDER\b|\bLIMIT\b|\bHAVING\b|\bUNION\b|\bINTERSECT\b|\bEXCEPT\b|$)',
            sql, re.I | re.DOTALL
        )
        if where_full:
            new_sql = sql[:where_full.start()].rstrip() + " " + sql[where_full.end():].lstrip()
        else:
            new_sql = sql
    new_sql = re.sub(r'\s+', ' ', new_sql).strip()
    return new_sql, removals


print("Dependency functions loaded (extract_sql, fix_identifiers, filter_removal)")


# SQL Value Grounding Post-Processor

def ground_sql_values(sql: str, db_path: str, schema_ddl: str) -> Tuple[str, List[str]]:
    """
    Replace string literals in SQL with exact DB values using cosine similarity.
    Uses the already-loaded embedding model (_emb_model) for semantic matching.
    Returns: (fixed_sql, list_of_replacements)
    """
    if not sql or not db_path or not os.path.exists(db_path):
        return sql, []

    # Extract string literals from SQL
    literals = []
    for m in re.finditer(r"'([^']*)'", sql):
        val = m.group(1)
        if val and len(val) >= 2:
            literals.append((m.start(1), m.end(1), val))

    if not literals:
        return sql, []

    def _find_column_for_literal(sql_text, lit_start):
        """Look backwards from literal position to find column name."""
        prefix = sql_text[:lit_start].strip()
        m = re.search(r'(\w+(?:\.\w+)?)\s*(?:=|!=|<>|LIKE|like|IN|in)\s*["\']?\s*$', prefix)
        return m.group(1) if m else None

    # Load DB values per column
    schema_tables = extract_tables_columns(schema_ddl)
    if not schema_tables:
        return sql, []

    col_values = {}
    try:
        conn = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True, timeout=10)
        conn.text_factory = str
        cursor = conn.cursor()
        for tbl, cols in schema_tables.items():
            for col in cols:
                try:
                    cursor.execute(f'PRAGMA table_info("{tbl}")')
                    col_info = {row[1].lower(): row[2] for row in cursor.fetchall()}
                    col_type = col_info.get(col.lower(), "TEXT").upper()
                    if not any(t in col_type for t in ["TEXT", "VARCHAR", "CHAR", "CLOB"]):
                        continue
                    cursor.execute(
                        f'SELECT DISTINCT "{col}" FROM "{tbl}" '
                        f'WHERE "{col}" IS NOT NULL AND TRIM("{col}") != "" '
                        f'LIMIT 200')
                    vals = set()
                    for row in cursor.fetchall():
                        v = str(row[0]).strip()
                        if v:
                            vals.add(v)
                    if vals:
                        col_values[f"{tbl}.{col}".lower()] = vals
                        col_values[col.lower()] = vals
                except Exception:
                    continue
        conn.close()
    except Exception:
        return sql, []

    if not col_values:
        return sql, []

    all_values = set()
    for vals in col_values.values():
        all_values.update(vals)

    # Check each literal and replace if needed (right-to-left)
    replacements = []
    fixed_sql = sql

    for start, end, literal in sorted(literals, key=lambda x: -x[0]):
        if literal in all_values:
            continue  # exact match — no fix needed

        # Case-insensitive exact match first (free, no embedding needed)
        lit_lower = literal.lower()
        case_match = None
        for v in all_values:
            if v.lower() == lit_lower:
                case_match = v
                break
        if case_match:
            fixed_sql = fixed_sql[:start] + case_match + fixed_sql[end:]
            replacements.append(f"case: '{literal}' → '{case_match}'")
            continue

        # Find target column for this literal
        col_name = _find_column_for_literal(fixed_sql, start - 1)
        if col_name:
            candidates = col_values.get(col_name.lower(), set())
            if not candidates:
                bare = col_name.split(".")[-1].lower()
                candidates = col_values.get(bare, set())
        else:
            candidates = all_values

        if not candidates:
            continue

        # Cosine similarity matching using embedding model
        candidates_list = [v for v in candidates if len(v) >= 2]
        if not candidates_list:
            continue

        try:
            # Encode the literal and all candidate DB values
            lit_embedding = _emb_model.encode(
                [literal], normalize_embeddings=True,
                show_progress_bar=False,
            )[0]
            cand_embeddings = _emb_model.encode(
                candidates_list, normalize_embeddings=True,
                show_progress_bar=False,
            )

            # Cosine similarity (dot product of normalized vectors)
            similarities = cand_embeddings @ lit_embedding
            best_idx = int(np.argmax(similarities))
            best_score = float(similarities[best_idx])
            best_match = candidates_list[best_idx]

            if best_score >= SQL_VALUE_SIM_THRESHOLD:
                fixed_sql = fixed_sql[:start] + best_match + fixed_sql[end:]
                replacements.append(
                    f"cosine({best_score:.2f}): '{literal}' → '{best_match}' (col={col_name or '?'})")
        except Exception:
            continue

    return fixed_sql, replacements

## Preview: What the Inference Model Sees

In [ ]:
#@title Preview: What the Model Sees { display-mode: "form" }


PREVIEW_SAMPLE_IDS = [0, 25, 50, 100, 200, 400, 600, 800,  1000]  #@param {type:"raw"}

print("=" * 80)
print("  PROMPT PREVIEW — What the model receives at inference")
print("=" * 80)

for sample_idx in PREVIEW_SAMPLE_IDS:
    if sample_idx >= len(eval_samples):
        continue
    sample = eval_samples[sample_idx]

    prompt = format_tier1_prompt(sample)

    # Count tokens (rough estimate)
    try:
        token_count = len(infer_tok.encode(prompt))
    except:
        token_count = len(prompt) // 3  # rough fallback

    print(f"\n{'━' * 80}")
    print(f"  Sample: {sample.id} | {sample.db_id} | ~{token_count} tokens")
    print(f"  Question: {sample.arabic_question[:100]}")
    print(f"  Gold SQL: {sample.gold_sql[:100]}")
    print(f"{'━' * 80}")

    # Print prompt with section markers
    for line in prompt.split("\n"):
        if line.strip().startswith("★"):
            print(f"  \033[92m{line}\033[0m")  # green for column hints
        elif line.strip().startswith("Likely relevant"):
            print(f"  \033[92m{line}\033[0m")  # green header
        elif line.strip().startswith("Sample column values"):
            print(f"  \033[93m{line}\033[0m")  # yellow for value hints
        elif line.strip().startswith("Question:"):
            print(f"  \033[96m{line}\033[0m")  # cyan for question
        else:
            print(f"  {line}")

    print()

# Summary stats
print("=" * 80)
print("  PROMPT STATISTICS (across all eval samples)")
print("=" * 80)

token_counts = []
col_hint_counts = []
ar_desc_counts = []

for s in eval_samples[:50]:  # sample first 50 for speed
    p = format_tier1_prompt(s)
    try:
        tc = len(infer_tok.encode(p))
    except:
        tc = len(p) // 3
    token_counts.append(tc)
    col_hint_counts.append(p.count("★"))
    ar_desc_counts.append(p.count("—"))

print(f"  Token count (first 50 samples):")
print(f"    Min: {min(token_counts)}, Max: {max(token_counts)}, Avg: {sum(token_counts)//len(token_counts)}")
print(f"    Over MAX_SEQ_LENGTH ({MAX_SEQ_LENGTH_7B}): {sum(1 for t in token_counts if t > MAX_SEQ_LENGTH_7B)}/{len(token_counts)}")
print(f"  Column hints per prompt: avg {sum(col_hint_counts)/len(col_hint_counts):.1f}")
print(f"  Arabic descriptions per prompt: avg {sum(ar_desc_counts)/len(ar_desc_counts):.1f}")
print()

## Batched Generation + Self-Consistency Voting

In [ ]:
def _build_chat_text(tokenizer, prompt: str) -> str:
    """Build chat-formatted string from a user prompt."""
    messages = [
        {"role": "system", "content": TIER1_SYSTEM},
        {"role": "user", "content": prompt},
    ]
    return tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=False)


def generate_candidates_batched(model, tokenizer, prompt: str,
                                 n_greedy: int = 1,
                                 n_sampled: int = 0,
                                 batch_size: int = 4,
                                 temperature: float = 0.7,
                                 top_p: float = 0.95) -> List[str]:
    """
    Generate multiple SQL candidates for ONE prompt in batched GPU passes.

    Args:
        n_greedy:  number of greedy decodes (typically 1)
        n_sampled: number of sampled candidates (typically N-1)
        batch_size: max sequences per GPU pass (tune to VRAM)

    Returns:
        List of raw SQL strings (greedy first, then sampled).
    """
    chat = _build_chat_text(tokenizer, prompt)
    results = []

    # Save and set padding side for batched generation
    orig_pad_side = tokenizer.padding_side
    tokenizer.padding_side = "left"

    # Greedy pass (1 candidate)
    if n_greedy > 0:
        try:
            inp = tokenizer(
                [chat], return_tensors="pt",
                truncation=True, max_length=MAX_SEQ_LENGTH_7B,
            ).to(model.device)
            with torch.no_grad():
                out = model.generate(
                    **inp, max_new_tokens=300,
                    do_sample=False,
                    eos_token_id=tokenizer.eos_token_id,
                    pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
                )
            resp = tokenizer.decode(
                out[0][inp.input_ids.shape[1]:], skip_special_tokens=True)
            results.append(extract_sql_from_response(resp))
        except Exception as e:
            print(f"      Greedy generation failed: {e}")
            results.append("SELECT 1")

    # Sampled passes (batched)
    if n_sampled > 0:
        remaining = n_sampled
        while remaining > 0:
            bs = min(remaining, batch_size)
            batch_chats = [chat] * bs  # same prompt repeated
            try:
                inp = tokenizer(
                    batch_chats, return_tensors="pt",
                    padding=True, truncation=True,
                    max_length=MAX_SEQ_LENGTH_7B,
                ).to(model.device)
                with torch.no_grad():
                    out = model.generate(
                        **inp, max_new_tokens=300,
                        do_sample=True,
                        temperature=temperature,
                        top_p=top_p,
                        eos_token_id=tokenizer.eos_token_id,
                        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
                    )
                input_len = inp.input_ids.shape[1]
                for j in range(bs):
                    resp = tokenizer.decode(
                        out[j][input_len:], skip_special_tokens=True)
                    results.append(extract_sql_from_response(resp))
            except Exception as e:
                print(f"      Batched generation failed (bs={bs}): {e}")
                # Fallback: fill with greedy result
                for _ in range(bs):
                    results.append(results[0] if results else "SELECT 1")
            remaining -= bs

    tokenizer.padding_side = orig_pad_side
    return results


def generate_single_greedy(model, tokenizer, prompt: str) -> str:
    """Single greedy generation (used for self-correction retries)."""
    return generate_candidates_batched(
        model, tokenizer, prompt,
        n_greedy=1, n_sampled=0,
    )[0]


def _hash_exec_result(exec_result) -> str:
    """Hashable key from execution result for voting."""
    if exec_result is None:
        return "__NONE__"
    if isinstance(exec_result, str):
        return f"__ERR__{exec_result[:100]}"
    try:
        rows = exec_result["rows"] if isinstance(exec_result, dict) else exec_result
        def _nv(v):
            s = str(v).strip()
            try:
                f = float(s)
                return str(int(f)) if f == int(f) else f"{f:.6f}"
            except (ValueError, OverflowError):
                return s.lower()
        normalized = tuple(sorted(
            tuple(sorted(_nv(v) for v in row)) for row in rows))
        return str(normalized)
    except Exception:
        return str(exec_result)


def self_consistency_generate(model, tokenizer, sample,
                               n_candidates: int = 8,
                               temperature: float = 0.7,
                               top_p: float = 0.95,
                               use_ddl_override: bool = False,
                               ) -> Tuple[str, dict]:
    """
    Self-Consistency with execution-based voting (BATCHED).
    V4: multi-temperature + use_ddl_override.

    Multi-temperature strategy:
      Instead of sampling all candidates at one temperature, split
      across tiers (e.g., 2 at T=0.3, 3 at T=0.7, 2 at T=1.1).
      This produces both conservative near-greedy variants AND
      creative alternatives that try different SQL structures.
      Research: CHASE-SQL, Mixture-of-Temperatures.
    """
    prompt = format_tier1_prompt(sample, use_ddl_override=use_ddl_override)

    import zlib
    _seed_base = None
    if 'SC_DETERMINISTIC_SEEDS' in globals() and SC_DETERMINISTIC_SEEDS:
        _seed_base = SC_BASE_SEED + (zlib.crc32(str(sample.id).encode("utf-8")) % 1_000_000)

    slot_info = []
    if ENABLE_MULTI_TEMP:
        # Multi-temperature: 1 greedy + tiers of sampled at different T
        candidates_raw = generate_candidates_batched(
            model, tokenizer, prompt,
            n_greedy=1, n_sampled=0,
            batch_size=SC_BATCH_SIZE,
        )
        slot_info.append(("greedy", 0.0))
        temp_breakdown = {"greedy": 1}

        for tier_idx, (tier_temp, tier_count) in enumerate(SC_TEMP_TIERS):
            if tier_count <= 0:
                continue
            if _seed_base is not None:
                torch.manual_seed(_seed_base + 1000 * (tier_idx + 1))
            tier_candidates = generate_candidates_batched(
                model, tokenizer, prompt,
                n_greedy=0,
                n_sampled=tier_count,
                batch_size=SC_BATCH_SIZE,
                temperature=tier_temp,
                top_p=top_p,
            )
            candidates_raw.extend(tier_candidates)
            slot_info.extend([(f"T={tier_temp}", tier_temp)] * len(tier_candidates))
            temp_breakdown[f"T={tier_temp}"] = tier_count
    else:
        if _seed_base is not None:
            torch.manual_seed(_seed_base)
        candidates_raw = generate_candidates_batched(
            model, tokenizer, prompt,
            n_greedy=1,
            n_sampled=n_candidates - 1,
            batch_size=SC_BATCH_SIZE,
            temperature=temperature,
            top_p=top_p,
        )
        slot_info = [("greedy", 0.0)] + [(f"T={temperature}", temperature)] * (len(candidates_raw) - 1)
        temp_breakdown = {"greedy": 1, f"T={temperature}": n_candidates - 1}

    # Post-process each candidate
    candidates_pp = []
    for raw in candidates_raw:
        current = raw
        if ENABLE_ID_FIXING:
            current, _ = fix_sql_identifiers(current, sample.schema_ddl)
        if ENABLE_FILTER_REMOVAL:
            current, _ = remove_hallucinated_filters(current, sample.arabic_question)
        if ENABLE_SQL_VALUE_GROUNDING:
            current, _ = ground_sql_values(current, sample.db_path, sample.schema_ddl)
        candidates_pp.append(current)

    # Execute and vote
    if not sample.db_path or not os.path.exists(sample.db_path):
        return candidates_pp[0], {
            "method": "greedy_no_db",
            "n_candidates": len(candidates_raw),
        }

    result_groups = defaultdict(list)
    exec_failures = 0
    candidate_log = []  # logging only — voting below is UNCHANGED

    for _slot, sql in enumerate(candidates_pp):
        _label, _temp = slot_info[_slot] if _slot < len(slot_info) else (f"slot{_slot}", None)
        ok, result = execute_sql_on_db(sample.db_path, sql)
        _entry = {"slot": _slot, "label": _label, "temperature": _temp,
                  "raw_sql": candidates_raw[_slot] if _slot < len(candidates_raw) else None,
                  "sql": sql, "exec_ok": bool(ok)}
        if ok:
            key = _hash_exec_result(result)
            result_groups[key].append((sql, result))
            _entry["result_key"] = str(key)
            try:
                _entry["n_rows"] = len(result)
                _entry["empty_result"] = (len(result) == 0)
            except Exception:
                _entry["n_rows"] = None
                _entry["empty_result"] = None
        else:
            exec_failures += 1
            result_groups["__FAIL__"].append((sql, result))
            _entry["error"] = str(result)[:200]
        candidate_log.append(_entry)

    # Find the largest group (most common execution result)
    best_key, best_count = None, 0
    for key, group in result_groups.items():
        if key == "__FAIL__":
            continue
        if len(group) > best_count:
            best_count = len(group)
            best_key = key

    if best_key is not None:
        winner_group = result_groups[best_key]
        best_sql = min(winner_group, key=lambda x: len(x[0]))[0]
        _meta = {
            "method": "sc_vote",
            "n_candidates": len(candidates_raw),
            "n_unique_results": len(result_groups) - (1 if "__FAIL__" in result_groups else 0),
            "winner_votes": best_count,
            "exec_failures": exec_failures,
            "temp_breakdown": temp_breakdown if ENABLE_MULTI_TEMP else None,
        }
        if 'LOG_CANDIDATES' in globals() and LOG_CANDIDATES:
            _meta["candidates"] = candidate_log
        return best_sql, _meta
    else:
        # All candidates failed execution → return greedy
        _meta = {
            "method": "sc_all_failed",
            "n_candidates": len(candidates_raw),
            "exec_failures": exec_failures,
            "temp_breakdown": temp_breakdown if ENABLE_MULTI_TEMP else None,
        }
        if 'LOG_CANDIDATES' in globals() and LOG_CANDIDATES:
            _meta["candidates"] = candidate_log
        return candidates_pp[0], _meta

In [ ]:
#@title Export Gold Result Hashes (AR) { display-mode: "form" }
import json as _json
gold_hashes = {}
_fail = 0
for _s in eval_samples:
    if _s.db_path and os.path.exists(_s.db_path):
        _ok, _r = execute_sql_on_db(_s.db_path, _s.gold_sql)
        gold_hashes[str(_s.id)] = str(_hash_exec_result(_r)) if _ok else None
        _fail += (0 if _ok else 1)
_p = "/content/gold_hashes_AR.json"
with open(_p, "w") as _f: _json.dump(gold_hashes, _f)
print(f"{len(gold_hashes)} gold hashes → {_p} (gold failures: {_fail})")
try:
    import shutil as _sh; _sh.copy2(_p, "/content/drive/MyDrive/gold_hashes_AR.json"); print("Drive copy")
except Exception as _e: print(f"(Drive copy skipped: {_e})")

## Execution-Guided Self-Correction

In [ ]:
def self_correct(model, tokenizer, sample, failed_sql: str,
                 error_msg: str, max_retries: int = 2) -> Tuple[str, int]:
    """Retry SQL generation with error feedback."""
    if ENABLE_MSCHEMA:
        schema_str = get_mschema(sample.db_id, sample.schema_ddl)
    else:
        schema_str = sample.schema_ddl[:4500]

    value_hints = ""
    if ENABLE_VALUE_HINTS and sample.db_path:
        value_hints = sample_db_values_question_aware(
            sample.db_path, sample.schema_ddl,
            sample.arabic_question, max_per_col=VALUE_HINT_MAX_PER_COL)

    current_sql = failed_sql
    current_error = error_msg

    for attempt in range(max_retries):
        retry_prompt = TIER1_RETRY_PROMPT.format(
            schema=schema_str, value_hints=value_hints,
            failed_sql=current_sql[:200], error_msg=current_error[:150],
            question=sample.arabic_question)
        corrected = generate_single_greedy(model, tokenizer, retry_prompt)
        if ENABLE_ID_FIXING:
            corrected, _ = fix_sql_identifiers(corrected, sample.schema_ddl)
        if ENABLE_SQL_VALUE_GROUNDING:
            corrected, _ = ground_sql_values(corrected, sample.db_path, sample.schema_ddl)
        if sample.db_path and os.path.exists(sample.db_path):
            ok, result = execute_sql_on_db(sample.db_path, corrected)
            if ok:
                return corrected, attempt + 1
            else:
                current_sql = corrected
                current_error = str(result)
        else:
            return corrected, attempt + 1
    return current_sql, max_retries

## Combined Pipeline + Evaluation Function

In [ ]:
# COMBINED PIPELINE

def evaluate_single_sample(model, tokenizer, sample, idx: int) -> dict:
    """Full Tier 1 pipeline for one sample — V4 with low-confidence fallback."""
    metadata = {}

    # Step 1: Generate (batched SC or single greedy)
    if ENABLE_SELF_CONSISTENCY:
        pred_sql, sc_meta = self_consistency_generate(
            model, tokenizer, sample,
            n_candidates=SC_NUM_CANDIDATES,
            temperature=SC_TEMPERATURE, top_p=SC_TOP_P)
        metadata.update(sc_meta)

        # Low-confidence fallback — re-run with DDL + extra candidates
        if (ENABLE_LOW_CONF_FALLBACK
            and sc_meta.get("winner_votes", 99) <= LOW_CONF_VOTE_THRESHOLD):

            pred_sql_2, sc_meta_2 = self_consistency_generate(
                model, tokenizer, sample,
                n_candidates=SC_NUM_CANDIDATES + LOW_CONF_EXTRA_CANDIDATES,
                temperature=SC_TEMPERATURE * 0.85,
                top_p=SC_TOP_P,
                use_ddl_override=True,
            )
            votes_1 = sc_meta.get("winner_votes", 0)
            votes_2 = sc_meta_2.get("winner_votes", 0)
            total_1 = sc_meta.get("n_candidates", 1)
            total_2 = sc_meta_2.get("n_candidates", 1)
            conf_1 = votes_1 / total_1
            conf_2 = votes_2 / total_2

            if conf_2 > conf_1:
                pred_sql = pred_sql_2
                metadata.update(sc_meta_2)
                metadata["fallback_used"] = True
                metadata["fallback_reason"] = f"conf {conf_1:.2f}->{conf_2:.2f}"
            else:
                metadata["fallback_used"] = False
                metadata["fallback_reason"] = f"kept orig {conf_1:.2f}>={conf_2:.2f}"
    else:
        prompt = format_tier1_prompt(sample)
        raw_sql = generate_single_greedy(model, tokenizer, prompt)
        pred_sql = raw_sql
        if ENABLE_ID_FIXING:
            pred_sql, idf = fix_sql_identifiers(pred_sql, sample.schema_ddl)
            metadata["id_fixes"] = len(idf)
        if ENABLE_FILTER_REMOVAL:
            pred_sql, fr = remove_hallucinated_filters(pred_sql, sample.arabic_question)
            metadata["filter_removals"] = len(fr)
        if ENABLE_SQL_VALUE_GROUNDING:
            pred_sql, vg = ground_sql_values(pred_sql, sample.db_path, sample.schema_ddl)
            metadata["value_groundings"] = len(vg)
        metadata["method"] = "single"

    # Step 2: Self-correction (if execution fails)
    if ENABLE_SELF_CORRECTION and sample.db_path and os.path.exists(sample.db_path):
        ok, result = execute_sql_on_db(sample.db_path, pred_sql)
        if not ok:
            corrected, retries = self_correct(
                model, tokenizer, sample,
                pred_sql, str(result), max_retries=SC_MAX_RETRIES)
            if corrected != pred_sql:
                pred_sql = corrected
                metadata["self_corrected"] = True
                metadata["retries_used"] = retries

    # Step 3: Compute metrics
    ex = None
    if sample.db_path and os.path.exists(sample.db_path):
        gold_ok, gold_res = execute_sql_on_db(sample.db_path, sample.gold_sql)
        pred_ok, pred_res = execute_sql_on_db(sample.db_path, pred_sql)
        if gold_ok and pred_ok:
            ex = compare_results(gold_res, pred_res)
        elif gold_ok and not pred_ok:
            ex = False

    em = compute_exact_match(pred_sql, sample.gold_sql)
    bleu = compute_bleu(pred_sql, sample.gold_sql)
    sqam = compute_sqam(pred_sql, sample.gold_sql)
    tsed = compute_tsed(pred_sql, sample.gold_sql)

    return {
        "id": sample.id, "db_id": sample.db_id,
        "question": sample.arabic_question,
        "gold_sql": sample.gold_sql, "pred_sql": pred_sql,
        "ex": ex, "em": em,
        "bleu": bleu, "sqam": sqam, "tsed": tsed,
        "metadata": metadata,
    }

## Run Evaluation (Dev Set)

In [ ]:
# RUN EVALUATION

print(f"Evaluating {len(eval_samples)} dev samples with Tier 1 enhancements (BATCHED)...")
print(f"   Model: {FINETUNE_7B_BASE} + Arabic adapter")
techniques = " + ".join(filter(None, [
    "M-Schema" if ENABLE_MSCHEMA else None,
    "ValueHints" if ENABLE_VALUE_HINTS else None,
    f"ColLink(emb,K={COL_LINK_TOP_K})" if ENABLE_COLUMN_LINKING else None,
    f"ArColDesc" if ENABLE_COLUMN_LINKING and ENABLE_AR_COL_DESC else None,
    "EnglishHint" if ENABLE_ENGLISH_HINT else None,
    "BilingualSchema" if ENABLE_BILINGUAL_SCHEMA else None,
    f"ValueInject(K={VALUE_INJECTION_TOP_K})" if ENABLE_VALUE_INJECTION else None,
    "BilingualEmb" if ENABLE_BILINGUAL_EMBEDDINGS else None,
    f"SampleRows({SAMPLE_ROWS_LIMIT}×{SAMPLE_ROWS_MAX_TABLES})" if ENABLE_SAMPLE_ROWS else None,
    f"SC(N={SC_NUM_CANDIDATES},batch={SC_BATCH_SIZE})" if ENABLE_SELF_CONSISTENCY else None,
    f"MultiTemp({','.join(f'{t}×{c}' for t,c in SC_TEMP_TIERS)})" if ENABLE_MULTI_TEMP else None,
    f"LowConfFB(≤{LOW_CONF_VOTE_THRESHOLD}→+{LOW_CONF_EXTRA_CANDIDATES})" if ENABLE_LOW_CONF_FALLBACK else None,
    "SelfCorrect" if ENABLE_SELF_CORRECTION else None,
    "IDFix" if ENABLE_ID_FIXING else None,
    "FilterRemoval" if ENABLE_FILTER_REMOVAL else None,
    f"SQLValGround(t={SQL_VALUE_SIM_THRESHOLD})" if ENABLE_SQL_VALUE_GROUNDING else None,
]))
print(f"   Techniques: {techniques}")
if ENABLE_SELF_CONSISTENCY:
    passes_per_sample = 1 + math.ceil((SC_NUM_CANDIDATES - 1) / SC_BATCH_SIZE)
    est_passes = len(eval_samples) * passes_per_sample
    print(f"   Est. GPU passes: ~{est_passes} ({passes_per_sample}/sample)")
    print(f"   Est. time: ~{est_passes * 1.5 / 60:.0f} min on T4 GPU")
    print(f"   Speedup vs sequential: ~{SC_NUM_CANDIDATES / passes_per_sample:.1f}x")
print()

results = []
t0 = time.time()
sc_votes = 0
sc_fallbacks = 0
self_corrections = 0
total_retries = 0
fallback_used = 0
fallback_wins = 0

for i, sample in enumerate(eval_samples):
    if i < RESUME_FROM:
        continue

    result = evaluate_single_sample(infer_model, infer_tok, sample, i)
    results.append(result)

    meta = result.get("metadata", {})
    if meta.get("method") == "sc_vote":
        sc_votes += 1
    elif meta.get("method", "").startswith("sc_"):
        sc_fallbacks += 1
    if meta.get("self_corrected"):
        self_corrections += 1
        total_retries += meta.get("retries_used", 0)
    if meta.get("fallback_used"):
        fallback_used += 1
        if result["ex"] is True:
            fallback_wins += 1

    ex_i = "OK" if result["ex"] is True else ("FAIL" if result["ex"] is False else "WARN")
    em_i = "OK" if result["em"] else "FAIL"
    m = meta.get("method", "?")
    v = meta.get("winner_votes", "")
    c = " SC" if meta.get("self_corrected") else ""
    fb = " FB" if meta.get("fallback_used") else ""

    print(f"{'='*80}")
    print(f"{sample.id} | {sample.db_id} [{i+1}/{len(eval_samples)}]")
    print(f"{sample.arabic_question[:120]}")
    print(f"PRED: {result['pred_sql'][:200]}")
    print(f"GOLD: {sample.gold_sql[:200]}")
    print(f"EX={ex_i}  EM={em_i}  BLEU={result['bleu']:.3f}"
          f"  [{m}{f':votes={v}' if v else ''}{c}{fb}]")

    if (i + 1) % 50 == 0 or len(results) % 50 == 0:
        ex_sf = sum(1 for r in results if r["ex"] is True)
        ex_tf = sum(1 for r in results if r["ex"] is not None)
        em_sf = sum(1 for r in results if r["em"] is True)
        bl_sf = sum(r["bleu"] for r in results) / len(results)
        el = time.time() - t0
        rate = el / len(results)
        remaining_samples = len(eval_samples) - i - 1
        eta = rate * remaining_samples
        print(f"\n   [{i+1}/{len(eval_samples)}] ({len(results)} evaluated):"
              f" EX={100*ex_sf/max(ex_tf,1):.1f}% ({ex_sf}/{ex_tf})"
              f"  EM={100*em_sf/len(results):.1f}%"
              f"  BLEU={bl_sf:.3f}"
              f"  {el:.0f}s ({rate:.1f}s/sample, ETA {eta/60:.0f}min)"
              f"  SC={sc_votes} Fix={self_corrections} FB={fallback_used}\n")

    if (i + 1) % 100 == 0 and torch.cuda.is_available():
        torch.cuda.empty_cache()

elapsed = time.time() - t0


# RESULTS

total    = len(results)
ex_count = sum(1 for r in results if r["ex"] is True)
ex_total = sum(1 for r in results if r["ex"] is not None)
em_count = sum(1 for r in results if r["em"] is True)
avg_bleu = sum(r["bleu"] for r in results) / total
avg_sqam = sum(r["sqam"] for r in results) / total
avg_tsed = sum(r["tsed"] for r in results) / total

print()
print("=" * 70)
print(f"  RESULTS: Tier 1 Enhanced Evaluation — V4 (BATCHED)")
print(f"  Model: Qwen2.5-Coder-7B + Arabic QLoRA Adapter")
print(f"  Techniques: {techniques}")
print(f"  Evaluation: Ar-Spider dev.json ({total} samples)")
if RESUME_FROM > 0:
    print(f"  Resumed from sample {RESUME_FROM}")
print("=" * 70)
print(f"  EX  (Execution Accuracy) : {ex_count}/{ex_total} = {100*ex_count/max(ex_total,1):.1f}%")
print(f"  EM  (Exact Match)        : {em_count}/{total} = {100*em_count/total:.1f}%")
print(f"  BLEU                     : {avg_bleu:.3f}")
print(f"  SQAM                     : {avg_sqam:.3f}")
print(f"  TSED                     : {avg_tsed:.3f}")
print("=" * 70)
print()
print(f"   Total: {elapsed:.1f}s ({elapsed/total:.1f}s/sample)")
print()
if ENABLE_SELF_CONSISTENCY:
    print(f"  SC majority wins : {sc_votes}")
    print(f"  SC fallbacks     : {sc_fallbacks}")
if ENABLE_SELF_CORRECTION:
    print(f"  Self-corrections : {self_corrections}")
    print(f"  Total retries    : {total_retries}")
if ENABLE_LOW_CONF_FALLBACK:
    print(f"  Fallback triggered: {fallback_used}")
    print(f"  Fallback → EX= : {fallback_wins}")
if ENABLE_COLUMN_LINKING:
    print(f"  Column cache hit : {len(_col_emb_cache)} databases pre-encoded")
    if ENABLE_AR_COL_DESC:
        total_descs = sum(len(v) for v in _ar_col_descriptions.values())
        print(f"  Arabic descriptions: {total_descs} columns across {len(_ar_col_descriptions)} databases")
if ENABLE_BILINGUAL_SCHEMA:
    total_glosses = sum(len(v) for v in _ar_table_glosses.values())
    print(f"  Arabic table glosses: {total_glosses} tables across {len(_ar_table_glosses)} databases")
if ENABLE_VALUE_INJECTION:
    total_cached_vals = sum(len(v["values"]) for v in _db_value_emb_cache.values())
    print(f"  Value embeddings: {total_cached_vals} values across {len(_db_value_emb_cache)} databases")
if ENABLE_BILINGUAL_EMBEDDINGS:
    print(f"  Bilingual column embeddings: (EN+AR in vector space)")
if ENABLE_SAMPLE_ROWS:
    print(f"  Sample rows: {SAMPLE_ROWS_LIMIT} rows × {SAMPLE_ROWS_MAX_TABLES} tables per prompt")
print()

V2_EX, V2_TOTAL = 603, 1032
V4_EX, V4_TOTAL = 735, 1034
delta_v2 = ex_count - V2_EX
delta_v2_pct = 100*ex_count/max(ex_total,1) - 100*V2_EX/V2_TOTAL
delta_v4 = ex_count - V4_EX
delta_v4_pct = 100*ex_count/max(ex_total,1) - 100*V4_EX/V4_TOTAL
print(f"  vs V2 Baseline (58.4% = 603/1032):")
print(f"     Delta: {'+' if delta_v2 >= 0 else ''}{delta_v2} samples"
      f" ({'+' if delta_v2_pct >= 0 else ''}{delta_v2_pct:.1f}%)")
print(f"  vs V4 Baseline (71.1% = 735/1034):")
print(f"     Delta: {'+' if delta_v4 >= 0 else ''}{delta_v4} samples"
      f" ({'+' if delta_v4_pct >= 0 else ''}{delta_v4_pct:.1f}%)")
print()


# Per-database breakdown
print(f"Per-database EX breakdown:")
db_stats = defaultdict(lambda: {"correct": 0, "total": 0})
for r in results:
    if r["ex"] is not None:
        db_stats[r["db_id"]]["total"] += 1
        if r["ex"]:
            db_stats[r["db_id"]]["correct"] += 1

for db_id in sorted(db_stats, key=lambda d: db_stats[d]["correct"]/max(db_stats[d]["total"],1), reverse=True):
    s = db_stats[db_id]
    pct = 100 * s["correct"] / s["total"]
    bar = "█" * int(pct // 5) + "░" * (20 - int(pct // 5))
    print(f"   {db_id:30s} {s['correct']:3d}/{s['total']:3d} ({pct:5.1f}%) {bar}")


# Save results
config_str = (
    f"MSchema={'ON' if ENABLE_MSCHEMA else 'OFF'} | "
    f"ValHints={'ON' if ENABLE_VALUE_HINTS else 'OFF'} | "
    f"ColLink={'emb,K='+str(COL_LINK_TOP_K) if ENABLE_COLUMN_LINKING else 'OFF'} | "
    f"ArDesc={'ON' if ENABLE_AR_COL_DESC else 'OFF'} | "
    f"EnHint={'ON' if ENABLE_ENGLISH_HINT else 'OFF'} | "
    f"BilingualSchema={'ON' if ENABLE_BILINGUAL_SCHEMA else 'OFF'} | "
    f"ValueInject={'K='+str(VALUE_INJECTION_TOP_K) if ENABLE_VALUE_INJECTION else 'OFF'} | "
    f"BilingualEmb={'ON' if ENABLE_BILINGUAL_EMBEDDINGS else 'OFF'} | "
    f"SampleRows={'ON' if ENABLE_SAMPLE_ROWS else 'OFF'} | "
    f"SC={'N='+str(SC_NUM_CANDIDATES)+',batch='+str(SC_BATCH_SIZE) if ENABLE_SELF_CONSISTENCY else 'OFF'} | "
    f"MultiTemp={','.join(f'{t}x{c}' for t,c in SC_TEMP_TIERS) if ENABLE_MULTI_TEMP else 'OFF'} | "
    f"LowConfFB={'≤'+str(LOW_CONF_VOTE_THRESHOLD) if ENABLE_LOW_CONF_FALLBACK else 'OFF'} | "
    f"Correct={'ON' if ENABLE_SELF_CORRECTION else 'OFF'} | "
    f"ID={'ON' if ENABLE_ID_FIXING else 'OFF'} | "
    f"Filter={'ON' if ENABLE_FILTER_REMOVAL else 'OFF'} | "
    f"SQLValGround={'t='+str(SQL_VALUE_SIM_THRESHOLD) if ENABLE_SQL_VALUE_GROUNDING else 'OFF'}"
)

output_path = "./data/eval_results_7b_dev_v4.json"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
with open(output_path, "w", encoding="utf-8") as f:
    json.dump({
        "model": FINETUNE_7B_BASE,
        "eval_split": "dev", "eval_count": len(results),
        "config": config_str,
        "tier1_config": {
            "mschema": ENABLE_MSCHEMA, "value_hints": ENABLE_VALUE_HINTS,
            "column_linking": ENABLE_COLUMN_LINKING,
            "col_link_top_k": COL_LINK_TOP_K if ENABLE_COLUMN_LINKING else 0,
            "ar_col_descriptions": ENABLE_AR_COL_DESC,
            "english_hint": ENABLE_ENGLISH_HINT,
            "bilingual_schema": ENABLE_BILINGUAL_SCHEMA,
            "value_injection": ENABLE_VALUE_INJECTION,
            "value_injection_top_k": VALUE_INJECTION_TOP_K if ENABLE_VALUE_INJECTION else 0,
            "bilingual_embeddings": ENABLE_BILINGUAL_EMBEDDINGS,
            "sample_rows": ENABLE_SAMPLE_ROWS,
            "self_consistency": ENABLE_SELF_CONSISTENCY,
            "sc_n": SC_NUM_CANDIDATES if ENABLE_SELF_CONSISTENCY else 0,
            "sc_temp": SC_TEMPERATURE if ENABLE_SELF_CONSISTENCY else 0,
            "sc_batch_size": SC_BATCH_SIZE if ENABLE_SELF_CONSISTENCY else 0,
            "multi_temp": ENABLE_MULTI_TEMP,
            "temp_tiers": SC_TEMP_TIERS if ENABLE_MULTI_TEMP else None,
            "low_conf_fallback": ENABLE_LOW_CONF_FALLBACK,
            "low_conf_threshold": LOW_CONF_VOTE_THRESHOLD if ENABLE_LOW_CONF_FALLBACK else 0,
            "low_conf_extra": LOW_CONF_EXTRA_CANDIDATES if ENABLE_LOW_CONF_FALLBACK else 0,
            "self_correction": ENABLE_SELF_CORRECTION,
            "max_retries": SC_MAX_RETRIES if ENABLE_SELF_CORRECTION else 0,
        },
        "metrics": {
            "EX": f"{100*ex_count/max(ex_total,1):.1f}%",
            "EX_count": f"{ex_count}/{ex_total}",
            "EM": f"{100*em_count/total:.1f}%",
            "BLEU": f"{avg_bleu:.3f}",
        },
        "stats": {
            "sc_votes": sc_votes, "sc_fallbacks": sc_fallbacks,
            "self_corrections": self_corrections, "total_retries": total_retries,
            "fallback_triggered": fallback_used, "fallback_wins": fallback_wins,
            "elapsed_s": round(elapsed),
            "s_per_sample": round(elapsed / total, 1),
        },
        "results": results,
    }, f, ensure_ascii=False, indent=2)

print(f"\n Results saved -> {output_path}")
print(f"   Config: {config_str}")

In [ ]:
import time

for i in range(10000):
  print(i)
  time.sleep(90)

## Results Summary

In [ ]:
#@title Results Summary { display-mode: "form" }

total = len(results)
ex_count = sum(1 for r in results if r["ex"] is True)
ex_total = sum(1 for r in results if r["ex"] is not None)
em_count = sum(1 for r in results if r["em"] is True)
avg_bleu = sum(r["bleu"] for r in results) / total
avg_sqam = sum(r["sqam"] for r in results) / total
avg_tsed = sum(r["tsed"] for r in results) / total

print("=" * 60)
print(f"  RESULTS: Qwen2.5-Coder-7B + Arabic QLoRA Adapter")
print(f"  Evaluation: Ar-Spider dev.json ({total} samples)")
print(f"  Training: Ar-Spider train.json (zero leakage)")
print("=" * 60)
print(f"  EX  (Execution Accuracy) : {ex_count}/{ex_total} = {100*ex_count/max(ex_total,1):.1f}%")
print(f"  EM  (Exact Match)        : {em_count}/{total} = {100*em_count/total:.1f}%")
print(f"  BLEU                     : {avg_bleu:.3f}")
print(f"  SQAM                     : {avg_sqam:.3f}")
print(f"  TSED                     : {avg_tsed:.3f}")
print("=" * 60)

# Per-database breakdown
print(f"\n Per-database EX breakdown:")
from collections import defaultdict
db_stats = defaultdict(lambda: {"correct": 0, "total": 0})
for r in results:
    if r["ex"] is not None:
        db_stats[r["db_id"]]["total"] += 1
        if r["ex"]:
            db_stats[r["db_id"]]["correct"] += 1

for db_id in sorted(db_stats, key=lambda d: db_stats[d]["correct"]/max(db_stats[d]["total"],1), reverse=True):
    s = db_stats[db_id]
    pct = 100 * s["correct"] / s["total"]
    bar = "█" * int(pct // 5) + "░" * (20 - int(pct // 5))
    print(f"   {db_id:30s} {s['correct']:3d}/{s['total']:3d} ({pct:5.1f}%) {bar}")

# Error analysis
print(f"\n Error samples (EX=, lowest BLEU):")
errors = sorted([r for r in results if r["ex"] is False], key=lambda r: r["bleu"])
for r in errors[:10]:
    print(f"   {r['id']} | {r['db_id']}")
    print(f"     Q: {r['question'][:60]}")
    print(f"     Gold: {r['gold_sql'][:60]}")
    print(f"     Pred: {r['pred_sql'][:60]}")
    print(f"     BLEU={r['bleu']:.3f} SQAM={r['sqam']:.3f}")
    print()

## Save Results to JSON

In [ ]:
#@title Save Results to JSON { display-mode: "form" }

output_path = "/content/eval_results_7b_dev_Arabic_DOCUMENTED_FULL.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump({
        "model": FINETUNE_7B_BASE,
        "adapter": str(ADAPTER_7B_OUTPUT),
        "eval_split": "dev",
        "pipeline_mode": "documented_full_option_B",
        "components_active": {
            "self_correction": ENABLE_SELF_CORRECTION,
            "id_fixing": ENABLE_ID_FIXING,
            "filter_removal": ENABLE_FILTER_REMOVAL,
            "sql_value_grounding": ENABLE_SQL_VALUE_GROUNDING,
            "low_conf_fallback": ENABLE_LOW_CONF_FALLBACK,
            "multi_temp_voting": ENABLE_MULTI_TEMP,
        },
        "seeding": {
            "deterministic": SC_DETERMINISTIC_SEEDS if 'SC_DETERMINISTIC_SEEDS' in globals() else False,
            "base_seed": SC_BASE_SEED if 'SC_BASE_SEED' in globals() else None,
        },
        "eval_count": len(results),
        "metrics": {
            "EX": f"{100*ex_count/max(ex_total,1):.1f}%",
            "EM": f"{100*em_count/total:.1f}%",
            "BLEU": f"{avg_bleu:.3f}",
            "SQAM": f"{avg_sqam:.3f}",
            "TSED": f"{avg_tsed:.3f}",
        },
        "results": results,
    }, f, ensure_ascii=False, indent=2)

print(f"Results saved → {output_path}")
print(f"   Download with: from google.colab import files; files.download('{output_path}')")

## Copy Results to Google Drive

In [ ]:
#@title Copy Results to Google Drive { display-mode: "form" }

import shutil

RESULTS_LOCAL = "/content/eval_results_7b_dev_Arabic_DOCUMENTED_FULL.json"
RESULTS_DRIVE = "/content/drive/MyDrive/eval_results_7b_dev_AR_documented_full.json"  #@param {type:"string"}

shutil.copy2(RESULTS_LOCAL, RESULTS_DRIVE)
print(f"\u2705 Results copied to: {RESULTS_DRIVE}")

## EM Normalizer Impact Analysis

In [ ]:
import json, re, math
from collections import Counter, defaultdict
from typing import List, Tuple

# Load Results

with open("/content/eval_results_7b_dev_Arabic_DOCUMENTED_FULL.json", "r", encoding="utf-8") as f:
    data = json.load(f)

results = data["results"]
print(f"Loaded {len(results)} samples")
print(f"Reported metrics: {data['metrics']}")
print()


def normalize_v0(sql):
    """Original normalizer from your script."""
    s = sql.strip().rstrip(";").strip().lower()
    s = re.sub(r'\s+', ' ', s)
    s = re.sub(r'[`"\[\]]', '', s)
    s = re.sub(r'\(\s+', '(', s)
    s = re.sub(r'\s+\)', ')', s)
    s = re.sub(
        r'\bas\s+(?!integer|real|text|numeric|blob|int|varchar|char|'
        r'float|double|boolean|date|time|datetime)\w+',
        '', s
    )
    return re.sub(r'\s+', ' ', s).strip()


# Strip alias prefixes (T1., T2.)

def normalize_v1(sql):
    s = normalize_v0(sql)
    s = re.sub(r'\bt\d+\.', '', s)
    return re.sub(r'\s+', ' ', s).strip()


# Remove explicit ASC

def normalize_v2(sql):
    s = normalize_v1(sql)
    s = re.sub(r'\basc\b', '', s)
    return re.sub(r'\s+', ' ', s).strip()


# Normalize equivalences (INNER JOIN, <>, COUNT(1))

def normalize_v3(sql):
    s = normalize_v2(sql)
    s = re.sub(r'\binner\s+join\b', 'join', s)
    s = s.replace('<>', '!=')
    s = re.sub(r'count\s*\(\s*1\s*\)', 'count(*)', s)
    return re.sub(r'\s+', ' ', s).strip()


# Unify quote style (double → single)

def normalize_v4(sql):
    s = normalize_v3(sql)
    s = s.replace('"', "'")
    return s


# Sort SELECT columns

def _split_at_depth_zero(s, delimiter_re):
    tokens = re.split(f'({delimiter_re})', s, flags=re.I)
    items, depth, current = [], 0, ""
    for token in tokens:
        depth += token.count('(') - token.count(')')
        if re.fullmatch(delimiter_re, token.strip(), re.I) and depth == 0:
            if current.strip():
                items.append(current.strip())
            current = ""
        else:
            current += token
    if current.strip():
        items.append(current.strip())
    return items

def _sort_select(sql):
    m = re.match(r'(select\s+(?:distinct\s+)?)(.*?)(\s+from\b.*)', sql, re.I | re.DOTALL)
    if not m:
        return sql
    prefix, cols_str, rest = m.group(1), m.group(2), m.group(3)
    items = _split_at_depth_zero(cols_str, r',')
    if len(items) <= 1:
        return sql
    items = sorted(item.strip() for item in items)
    return prefix + ' , '.join(items) + rest

def normalize_v5(sql):
    s = normalize_v4(sql)
    s = _sort_select(s)
    return re.sub(r'\s+', ' ', s).strip()


# Sort WHERE AND-conditions

def _sort_where(sql):
    where_m = re.search(
        r'(\bwhere\b\s+)(.*?)(\s*(?:\bgroup\b|\border\b|\blimit\b|\bhaving\b|\bunion\b|\bintersect\b|\bexcept\b|$))',
        sql, re.I | re.DOTALL
    )
    if not where_m:
        return sql
    prefix = where_m.group(1)
    body = where_m.group(2)
    suffix = where_m.group(3)
    if re.search(r'\bor\b', body, re.I):
        return sql
    conditions = _split_at_depth_zero(body, r'\band\b')
    if len(conditions) <= 1:
        return sql
    conditions = sorted(c.strip() for c in conditions)
    new_where = prefix + ' and '.join(conditions) + suffix
    return sql[:where_m.start()] + new_where + sql[where_m.end():]

def normalize_v6(sql):
    s = normalize_v5(sql)
    s = _sort_where(s)
    return re.sub(r'\s+', ' ', s).strip()


# Sort IN-list values

def _sort_in_lists(sql):
    def sort_in(match):
        prefix = match.group(1)
        values_str = match.group(2)
        if 'select' in values_str.lower():
            return match.group(0)
        items = [v.strip() for v in values_str.split(',')]
        return prefix + ', '.join(sorted(items)) + ')'
    return re.sub(r'(\bin\s*\()([^)]+)\)', sort_in, sql, flags=re.I)

def normalize_v7(sql):
    s = normalize_v6(sql)
    s = _sort_in_lists(s)
    return re.sub(r'\s+', ' ', s).strip()


# Normalize ON clause operand order

def _normalize_on(sql):
    def norm_on(match):
        left = match.group(1).strip()
        right = match.group(2).strip()
        if left > right:
            left, right = right, left
        return f"on {left} = {right}"
    return re.sub(r'\bon\s+(\S+)\s*=\s*(\S+)', norm_on, sql, flags=re.I)

def normalize_v8(sql):
    s = normalize_v7(sql)
    s = _normalize_on(s)
    return re.sub(r'\s+', ' ', s).strip()


# Canonicalize aliases (T1=alpha-first table)

def _canonicalize_aliases(sql):
    alias_defs = re.findall(
        r'(?:from|join)\s+(\w+)\s+(?:as\s+)?(\w+)', sql, re.I
    )
    if not alias_defs:
        return sql
    kw = {'join','inner','left','right','outer','on','where','group',
          'order','having','limit','cross','and','or','not','select'}
    alias_defs = [(t, a) for t, a in alias_defs if a.lower() not in kw]
    if not alias_defs:
        return sql
    table_order = {}
    for table, old_alias in alias_defs:
        tl = table.lower()
        if tl not in table_order:
            table_order[tl] = table
    sorted_tables = sorted(table_order.keys())
    table_to_new = {}
    for i, tl in enumerate(sorted_tables):
        table_to_new[tl] = f"t{i+1}"
    old_to_canonical = {}
    for table, old_alias in alias_defs:
        new_alias = table_to_new.get(table.lower())
        if new_alias:
            old_to_canonical[old_alias.lower()] = new_alias
    result = sql
    for old, new in sorted(old_to_canonical.items(), key=lambda x: -len(x[0])):
        result = re.sub(r'\b' + re.escape(old) + r'\b', new, result)
    return result

def normalize_v9(sql):
    s = normalize_v8(sql)
    s = _canonicalize_aliases(s)
    # Re-strip alias prefixes after canonicalization
    s = re.sub(r'\bt\d+\.', '', s)
    return re.sub(r'\s+', ' ', s).strip()


# Normalize operator spacing

def normalize_v10(sql):
    s = normalize_v9(sql)
    s = re.sub(r'\s*(=|!=|>=|<=|>|<)\s*', r' \1 ', s)
    return re.sub(r'\s+', ' ', s).strip()


# Normalize string quoting in WHERE values

def _normalize_value_quotes(sql):
    """Strip single quotes from simple string values in comparisons."""
    # Match: col = 'value' → col = value  AND  col = value → col = value
    result = re.sub(r"(\s*(?:=|!=|<>|like)\s*)'([^']*)'", r'\1\2', sql, flags=re.I)
    # Also strip from IN lists: IN ('a', 'b') → IN (a, b)
    result = re.sub(r"(\bin\s*\()([^)]+)\)", lambda m: m.group(1) + m.group(2).replace("'","") + ")", result, flags=re.I)
    return result

def normalize_v11(sql):
    s = normalize_v10(sql)
    s = _normalize_value_quotes(s)
    return re.sub(r'\s+', ' ', s).strip()


# Normalize numeric equivalences

def _normalize_quoted_numbers(sql):
    """Strip quotes from numeric values: '1' → 1, '100' → 100."""
    return re.sub(r"'(\d+(?:\.\d+)?)'", r'\1', sql)

def normalize_v12(sql):
    s = normalize_v11(sql)
    s = _normalize_quoted_numbers(s)
    return re.sub(r'\s+', ' ', s).strip()


# Sort GROUP BY columns

def _sort_groupby(sql):
    m = re.search(
        r'(\bgroup by\b\s+)(.*?)(\s*(?:\bhaving\b|\border\b|\blimit\b|$))',
        sql, re.I | re.DOTALL
    )
    if not m:
        return sql
    prefix, items_str, suffix = m.group(1), m.group(2), m.group(3)
    items = sorted(i.strip() for i in items_str.split(','))
    return sql[:m.start()] + prefix + ' , '.join(items) + suffix + sql[m.end():]

def normalize_v13(sql):
    s = normalize_v12(sql)
    s = _sort_groupby(s)
    return re.sub(r'\s+', ' ', s).strip()


# Run All Normalizers

normalizers = [
    ("V0  Original (baseline)",            normalize_v0),
    ("V1  + Strip T1./T2. prefixes",       normalize_v1),
    ("V2  + Remove explicit ASC",          normalize_v2),
    ("V3  + Equivalences (INNER JOIN, <>)", normalize_v3),
    ("V4  + Unify quotes (\" → ')",        normalize_v4),
    ("V5  + Sort SELECT columns",          normalize_v5),
    ("V6  + Sort WHERE conditions",        normalize_v6),
    ("V7  + Sort IN-list values",          normalize_v7),
    ("V8  + Normalize ON operand order",   normalize_v8),
    ("V9  + Canonicalize aliases",         normalize_v9),
    ("V10 + Normalize operator spacing",   normalize_v10),
    ("V11 + Strip value quotes (= 'x' → = x)", normalize_v11),
    ("V12 + Strip numeric quotes ('1' → 1)",    normalize_v12),
    ("V13 + Sort GROUP BY columns",        normalize_v13),
]

print("=" * 78)
print(f"  EM NORMALIZER IMPACT ANALYSIS — {len(results)} samples")
print("=" * 78)
print()
print(f"  {'Normalizer':<45s} {'EM':>6s} {'Count':>6s} {'Δ':>6s}")
print(f"  {'-'*45} {'-'*6} {'-'*6} {'-'*6}")

baseline_em = None
prev_em = 0

for name, norm_fn in normalizers:
    em_count = 0
    for r in results:
        pred = r["pred_sql"]
        gold = r["gold_sql"]
        if norm_fn(pred) == norm_fn(gold):
            em_count += 1
    em_pct = 100 * em_count / len(results)

    if baseline_em is None:
        baseline_em = em_count
        delta_str = "  —"
    else:
        delta = em_count - prev_em
        delta_str = f"+{delta:>3d}" if delta > 0 else f" {delta:>3d}"

    print(f"  {name:<45s} {em_pct:5.1f}% {em_count:>5d}  {delta_str}")
    prev_em = em_count

total_gain = prev_em - baseline_em
print(f"  {'-'*45} {'-'*6} {'-'*6} {'-'*6}")
print(f"  {'TOTAL GAIN':<45s} {'+' + f'{100*total_gain/len(results):.1f}':>5s}% {'+' + str(total_gain):>5s}")
print()


print("=" * 78)
print("  SAMPLES RESCUED BY BEST NORMALIZER (V13) vs ORIGINAL (V0)")
print("=" * 78)
print()

rescued = []
lost = []
for r in results:
    v0_match = normalize_v0(r["pred_sql"]) == normalize_v0(r["gold_sql"])
    vbest_match = normalize_v13(r["pred_sql"]) == normalize_v13(r["gold_sql"])
    if vbest_match and not v0_match:
        rescued.append(r)
    elif v0_match and not vbest_match:
        lost.append(r)

print(f"  Rescued (V0= → V10=): {len(rescued)} samples")
print(f"  Lost    (V0= → V10=): {len(lost)} samples")
print(f"  Net gain: +{len(rescued) - len(lost)}")
print()

if lost:
    print("   SAMPLES LOST (need investigation):")
    for r in lost[:10]:
        print(f"    {r['id']} | {r['db_id']}")
        print(f"      PRED: {r['pred_sql'][:90]}")
        print(f"      GOLD: {r['gold_sql'][:90]}")
        print(f"      V0 norm PRED: {normalize_v0(r['pred_sql'])[:90]}")
        print(f"      V0 norm GOLD: {normalize_v0(r['gold_sql'])[:90]}")
        print(f"      V13 norm PRED: {normalize_v13(r['pred_sql'])[:90]}")
        print(f"      V13 norm GOLD: {normalize_v13(r['gold_sql'])[:90]}")
        print()

# Breakdown: rescued by EX status

print("=" * 78)
print("  RESCUED SAMPLES BY EX STATUS")
print("=" * 78)
print()

rescued_ex_true = [r for r in rescued if r["ex"] is True]
rescued_ex_false = [r for r in rescued if r["ex"] is False]
rescued_ex_none = [r for r in rescued if r["ex"] is None]

print(f"  Rescued + EX= (correct queries, format mismatch only): {len(rescued_ex_true)}")
print(f"  Rescued + EX= (coincidental string match):            {len(rescued_ex_false)}")
print(f"  Rescued + EX= (no DB to verify):                     {len(rescued_ex_none)}")
print()


# Category breakdown: what type of mismatch does each fix resolve?

print("=" * 78)
print("  MISMATCH CATEGORY ANALYSIS (rescued samples)")
print("=" * 78)
print()

categories = Counter()
for r in rescued:
    pred_v0 = normalize_v0(r["pred_sql"])
    gold_v0 = normalize_v0(r["gold_sql"])
    reasons = []

    # Check what specifically differs
    pred_v1 = normalize_v1(r["pred_sql"])
    gold_v1 = normalize_v1(r["gold_sql"])
    if pred_v1 == gold_v1 and pred_v0 != gold_v0:
        reasons.append("ALIAS_PREFIX (T1.col → col)")

    pred_v2 = normalize_v2(r["pred_sql"])
    gold_v2 = normalize_v2(r["gold_sql"])
    if pred_v2 == gold_v2 and pred_v1 != gold_v1:
        reasons.append("EXPLICIT_ASC")

    pred_v3 = normalize_v3(r["pred_sql"])
    gold_v3 = normalize_v3(r["gold_sql"])
    if pred_v3 == gold_v3 and pred_v2 != gold_v2:
        reasons.append("EQUIVALENCE (INNER JOIN, <>)")

    pred_v4 = normalize_v4(r["pred_sql"])
    gold_v4 = normalize_v4(r["gold_sql"])
    if pred_v4 == gold_v4 and pred_v3 != gold_v3:
        reasons.append("QUOTE_STYLE")

    pred_v5 = normalize_v5(r["pred_sql"])
    gold_v5 = normalize_v5(r["gold_sql"])
    if pred_v5 == gold_v5 and pred_v4 != gold_v4:
        reasons.append("SELECT_ORDER")

    pred_v6 = normalize_v6(r["pred_sql"])
    gold_v6 = normalize_v6(r["gold_sql"])
    if pred_v6 == gold_v6 and pred_v5 != gold_v5:
        reasons.append("WHERE_ORDER")

    pred_v9 = normalize_v9(r["pred_sql"])
    gold_v9 = normalize_v9(r["gold_sql"])
    if pred_v9 == gold_v9 and pred_v6 != gold_v6:
        reasons.append("ALIAS_CANON / ON_ORDER / IN_LIST")

    pred_vbest = normalize_v13(r["pred_sql"])
    gold_vbest = normalize_v13(r["gold_sql"])
    if pred_vbest == gold_vbest and pred_v9 != gold_v9:
        reasons.append("OPERATOR_SPACING")

    if not reasons:
        reasons.append("COMBINED (multiple fixes needed)")

    for reason in reasons:
        categories[reason] += 1

for cat, count in categories.most_common():
    bar = "█" * int(count / 2)
    print(f"  {cat:<45s} {count:>4d}  {bar}")
print()


# Per-database EM improvement

print("=" * 78)
print("  PER-DATABASE EM IMPROVEMENT (V0 → V13)")
print("=" * 78)
print()
print(f"  {'Database':<30s} {'V0 EM':>7s} {'V10 EM':>7s} {'Δ':>5s} {'Total':>6s}")
print(f"  {'-'*30} {'-'*7} {'-'*7} {'-'*5} {'-'*6}")

db_stats = defaultdict(lambda: {"v0": 0, "v10": 0, "total": 0})
for r in results:
    db = r["db_id"]
    db_stats[db]["total"] += 1
    if normalize_v0(r["pred_sql"]) == normalize_v0(r["gold_sql"]):
        db_stats[db]["v0"] += 1
    if normalize_v13(r["pred_sql"]) == normalize_v13(r["gold_sql"]):
        db_stats[db]["v10"] += 1

# Sort by improvement
sorted_dbs = sorted(db_stats.items(),
                     key=lambda x: (x[1]["v10"] - x[1]["v0"]) / max(x[1]["total"], 1),
                     reverse=True)

for db, s in sorted_dbs[:25]:
    v0_pct = 100 * s["v0"] / s["total"]
    v10_pct = 100 * s["v10"] / s["total"]
    delta = s["v10"] - s["v0"]
    if delta > 0:
        print(f"  {db:<30s} {v0_pct:5.1f}% {v10_pct:6.1f}% {'+' + str(delta):>4s}  ({s['total']:>4d})")

print(f"\n  ... showing top 25 databases with improvement")


print()
print("=" * 78)
print("  REMAINING GAPS: EX= but EM= even with V13 normalizer")
print("=" * 78)
print()

still_gap = []
for r in results:
    is_ex_true = r["ex"] is True
    is_em_vbest = normalize_v13(r["pred_sql"]) == normalize_v13(r["gold_sql"])
    if is_ex_true and not is_em_vbest:
        still_gap.append(r)

print(f"  Total EX= but EM= (V13): {len(still_gap)} samples")
print(f"  (These need MODEL improvements, not normalizer improvements)")
print()

# Categorize remaining gaps
gap_reasons = Counter()
for r in still_gap:
    pred = normalize_v13(r["pred_sql"])
    gold = normalize_v13(r["gold_sql"])

    reasons = []

    # Different number of JOINs
    pred_joins = len(re.findall(r'\bjoin\b', pred))
    gold_joins = len(re.findall(r'\bjoin\b', gold))
    if pred_joins != gold_joins:
        reasons.append("DIFFERENT_JOIN_COUNT")

    # Different subquery nesting
    if pred.count('select') != gold.count('select'):
        reasons.append("DIFFERENT_NESTING")

    # Different string values
    pred_strs = set(re.findall(r"'([^']+)'", pred))
    gold_strs = set(re.findall(r"'([^']+)'", gold))
    if pred_strs != gold_strs and (pred_strs or gold_strs):
        reasons.append("DIFFERENT_STRING_VALUES")

    # Different aggregation
    for agg in ['count', 'sum', 'avg', 'min', 'max']:
        if (agg + '(') in pred and (agg + '(') not in gold:
            reasons.append("WRONG_AGG"); break
        if (agg + '(') not in pred and (agg + '(') in gold:
            reasons.append("MISSING_AGG"); break

    # DISTINCT mismatch
    if 'distinct' in pred and 'distinct' not in gold:
        reasons.append("EXTRA_DISTINCT")
    elif 'distinct' not in pred and 'distinct' in gold:
        reasons.append("MISSING_DISTINCT")

    if not reasons:
        reasons.append("OTHER_SURFACE_DIFF")

    for reason in reasons:
        gap_reasons[reason] += 1

for cat, count in gap_reasons.most_common():
    pct = 100 * count / len(still_gap)
    bar = "█" * int(pct / 3)
    print(f"  {cat:<30s} {count:>4d} ({pct:4.1f}%) {bar}")

print()
print(f"  Examples of remaining gaps:")
for r in still_gap[:8]:
    print(f"    {r['id']} | {r['db_id']}")
    print(f"      PRED: {normalize_v13(r['pred_sql'])[:100]}")
    print(f"      GOLD: {normalize_v13(r['gold_sql'])[:100]}")
    print()


# Final Summary

ex_count = sum(1 for r in results if r["ex"] is True)
ex_total = sum(1 for r in results if r["ex"] is not None)
em_v0 = sum(1 for r in results if normalize_v0(r["pred_sql"]) == normalize_v0(r["gold_sql"]))
em_vbest = sum(1 for r in results if normalize_v13(r["pred_sql"]) == normalize_v13(r["gold_sql"]))

print("=" * 78)
print("  FINAL SUMMARY")
print("=" * 78)
print()
print(f"  EX  (unchanged):        {ex_count}/{ex_total} = {100*ex_count/ex_total:.1f}%")
print(f"  EM  (V0 original):      {em_v0}/{len(results)} = {100*em_v0/len(results):.1f}%")
print(f"  EM  (V10 improved):     {em_vbest}/{len(results)} = {100*em_vbest/len(results):.1f}%")
print(f"  EM  improvement:        +{em_vbest - em_v0} samples (+{100*(em_vbest-em_v0)/len(results):.1f}%)")
print(f"  EX−EM gap (V0):        {100*ex_count/ex_total - 100*em_v0/len(results):.1f}%")
print(f"  EX−EM gap (V13):       {100*ex_count/ex_total - 100*em_vbest/len(results):.1f}%")
print(f"  Theoretical EM ceiling: {100*ex_count/len(results):.1f}% (if all EX= → EM=)")
print()

## Save Normalized Results

In [ ]:
import json, re, math, os
from collections import Counter, defaultdict
from typing import List, Tuple

# Configuration

INPUT_PATH = "/content/eval_results_7b_dev_Arabic_DOCUMENTED_FULL.json"  #@param {type:"string"}
OUTPUT_PATH = "/content/eval_results_7b_dev_normalized.json"  #@param {type:"string"}

# Load Results

with open(INPUT_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

results = data["results"]
print(f"Loaded {len(results)} samples")
print(f"Reported metrics: {data['metrics']}")
print()


def normalize_v0(sql):
    """Original normalizer from your script."""
    s = sql.strip().rstrip(";").strip().lower()
    s = re.sub(r'\s+', ' ', s)
    s = re.sub(r'[`"\[\]]', '', s)
    s = re.sub(r'\(\s+', '(', s)
    s = re.sub(r'\s+\)', ')', s)
    s = re.sub(
        r'\bas\s+(?!integer|real|text|numeric|blob|int|varchar|char|'
        r'float|double|boolean|date|time|datetime)\w+',
        '', s
    )
    return re.sub(r'\s+', ' ', s).strip()


# Strip alias prefixes (T1., T2.)

def normalize_v1(sql):
    s = normalize_v0(sql)
    s = re.sub(r'\bt\d+\.', '', s)
    return re.sub(r'\s+', ' ', s).strip()


# Remove explicit ASC

def normalize_v2(sql):
    s = normalize_v1(sql)
    s = re.sub(r'\basc\b', '', s)
    return re.sub(r'\s+', ' ', s).strip()


# Normalize equivalences (INNER JOIN, <>, COUNT(1))

def normalize_v3(sql):
    s = normalize_v2(sql)
    s = re.sub(r'\binner\s+join\b', 'join', s)
    s = s.replace('<>', '!=')
    s = re.sub(r'count\s*\(\s*1\s*\)', 'count(*)', s)
    return re.sub(r'\s+', ' ', s).strip()


# Unify quote style (double → single)

def normalize_v4(sql):
    s = normalize_v3(sql)
    s = s.replace('"', "'")
    return s


# Sort SELECT columns

def _split_at_depth_zero(s, delimiter_re):
    tokens = re.split(f'({delimiter_re})', s, flags=re.I)
    items, depth, current = [], 0, ""
    for token in tokens:
        depth += token.count('(') - token.count(')')
        if re.fullmatch(delimiter_re, token.strip(), re.I) and depth == 0:
            if current.strip():
                items.append(current.strip())
            current = ""
        else:
            current += token
    if current.strip():
        items.append(current.strip())
    return items

def _sort_select(sql):
    m = re.match(r'(select\s+(?:distinct\s+)?)(.*?)(\s+from\b.*)', sql, re.I | re.DOTALL)
    if not m:
        return sql
    prefix, cols_str, rest = m.group(1), m.group(2), m.group(3)
    items = _split_at_depth_zero(cols_str, r',')
    if len(items) <= 1:
        return sql
    items = sorted(item.strip() for item in items)
    return prefix + ' , '.join(items) + rest

def normalize_v5(sql):
    s = normalize_v4(sql)
    s = _sort_select(s)
    return re.sub(r'\s+', ' ', s).strip()


# Sort WHERE AND-conditions

def _sort_where(sql):
    where_m = re.search(
        r'(\bwhere\b\s+)(.*?)(\s*(?:\bgroup\b|\border\b|\blimit\b|\bhaving\b|\bunion\b|\bintersect\b|\bexcept\b|$))',
        sql, re.I | re.DOTALL
    )
    if not where_m:
        return sql
    prefix = where_m.group(1)
    body = where_m.group(2)
    suffix = where_m.group(3)
    if re.search(r'\bor\b', body, re.I):
        return sql
    conditions = _split_at_depth_zero(body, r'\band\b')
    if len(conditions) <= 1:
        return sql
    conditions = sorted(c.strip() for c in conditions)
    new_where = prefix + ' and '.join(conditions) + suffix
    return sql[:where_m.start()] + new_where + sql[where_m.end():]

def normalize_v6(sql):
    s = normalize_v5(sql)
    s = _sort_where(s)
    return re.sub(r'\s+', ' ', s).strip()


# Sort IN-list values

def _sort_in_lists(sql):
    def sort_in(match):
        prefix = match.group(1)
        values_str = match.group(2)
        if 'select' in values_str.lower():
            return match.group(0)
        items = [v.strip() for v in values_str.split(',')]
        return prefix + ', '.join(sorted(items)) + ')'
    return re.sub(r'(\bin\s*\()([^)]+)\)', sort_in, sql, flags=re.I)

def normalize_v7(sql):
    s = normalize_v6(sql)
    s = _sort_in_lists(s)
    return re.sub(r'\s+', ' ', s).strip()


# Normalize ON clause operand order

def _normalize_on(sql):
    def norm_on(match):
        left = match.group(1).strip()
        right = match.group(2).strip()
        if left > right:
            left, right = right, left
        return f"on {left} = {right}"
    return re.sub(r'\bon\s+(\S+)\s*=\s*(\S+)', norm_on, sql, flags=re.I)

def normalize_v8(sql):
    s = normalize_v7(sql)
    s = _normalize_on(s)
    return re.sub(r'\s+', ' ', s).strip()


# Canonicalize aliases (T1=alpha-first table)

def _canonicalize_aliases(sql):
    alias_defs = re.findall(
        r'(?:from|join)\s+(\w+)\s+(?:as\s+)?(\w+)', sql, re.I
    )
    if not alias_defs:
        return sql
    kw = {'join','inner','left','right','outer','on','where','group',
          'order','having','limit','cross','and','or','not','select'}
    alias_defs = [(t, a) for t, a in alias_defs if a.lower() not in kw]
    if not alias_defs:
        return sql
    table_order = {}
    for table, old_alias in alias_defs:
        tl = table.lower()
        if tl not in table_order:
            table_order[tl] = table
    sorted_tables = sorted(table_order.keys())
    table_to_new = {}
    for i, tl in enumerate(sorted_tables):
        table_to_new[tl] = f"t{i+1}"
    old_to_canonical = {}
    for table, old_alias in alias_defs:
        new_alias = table_to_new.get(table.lower())
        if new_alias:
            old_to_canonical[old_alias.lower()] = new_alias
    result = sql
    for old, new in sorted(old_to_canonical.items(), key=lambda x: -len(x[0])):
        result = re.sub(r'\b' + re.escape(old) + r'\b', new, result)
    return result

def normalize_v9(sql):
    s = normalize_v8(sql)
    s = _canonicalize_aliases(s)
    # Re-strip alias prefixes after canonicalization
    s = re.sub(r'\bt\d+\.', '', s)
    return re.sub(r'\s+', ' ', s).strip()


# Normalize operator spacing

def normalize_v10(sql):
    s = normalize_v9(sql)
    s = re.sub(r'\s*(=|!=|>=|<=|>|<)\s*', r' \1 ', s)
    return re.sub(r'\s+', ' ', s).strip()


# Normalize string quoting in WHERE values

def _normalize_value_quotes(sql):
    """Strip single quotes from simple string values in comparisons."""
    result = re.sub(r"(\s*(?:=|!=|<>|like)\s*)'([^']*)'", r'\1\2', sql, flags=re.I)
    result = re.sub(r"(\bin\s*\()([^)]+)\)", lambda m: m.group(1) + m.group(2).replace("'","") + ")", result, flags=re.I)
    return result

def normalize_v11(sql):
    s = normalize_v10(sql)
    s = _normalize_value_quotes(s)
    return re.sub(r'\s+', ' ', s).strip()


# Normalize numeric equivalences

def _normalize_quoted_numbers(sql):
    """Strip quotes from numeric values: '1' → 1, '100' → 100."""
    return re.sub(r"'(\d+(?:\.\d+)?)'", r'\1', sql)

def normalize_v12(sql):
    s = normalize_v11(sql)
    s = _normalize_quoted_numbers(s)
    return re.sub(r'\s+', ' ', s).strip()


# Sort GROUP BY columns

def _sort_groupby(sql):
    m = re.search(
        r'(\bgroup by\b\s+)(.*?)(\s*(?:\bhaving\b|\border\b|\blimit\b|$))',
        sql, re.I | re.DOTALL
    )
    if not m:
        return sql
    prefix, items_str, suffix = m.group(1), m.group(2), m.group(3)
    items = sorted(i.strip() for i in items_str.split(','))
    return sql[:m.start()] + prefix + ' , '.join(items) + suffix + sql[m.end():]

def normalize_v13(sql):
    s = normalize_v12(sql)
    s = _sort_groupby(s)
    return re.sub(r'\s+', ' ', s).strip()


# Run All Normalizers

normalizers = [
    ("V0  Original (baseline)",            normalize_v0),
    ("V1  + Strip T1./T2. prefixes",       normalize_v1),
    ("V2  + Remove explicit ASC",          normalize_v2),
    ("V3  + Equivalences (INNER JOIN, <>)", normalize_v3),
    ("V4  + Unify quotes (\" → ')",        normalize_v4),
    ("V5  + Sort SELECT columns",          normalize_v5),
    ("V6  + Sort WHERE conditions",        normalize_v6),
    ("V7  + Sort IN-list values",          normalize_v7),
    ("V8  + Normalize ON operand order",   normalize_v8),
    ("V9  + Canonicalize aliases",         normalize_v9),
    ("V10 + Normalize operator spacing",   normalize_v10),
    ("V11 + Strip value quotes (= 'x' → = x)", normalize_v11),
    ("V12 + Strip numeric quotes ('1' → 1)",    normalize_v12),
    ("V13 + Sort GROUP BY columns",        normalize_v13),
]

print("=" * 78)
print(f"  EM NORMALIZER IMPACT ANALYSIS — {len(results)} samples")
print("=" * 78)
print()
print(f"  {'Normalizer':<45s} {'EM':>6s} {'Count':>6s} {'Δ':>6s}")
print(f"  {'-'*45} {'-'*6} {'-'*6} {'-'*6}")

baseline_em = None
prev_em = 0

for name, norm_fn in normalizers:
    em_count = 0
    for r in results:
        pred = r["pred_sql"]
        gold = r["gold_sql"]
        if norm_fn(pred) == norm_fn(gold):
            em_count += 1
    em_pct = 100 * em_count / len(results)

    if baseline_em is None:
        baseline_em = em_count
        delta_str = "  —"
    else:
        delta = em_count - prev_em
        delta_str = f"+{delta:>3d}" if delta > 0 else f" {delta:>3d}"

    print(f"  {name:<45s} {em_pct:5.1f}% {em_count:>5d}  {delta_str}")
    prev_em = em_count

total_gain = prev_em - baseline_em
print(f"  {'-'*45} {'-'*6} {'-'*6} {'-'*6}")
print(f"  {'TOTAL GAIN':<45s} {'+' + f'{100*total_gain/len(results):.1f}':>5s}% {'+' + str(total_gain):>5s}")
print()


print("=" * 78)
print("  SAMPLES RESCUED BY BEST NORMALIZER (V13) vs ORIGINAL (V0)")
print("=" * 78)
print()

rescued = []
lost = []
for r in results:
    v0_match = normalize_v0(r["pred_sql"]) == normalize_v0(r["gold_sql"])
    vbest_match = normalize_v13(r["pred_sql"]) == normalize_v13(r["gold_sql"])
    if vbest_match and not v0_match:
        rescued.append(r)
    elif v0_match and not vbest_match:
        lost.append(r)

print(f"  Rescued (V0= → V13=): {len(rescued)} samples")
print(f"  Lost    (V0= → V13=): {len(lost)} samples")
print(f"  Net gain: +{len(rescued) - len(lost)}")
print()

if lost:
    print("   SAMPLES LOST (need investigation):")
    for r in lost[:10]:
        print(f"    {r['id']} | {r['db_id']}")
        print(f"      PRED: {r['pred_sql'][:90]}")
        print(f"      GOLD: {r['gold_sql'][:90]}")
        print(f"      V0 norm PRED: {normalize_v0(r['pred_sql'])[:90]}")
        print(f"      V0 norm GOLD: {normalize_v0(r['gold_sql'])[:90]}")
        print(f"      V13 norm PRED: {normalize_v13(r['pred_sql'])[:90]}")
        print(f"      V13 norm GOLD: {normalize_v13(r['gold_sql'])[:90]}")
        print()

# Breakdown: rescued by EX status

print("=" * 78)
print("  RESCUED SAMPLES BY EX STATUS")
print("=" * 78)
print()

rescued_ex_true = [r for r in rescued if r["ex"] is True]
rescued_ex_false = [r for r in rescued if r["ex"] is False]
rescued_ex_none = [r for r in rescued if r["ex"] is None]

print(f"  Rescued + EX= (correct queries, format mismatch only): {len(rescued_ex_true)}")
print(f"  Rescued + EX= (coincidental string match):            {len(rescued_ex_false)}")
print(f"  Rescued + EX= (no DB to verify):                     {len(rescued_ex_none)}")
print()


# Category breakdown: what type of mismatch does each fix resolve?

print("=" * 78)
print("  MISMATCH CATEGORY ANALYSIS (rescued samples)")
print("=" * 78)
print()

categories = Counter()
for r in rescued:
    pred_v0 = normalize_v0(r["pred_sql"])
    gold_v0 = normalize_v0(r["gold_sql"])
    reasons = []

    pred_v1 = normalize_v1(r["pred_sql"])
    gold_v1 = normalize_v1(r["gold_sql"])
    if pred_v1 == gold_v1 and pred_v0 != gold_v0:
        reasons.append("ALIAS_PREFIX (T1.col → col)")

    pred_v2 = normalize_v2(r["pred_sql"])
    gold_v2 = normalize_v2(r["gold_sql"])
    if pred_v2 == gold_v2 and pred_v1 != gold_v1:
        reasons.append("EXPLICIT_ASC")

    pred_v3 = normalize_v3(r["pred_sql"])
    gold_v3 = normalize_v3(r["gold_sql"])
    if pred_v3 == gold_v3 and pred_v2 != gold_v2:
        reasons.append("EQUIVALENCE (INNER JOIN, <>)")

    pred_v4 = normalize_v4(r["pred_sql"])
    gold_v4 = normalize_v4(r["gold_sql"])
    if pred_v4 == gold_v4 and pred_v3 != gold_v3:
        reasons.append("QUOTE_STYLE")

    pred_v5 = normalize_v5(r["pred_sql"])
    gold_v5 = normalize_v5(r["gold_sql"])
    if pred_v5 == gold_v5 and pred_v4 != gold_v4:
        reasons.append("SELECT_ORDER")

    pred_v6 = normalize_v6(r["pred_sql"])
    gold_v6 = normalize_v6(r["gold_sql"])
    if pred_v6 == gold_v6 and pred_v5 != gold_v5:
        reasons.append("WHERE_ORDER")

    pred_v9 = normalize_v9(r["pred_sql"])
    gold_v9 = normalize_v9(r["gold_sql"])
    if pred_v9 == gold_v9 and pred_v6 != gold_v6:
        reasons.append("ALIAS_CANON / ON_ORDER / IN_LIST")

    pred_vbest = normalize_v13(r["pred_sql"])
    gold_vbest = normalize_v13(r["gold_sql"])
    if pred_vbest == gold_vbest and pred_v9 != gold_v9:
        reasons.append("OPERATOR_SPACING")

    if not reasons:
        reasons.append("COMBINED (multiple fixes needed)")

    for reason in reasons:
        categories[reason] += 1

for cat, count in categories.most_common():
    bar = "█" * int(count / 2)
    print(f"  {cat:<45s} {count:>4d}  {bar}")
print()


# Per-database EM improvement

print("=" * 78)
print("  PER-DATABASE EM IMPROVEMENT (V0 → V13)")
print("=" * 78)
print()
print(f"  {'Database':<30s} {'V0 EM':>7s} {'V13 EM':>7s} {'Δ':>5s} {'Total':>6s}")
print(f"  {'-'*30} {'-'*7} {'-'*7} {'-'*5} {'-'*6}")

db_stats = defaultdict(lambda: {"v0": 0, "v13": 0, "total": 0})
for r in results:
    db = r["db_id"]
    db_stats[db]["total"] += 1
    if normalize_v0(r["pred_sql"]) == normalize_v0(r["gold_sql"]):
        db_stats[db]["v0"] += 1
    if normalize_v13(r["pred_sql"]) == normalize_v13(r["gold_sql"]):
        db_stats[db]["v13"] += 1

sorted_dbs = sorted(db_stats.items(),
                     key=lambda x: (x[1]["v13"] - x[1]["v0"]) / max(x[1]["total"], 1),
                     reverse=True)

for db, s in sorted_dbs[:25]:
    v0_pct = 100 * s["v0"] / s["total"]
    v13_pct = 100 * s["v13"] / s["total"]
    delta = s["v13"] - s["v0"]
    if delta > 0:
        print(f"  {db:<30s} {v0_pct:5.1f}% {v13_pct:6.1f}% {'+' + str(delta):>4s}  ({s['total']:>4d})")

print(f"\n  ... showing top 25 databases with improvement")


print()
print("=" * 78)
print("  REMAINING GAPS: EX= but EM= even with V13 normalizer")
print("=" * 78)
print()

still_gap = []
for r in results:
    is_ex_true = r["ex"] is True
    is_em_vbest = normalize_v13(r["pred_sql"]) == normalize_v13(r["gold_sql"])
    if is_ex_true and not is_em_vbest:
        still_gap.append(r)

print(f"  Total EX= but EM= (V13): {len(still_gap)} samples")
print(f"  (These need MODEL improvements, not normalizer improvements)")
print()

gap_reasons = Counter()
for r in still_gap:
    pred = normalize_v13(r["pred_sql"])
    gold = normalize_v13(r["gold_sql"])
    reasons = []

    pred_joins = len(re.findall(r'\bjoin\b', pred))
    gold_joins = len(re.findall(r'\bjoin\b', gold))
    if pred_joins != gold_joins:
        reasons.append("DIFFERENT_JOIN_COUNT")

    if pred.count('select') != gold.count('select'):
        reasons.append("DIFFERENT_NESTING")

    pred_strs = set(re.findall(r"'([^']+)'", pred))
    gold_strs = set(re.findall(r"'([^']+)'", gold))
    if pred_strs != gold_strs and (pred_strs or gold_strs):
        reasons.append("DIFFERENT_STRING_VALUES")

    for agg in ['count', 'sum', 'avg', 'min', 'max']:
        if (agg + '(') in pred and (agg + '(') not in gold:
            reasons.append("WRONG_AGG"); break
        if (agg + '(') not in pred and (agg + '(') in gold:
            reasons.append("MISSING_AGG"); break

    if 'distinct' in pred and 'distinct' not in gold:
        reasons.append("EXTRA_DISTINCT")
    elif 'distinct' not in pred and 'distinct' in gold:
        reasons.append("MISSING_DISTINCT")

    if not reasons:
        reasons.append("OTHER_SURFACE_DIFF")

    for reason in reasons:
        gap_reasons[reason] += 1

for cat, count in gap_reasons.most_common():
    pct = 100 * count / len(still_gap) if still_gap else 0
    bar = "█" * int(pct / 3)
    print(f"  {cat:<30s} {count:>4d} ({pct:4.1f}%) {bar}")

print()
print(f"  Examples of remaining gaps:")
for r in still_gap[:8]:
    print(f"    {r['id']} | {r['db_id']}")
    print(f"      PRED: {normalize_v13(r['pred_sql'])[:100]}")
    print(f"      GOLD: {normalize_v13(r['gold_sql'])[:100]}")
    print()


# Final Summary

ex_count = sum(1 for r in results if r["ex"] is True)
ex_total = sum(1 for r in results if r["ex"] is not None)
em_v0 = sum(1 for r in results if normalize_v0(r["pred_sql"]) == normalize_v0(r["gold_sql"]))
em_vbest = sum(1 for r in results if normalize_v13(r["pred_sql"]) == normalize_v13(r["gold_sql"]))

print("=" * 78)
print("  FINAL SUMMARY")
print("=" * 78)
print()
print(f"  EX  (unchanged):        {ex_count}/{ex_total} = {100*ex_count/ex_total:.1f}%")
print(f"  EM  (V0 original):      {em_v0}/{len(results)} = {100*em_v0/len(results):.1f}%")
print(f"  EM  (V13 improved):     {em_vbest}/{len(results)} = {100*em_vbest/len(results):.1f}%")
print(f"  EM  improvement:        +{em_vbest - em_v0} samples (+{100*(em_vbest-em_v0)/len(results):.1f}%)")
print(f"  EX−EM gap (V0):        {100*ex_count/ex_total - 100*em_v0/len(results):.1f}%")
print(f"  EX−EM gap (V13):       {100*ex_count/ex_total - 100*em_vbest/len(results):.1f}%")
print(f"  Theoretical EM ceiling: {100*ex_count/len(results):.1f}% (if all EX= → EM=)")
print()


# OUTPUT: Save normalized results to JSON

print("=" * 78)
print("  EXPORTING NORMALIZED RESULTS")
print("=" * 78)
print()

normalized_results = []
for r in results:
    entry = {
        "id": r["id"],
        "db_id": r["db_id"],
        "question": r.get("question", ""),
        "gold_sql": r["gold_sql"],
        "pred_sql": r["pred_sql"],
        "ex": r.get("ex"),
        "em": r.get("em"),
        "bleu": r.get("bleu"),
        "sqam": r.get("sqam"),
        "tsed": r.get("tsed"),
        "pred_sql_norm_v0": normalize_v0(r["pred_sql"]),
        "gold_sql_norm_v0": normalize_v0(r["gold_sql"]),
        "pred_sql_norm_v13": normalize_v13(r["pred_sql"]),
        "gold_sql_norm_v13": normalize_v13(r["gold_sql"]),
        # EM under each normalizer
        "em_v0": normalize_v0(r["pred_sql"]) == normalize_v0(r["gold_sql"]),
        "em_v13": normalize_v13(r["pred_sql"]) == normalize_v13(r["gold_sql"]),
        # Intermediate normalizers (for ablation)
        "em_v1_strip_alias": normalize_v1(r["pred_sql"]) == normalize_v1(r["gold_sql"]),
        "em_v2_no_asc": normalize_v2(r["pred_sql"]) == normalize_v2(r["gold_sql"]),
        "em_v3_equivalences": normalize_v3(r["pred_sql"]) == normalize_v3(r["gold_sql"]),
        "em_v4_unify_quotes": normalize_v4(r["pred_sql"]) == normalize_v4(r["gold_sql"]),
        "em_v5_sort_select": normalize_v5(r["pred_sql"]) == normalize_v5(r["gold_sql"]),
        "em_v6_sort_where": normalize_v6(r["pred_sql"]) == normalize_v6(r["gold_sql"]),
        "em_v7_sort_in": normalize_v7(r["pred_sql"]) == normalize_v7(r["gold_sql"]),
        "em_v8_norm_on": normalize_v8(r["pred_sql"]) == normalize_v8(r["gold_sql"]),
        "em_v9_canon_alias": normalize_v9(r["pred_sql"]) == normalize_v9(r["gold_sql"]),
        "em_v10_op_spacing": normalize_v10(r["pred_sql"]) == normalize_v10(r["gold_sql"]),
        "em_v11_strip_val_quotes": normalize_v11(r["pred_sql"]) == normalize_v11(r["gold_sql"]),
        "em_v12_strip_num_quotes": normalize_v12(r["pred_sql"]) == normalize_v12(r["gold_sql"]),
        "em_v13_sort_groupby": normalize_v13(r["pred_sql"]) == normalize_v13(r["gold_sql"]),
        # Rescued / lost status
        "rescued_by_v13": (normalize_v13(r["pred_sql"]) == normalize_v13(r["gold_sql"])) and not (normalize_v0(r["pred_sql"]) == normalize_v0(r["gold_sql"])),
        "lost_by_v13": (normalize_v0(r["pred_sql"]) == normalize_v0(r["gold_sql"])) and not (normalize_v13(r["pred_sql"]) == normalize_v13(r["gold_sql"])),
    }
    if "metadata" in r:
        entry["metadata"] = r["metadata"]
    normalized_results.append(entry)

# Build output JSON
output_data = {
    "source": INPUT_PATH,
    "model": data.get("model", "N/A"),
    "original_metrics": data.get("metrics", {}),
    "normalizer_summary": {
        "EX": f"{100*ex_count/max(ex_total,1):.1f}%",
        "EX_count": f"{ex_count}/{ex_total}",
        "EM_v0": f"{100*em_v0/len(results):.1f}%",
        "EM_v0_count": f"{em_v0}/{len(results)}",
        "EM_v13": f"{100*em_vbest/len(results):.1f}%",
        "EM_v13_count": f"{em_vbest}/{len(results)}",
        "EM_gain": f"+{em_vbest - em_v0} samples (+{100*(em_vbest-em_v0)/len(results):.1f}%)",
        "rescued_count": len(rescued),
        "lost_count": len(lost),
    },
    "per_normalizer_em": {},
    "results": normalized_results,
}

# Add per-normalizer EM counts
for name, norm_fn in normalizers:
    em_c = sum(1 for r in results if norm_fn(r["pred_sql"]) == norm_fn(r["gold_sql"]))
    output_data["per_normalizer_em"][name.strip()] = {
        "count": em_c,
        "total": len(results),
        "pct": round(100 * em_c / len(results), 1),
    }

# Save
os.makedirs(os.path.dirname(OUTPUT_PATH) if os.path.dirname(OUTPUT_PATH) else ".", exist_ok=True)
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(output_data, f, ensure_ascii=False, indent=2)

print(f"  Saved {len(normalized_results)} normalized results → {OUTPUT_PATH}")
print()
print(f"  Output contains per sample:")
print(f"    • Original pred_sql and gold_sql")
print(f"    • Normalized pred/gold (V0 and V13)")
print(f"    • EM match status under all 14 normalizer versions")
print(f"    • rescued_by_v13 / lost_by_v13 flags")
print(f"    • All original fields (ex, em, bleu, sqam, tsed, metadata)")
print()
print(f"  Output contains summary:")
print(f"    • EX and EM counts under V0 and V13")
print(f"    • Per-normalizer EM breakdown (V0–V13)")
print(f"    • Rescued/lost sample counts")

## Spider Exact Set Match Evaluation (Ar-Spider Paper Comparison)

In [ ]:
#@title Spider Exact Set Match Evaluation (Ar-Spider Paper Comparison) { display-mode: "form" }


import os, subprocess, json, sys, re

# CONFIGURATION — Set your paths here

RESULTS_JSON = "/content/eval_results_7b_dev_Arabic_DOCUMENTED_FULL.json"  #@param {type:"string"}
AR_SPIDER_DIR = "/content/arspider"                #@param {type:"string"}

# Step 1: Load results from JSON
print(f"Loading results from: {RESULTS_JSON}")
with open(RESULTS_JSON, "r", encoding="utf-8") as f:
    data = json.load(f)

results = data["results"]
print(f"   Loaded {len(results)} samples")
print(f"   Model: {data.get('model', 'N/A')}")
print(f"   Metrics: {data.get('metrics', {})}")

# Step 2: Download official Spider evaluation scripts
SPIDER_EVAL_DIR = "/content/spider_eval"
os.makedirs(SPIDER_EVAL_DIR, exist_ok=True)

if not os.path.exists(os.path.join(SPIDER_EVAL_DIR, "evaluation.py")):
    print("\n Downloading official Spider evaluation scripts...")
    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/taoyds/spider.git",
        os.path.join(SPIDER_EVAL_DIR, "spider_repo")
    ], check=True, capture_output=True)
    # Copy the two needed files
    for fname in ["evaluation.py", "process_sql.py"]:
        src = os.path.join(SPIDER_EVAL_DIR, "spider_repo", fname)
        dst = os.path.join(SPIDER_EVAL_DIR, fname)
        if os.path.exists(src):
            subprocess.run(["cp", src, dst])
    print("   Spider eval scripts ready")
else:
    print("\n Spider eval scripts already downloaded")

# Step 3: Export gold.sql and pred.sql

gold_path = os.path.join(SPIDER_EVAL_DIR, "gold.sql")
pred_path = os.path.join(SPIDER_EVAL_DIR, "pred.sql")

print(f"\n Exporting {len(results)} predictions...")

with open(gold_path, "w", encoding="utf-8") as gf, \
     open(pred_path, "w", encoding="utf-8") as pf:
    for r in results:
        # Collapse to single line — SQL may contain \n which breaks the format
        gold_sql = " ".join(r["gold_sql"].strip().rstrip(";").split())
        pred_sql = " ".join(r["pred_sql"].strip().rstrip(";").split())
        db_id = r["db_id"]

        gf.write(f"{gold_sql}\t{db_id}\n")
        pf.write(f"{pred_sql}\n")

print(f"   gold.sql: {gold_path}")
print(f"   pred.sql: {pred_path}")

# Verify
with open(gold_path) as f:
    gold_lines = len(f.readlines())
with open(pred_path) as f:
    pred_lines = len(f.readlines())
print(f"   Lines: gold={gold_lines}, pred={pred_lines}")
assert gold_lines == pred_lines, f"Mismatch! gold={gold_lines} pred={pred_lines}"

# Step 4: Locate tables.json and database directory
TABLES_JSON = os.path.join(AR_SPIDER_DIR, "tables.json")
DB_DIR = ""
for candidate in ["databases", "database"]:
    p = os.path.join(AR_SPIDER_DIR, candidate)
    if os.path.isdir(p):
        DB_DIR = p
        break

assert os.path.exists(TABLES_JSON), f"tables.json not found at {TABLES_JSON}"
assert DB_DIR, f"database directory not found in {AR_SPIDER_DIR}"
print(f"\n tables.json: {TABLES_JSON}")
print(f"database dir: {DB_DIR}")

# Step 5: Run official Spider evaluation
print("\n" + "=" * 70)
print("Running Official Spider Evaluation (Exact Set Match without Values)")
print("   Same metric used by Ar-Spider paper (Almohaimeed et al., 2024)")
print("=" * 70 + "\n")

eval_script = os.path.join(SPIDER_EVAL_DIR, "evaluation.py")
cmd = [
    sys.executable, eval_script,
    "--gold", gold_path,
    "--pred", pred_path,
    "--etype", "match",
    "--db", DB_DIR,
    "--table", TABLES_JSON,
]

print(f"Running: {' '.join(cmd)}\n")
result = subprocess.run(cmd, capture_output=True, text=True, cwd=SPIDER_EVAL_DIR)

# Print output
if result.stdout:
    print(result.stdout)
if result.stderr:
    print(result.stderr)

# Step 6: Parse and display comparison
output_text = result.stdout + "\n" + result.stderr

esm_match = re.search(r'exact match[^:]*:\s*([\d.]+)', output_text, re.I)
easy_match = re.search(r'easy[^:]*:\s*([\d.]+)', output_text, re.I)
medium_match = re.search(r'medium[^:]*:\s*([\d.]+)', output_text, re.I)
hard_match = re.search(r'hard[^:\n]*:\s*([\d.]+)', output_text, re.I)
extra_match = re.search(r'extra[^:]*:\s*([\d.]+)', output_text, re.I)

esm_score = float(esm_match.group(1)) if esm_match else None

if esm_score is not None:
    # Compute your EX/EM from the loaded results
    ex_count = sum(1 for r in results if r.get("ex") is True)
    ex_total = sum(1 for r in results if r.get("ex") is not None)
    em_count = sum(1 for r in results if r.get("em") is True)
    ex_pct = 100 * ex_count / max(ex_total, 1)
    em_pct = 100 * em_count / max(len(results), 1)
    esm_pct = 100 * esm_score

    print("\n" + "=" * 70)
    print("COMPARISON: Your System vs Ar-Spider Paper")
    print("=" * 70)
    row = "{:<45} {:>10}"
    print(row.format("", "Score"))
    print(row.format("─" * 45, "─" * 10))
    print()
    print("  Ar-Spider Paper (Almohaimeed et al., 2024):")
    print(row.format("    LGESQL + mBERT (ESM)", "52.51%"))
    print(row.format("    S2SQL + mBERT (ESM)", "55.90%"))
    print(row.format("    S2SQL + XLM-R (ESM)", "62.48%"))
    print(row.format("    S2SQL + XLM-R + CSR (ESM)", "64.00%"))
    print(row.format("    LGESQL + XLM-R (ESM)", "65.57%"))
    print(row.format("    LGESQL + XLM-R + CSR (ESM) ← best", "66.63%"))
    print()
    print("  Your System (Qwen2.5-Coder-7B + QLoRA V3 — Schema Linking + Skeleton-First):")
    print(row.format(f"    Exact Set Match (ESM, this eval)", f"{esm_pct:.1f}%"))
    print(row.format(f"    Execution Accuracy (EX)", f"{ex_pct:.1f}%"))
    print(row.format(f"    String Exact Match (EM)", f"{em_pct:.1f}%"))

    if easy_match:
        print()
        print("  By difficulty:")
        if easy_match:
            print(row.format("    Easy", f"{100*float(easy_match.group(1)):.1f}%"))
        if medium_match:
            print(row.format("    Medium", f"{100*float(medium_match.group(1)):.1f}%"))
        if hard_match:
            print(row.format("    Hard", f"{100*float(hard_match.group(1)):.1f}%"))
        if extra_match:
            print(row.format("    Extra Hard", f"{100*float(extra_match.group(1)):.1f}%"))

    print()
    print("=" * 70)
    print("   NOTE: Ar-Spider paper uses ESM *without values* (DISABLE_VALUE=True)")
    print("       Their models do NOT predict WHERE clause values at all.")
    print("       Your EX metric is stricter — requires correct values AND results.")
    print("=" * 70)
else:
    print("\n Could not auto-parse ESM score from output.")
    print("   Check the raw output above for results.")
    print("   You can also run manually:")
    print(f"   cd {SPIDER_EVAL_DIR}")
    print(f"   python evaluation.py --gold gold.sql --pred pred.sql \\")
    print(f"     --etype match --db {DB_DIR} --table {TABLES_JSON}")